# Sudanese ASR Zeroshot

Zero-shot Whisper small / medium / large-v2 / large-v3 / large-v3-turbo (+ the public AymanMansour/Whisper-Sudanese-Dialect-small checkpoint as an external reference) on the SDN speaker-disjoint test split and on OOOK-Eval-Final.

Data: Mansour & Mukhtar, Sudanese dialect speech dataset, Zenodo 10.5281/zenodo.6869079 (v1.1, CC BY 4.0); Mansour (2026) OOOK-Eval-Final (evaluation only, not redistributed).

Generated by `scripts/build_notebooks.py` from the sudanese-asr-eval repository. Do not edit by hand.

In [ ]:
!pip install -q transformers==4.56.2 peft==0.17.1 jiwer==3.1.0
!mkdir -p /kaggle/working/src/sudanese_asr_eval /kaggle/working/configs /kaggle/working/splits /kaggle/working/tests /kaggle/working/outputs
!nvidia-smi

In [ ]:
# Optional: use a Hugging Face token stored as a Kaggle secret named HF_TOKEN (not required).
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("no HF_TOKEN secret; anonymous Hugging Face access")

In [ ]:
%%writefile /kaggle/working/src/sudanese_asr_eval/__init__.py
"""Evaluation tools for Sudanese Arabic ASR: text normalisation (raw / basic / full)
and corpus-level WER/CER with bootstrap confidence intervals and paired tests."""
from .textnorm import VIEWS, basic, full, get_view, normalize, raw, whisper_basic

__version__ = "1.0.0"
__all__ = ["VIEWS", "basic", "full", "get_view", "normalize", "raw", "whisper_basic", "__version__"]


In [ ]:
%%writefile /kaggle/working/src/sudanese_asr_eval/textnorm.py
"""Text normalisation for Sudanese Arabic ASR scoring.

Three views are reported for every system:

  raw(s)    Unicode NFC + whitespace collapse. Nothing else.

  basic(s)  Re-implementation of Whisper's ``BasicTextNormalizer()`` (default
            arguments), as used by the Hugging Face Whisper fine-tuning template
            (``do_normalize_eval=True``): lower-case, drop bracketed spans, replace
            every Unicode Mark / Symbol / Punctuation character with a space,
            collapse whitespace. It does NOT unify alef / ya / ta marbuta.

  full(s)   The full Arabic normaliser, in this order:
     1. NFKC (folds Arabic presentation forms, e.g. U+FEFB lam-alef ligature)
     2. remove diacritics U+064B-U+065F, superscript alef U+0670,
        Quranic marks U+06D6-U+06ED and tatweel U+0640
     3. alef variants  أ إ آ ٱ  -> ا
     4. alef maqsura   ى -> ي ;  hamza seats  ؤ -> و , ئ -> ي
     5. ta marbuta     ة -> ه
     6. digits: Eastern-Arabic (٠-٩) and Persian (۰-۹) -> ASCII 0-9.
        Numbers are not verbalised: the corpus writes numbers as dialect words
        ("تسعمية تمنية") and no reliable Sudanese number verbaliser exists.
     7. punctuation and symbols (Unicode categories P* and S*, incl. ، ؛ ؟ « ») -> space
     8. Latin letters lower-cased
     9. a stand-alone conjunction "و" is attached to the next word
        (the corpus writes "و تسعمية" and "وتسعمية" interchangeably)
    10. whitespace collapse

Every view is applied identically to references and hypotheses.
In the paper the views are called raw, basic and norm; ``normalize`` is kept as
an alias of ``full`` so that the original experiment code runs unchanged.
"""
from __future__ import annotations

import re
import unicodedata

_DIACRITICS = re.compile(r"[ً-ٰٟۖ-ۭـ]")
_ALEF = {"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا"}
_YA_HAMZA = {"ى": "ي", "ؤ": "و", "ئ": "ي"}
_TA = {"ة": "ه"}
_DIGITS = {**{chr(0x0660 + i): str(i) for i in range(10)},
           **{chr(0x06F0 + i): str(i) for i in range(10)}}
_TRANS = str.maketrans({**_ALEF, **_YA_HAMZA, **_TA, **_DIGITS})
_WAW = re.compile(r"(?:(?<=\s)|^)و\s+(?=\S)")
_BRACKETS = re.compile(r"[<\[][^>\]]*[>\]]|\(([^)]+?)\)")


def raw(s: str) -> str:
    """Unicode NFC and whitespace collapse only."""
    return " ".join(unicodedata.normalize("NFC", s or "").split())


def whisper_basic(s: str) -> str:
    """Whisper ``BasicTextNormalizer()`` with default arguments (see module doc)."""
    s = (s or "").lower()
    s = _BRACKETS.sub("", s)
    s = "".join(" " if unicodedata.category(c)[0] in "MSP" else c for c in unicodedata.normalize("NFKC", s))
    return " ".join(s.split())


def _strip_punct(s: str) -> str:
    return "".join(" " if unicodedata.category(c)[0] in "PS" else c for c in s)


def full(s: str) -> str:
    """Full Arabic normaliser (the paper's *norm* view)."""
    s = unicodedata.normalize("NFKC", s or "")
    s = _DIACRITICS.sub("", s)
    s = s.translate(_TRANS)
    s = _strip_punct(s)
    s = s.lower()
    s = " ".join(s.split())
    s = _WAW.sub("و", s)
    return " ".join(s.split())


basic = whisper_basic
normalize = full

#: name -> function, in the order used in all tables
VIEWS = {"raw": raw, "basic": whisper_basic, "norm": full}
ALIASES = {"full": "norm", "whisper_basic": "basic"}


def get_view(name: str):
    name = ALIASES.get(name, name)
    if name not in VIEWS:
        raise KeyError(f"unknown view {name!r}; choose from {sorted(VIEWS) + sorted(ALIASES)}")
    return VIEWS[name]


def ablation_views():
    """Cumulative steps of the full normaliser, as in the paper's ablation table.

    Returns [(label, fn)]: raw, +p/d (NFKC, diacritics, punctuation, lower-case),
    +alef, +ya (ya and hamza seats), +ta (ta marbuta), +wa (= norm).
    Digit mapping is applied with the last step only; references contain no digits.
    """
    t_alef, t_ya, t_ta = (str.maketrans(m) for m in (_ALEF, _YA_HAMZA, _TA))

    def stage(k):
        def f(s):
            s = unicodedata.normalize("NFKC", s or "")
            s = _DIACRITICS.sub("", s)
            s = " ".join(_strip_punct(s).lower().split())
            if k >= 1:
                s = s.translate(t_alef)
            if k >= 2:
                s = s.translate(t_ya)
            if k >= 3:
                s = s.translate(t_ta)
            if k >= 4:
                s = " ".join(_WAW.sub("و", s).split())
            return s
        return f

    return [("raw", raw), ("+p/d", stage(0)), ("+alef", stage(1)), ("+ya", stage(2)),
            ("+ta", stage(3)), ("+wa", full)]


In [ ]:
%%writefile /kaggle/working/src/sudanese_asr_eval/metrics.py
"""Corpus-level WER/CER with utterance-level bootstrap confidence intervals.

WER = sum(word edits) / sum(reference words) over the whole set (not a mean of
per-utterance WERs). CER is the same over characters, with spaces kept as characters
(jiwer's default). Utterances whose (normalised) reference is empty are excluded from
scoring and counted separately.

Bootstrap: resample utterances with replacement B times (default 1000, fixed seed),
recompute corpus-level WER each time, report the 2.5 / 97.5 percentiles.
Paired bootstrap for system A vs B uses the same resampled indices for both systems.
"""
from __future__ import annotations

import numpy as np
import jiwer

from .textnorm import normalize, raw, whisper_basic

VIEWS = (("raw", raw), ("basic", whisper_basic), ("norm", normalize))


def _edits(refs, hyps, unit):
    errs, lens = [], []
    for r, h in zip(refs, hyps):
        if unit == "word":
            o = jiwer.process_words(r, h if h.strip() else "")
            n = len(r.split())
        else:
            o = jiwer.process_characters(r, h)
            n = len(r)
        errs.append(o.substitutions + o.deletions + o.insertions)
        lens.append(n)
    return np.asarray(errs, dtype=np.int64), np.asarray(lens, dtype=np.int64)


def per_utt(refs, hyps, views=VIEWS):
    """Return dict view -> {'idx','w_err','w_len','c_err','c_len'} arrays (one entry per scored utterance)."""
    out = {}
    for view, f in views:
        r = [f(x) for x in refs]
        h = [f(x) for x in hyps]
        keep = [i for i, x in enumerate(r) if x]
        r = [r[i] for i in keep]
        h = [h[i] for i in keep]
        we, wl = _edits(r, h, "word")
        ce, cl = _edits(r, h, "char")
        out[view] = dict(idx=np.asarray(keep), w_err=we, w_len=wl, c_err=ce, c_len=cl)
    return out


def bootstrap_ci(err, length, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(err)
    idx = rng.integers(0, n, size=(B, n))
    stats = err[idx].sum(1) / np.maximum(length[idx].sum(1), 1)
    return float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))


def summarize(refs, hyps, B=1000, seed=0):
    pu = per_utt(refs, hyps)
    res = {"n_utts": len(refs)}
    for view, d in pu.items():
        for unit in ("w", "c"):
            e, l = d[f"{unit}_err"], d[f"{unit}_len"]
            point = float(e.sum() / max(l.sum(), 1))
            lo, hi = bootstrap_ci(e, l, B, seed)
            name = ("wer" if unit == "w" else "cer") + "_" + view
            res[name] = round(100 * point, 2)
            res[name + "_ci95"] = [round(100 * lo, 2), round(100 * hi, 2)]
        res[f"n_scored_{view}"] = int(len(d["idx"]))
    return res


def paired_bootstrap(refs, hyps_a, hyps_b, view="norm", unit="w", B=1000, seed=0):
    """CI of WER(A) - WER(B) on the same utterances, plus P(A >= B) under resampling."""
    a, b = per_utt(refs, hyps_a)[view], per_utt(refs, hyps_b)[view]
    ea, eb, l = a[f"{unit}_err"], b[f"{unit}_err"], a[f"{unit}_len"]
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(l), size=(B, len(l)))
    L = np.maximum(l[idx].sum(1), 1)
    diff = (ea[idx].sum(1) - eb[idx].sum(1)) / L
    return dict(delta=round(100 * float((ea.sum() - eb.sum()) / l.sum()), 2),
                ci95=[round(100 * float(np.percentile(diff, 2.5)), 2), round(100 * float(np.percentile(diff, 97.5)), 2)],
                p_a_not_better=float((diff >= 0).mean()))




def corpus_rate(err, length):
    """Corpus-level error rate in % from per-utterance edit counts and lengths."""
    err, length = np.asarray(err), np.asarray(length)
    return 100.0 * float(err.sum() / max(length.sum(), 1))


def bootstrap_ci_counts(err, length, B=1000, seed=42):
    """95 % bootstrap CI (in %) from per-utterance counts, same resampling as `summarize`."""
    lo, hi = bootstrap_ci(np.asarray(err, dtype=np.int64), np.asarray(length, dtype=np.int64), B, seed)
    return 100 * lo, 100 * hi


def paired_bootstrap_counts(err_a, err_b, length, B=1000, seed=42):
    """Paired bootstrap of rate(A) - rate(B) from per-utterance counts (same as `paired_bootstrap`)."""
    ea, eb, l = (np.asarray(x, dtype=np.int64) for x in (err_a, err_b, length))
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(l), size=(B, len(l)))
    L = np.maximum(l[idx].sum(1), 1)
    diff = (ea[idx].sum(1) - eb[idx].sum(1)) / L
    return dict(delta=round(100 * float((ea.sum() - eb.sum()) / l.sum()), 2),
                ci95=[round(100 * float(np.percentile(diff, 2.5)), 2), round(100 * float(np.percentile(diff, 97.5)), 2)],
                p_a_not_better=float((diff >= 0).mean()))


In [ ]:
%%writefile /kaggle/working/src/sudanese_asr_eval/loops.py
"""Post-hoc diagnostic: collapse runaway repetition loops in a hypothesis.

A run of 3 or more consecutive copies of the same 1-4-word n-gram is collapsed
to one copy, repeatedly, until nothing changes. The rule is reference-free, but
it was chosen after looking at the outputs, so scores computed with it are
exploratory and are not part of the evaluation protocol. It can also collapse
genuine repetitions such as "لا لا لا".
"""
from __future__ import annotations


def collapse_loops(text: str, max_n: int = 4, min_repeats: int = 3) -> str:
    w = text.split()
    changed = True
    while changed:
        changed = False
        for n in range(1, max_n + 1):
            i, out = 0, []
            while i < len(w):
                g = w[i:i + n]
                k = 1
                while w[i + k * n:i + (k + 1) * n] == g and len(g) == n:
                    k += 1
                if k >= min_repeats:
                    out += g
                    i += k * n
                    changed = True
                else:
                    out.append(w[i])
                    i += 1
            w = out
    return " ".join(w)


def is_hallucination(ref: str, hyp: str) -> bool:
    """The paper's flag: hypothesis longer than 3x the reference and more than 5 words."""
    rl, hl = len(ref.split()), len(hyp.split())
    return hl > 3 * rl and hl > 5


In [ ]:
%%writefile /kaggle/working/src/data.py
"""Data loading. No `datasets` Audio decoding (it changes between versions); we read
WAV / parquet bytes ourselves with soundfile and resample with librosa (soxr).

Corpora
  * SDN  - Mansour & Mukhtar, Sudanese dialect speech dataset, Zenodo 10.5281/zenodo.6869079
           v1.1, CC BY 4.0. Read from the ORIGINAL Zenodo zip (md5 below). All reported
           numbers use v1.1; the later v1.2 (record 7426918, same concept DOI
           10.5281/zenodo.6721682) is a different archive and is not checked here. The HF mirror
           `arbml/sudanese_dialect_speech` must NOT be used: its transcripts are shifted by
           1-2 rows against the audio (only 85 / 3,547 rows match the Zenodo text files).
  * OOOK - Mansour's evaluation set used in arXiv 2601.06802 (OOOK-Eval), HF
           `AymanMansour/OOOK-Eval-Final` @ e256ab53 (797 clips). Evaluation only; no licence
           is stated on the card, so we never redistribute it.
"""
from __future__ import annotations

import csv
import hashlib
import io
import os
import urllib.request
import zipfile

import numpy as np
import soundfile as sf

SR = 16000
SDN_URL = "https://zenodo.org/records/6869079/files/Sudanese_dialect_speech_dataset.zip?download=1"
SDN_MD5 = "61effea8ff14daa690bb0e05f9930719"
SDN_FILES = "SDN Dialect Corpus v1.0/Dataset files/"
OOOK_REPO = "AymanMansour/OOOK-Eval-Final"
OOOK_REV = "e256ab538b6efb2e0e9e452b962d44b198662619"
OOOK_FILE = "data/train-00000-of-00001.parquet"


def _to16k(wav: np.ndarray, sr: int) -> np.ndarray:
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav = wav.astype(np.float32)
    if sr != SR:
        import librosa
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR, res_type="soxr_hq")
    return wav


def _md5(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 22), b""):
            h.update(chunk)
    return h.hexdigest()


def ensure_sdn_zip(cache_dir: str) -> str:
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, "Sudanese_dialect_speech_dataset.zip")
    if not os.path.exists(path):
        print("[data] downloading SDN corpus from Zenodo (1.45 GB) ...", flush=True)
        tmp = path + ".part"
        with urllib.request.urlopen(SDN_URL, timeout=120) as r, open(tmp, "wb") as f:
            while True:
                b = r.read(1 << 22)
                if not b:
                    break
                f.write(b)
        os.replace(tmp, path)
    got = _md5(path)
    if got != SDN_MD5:
        raise RuntimeError(f"SDN zip md5 mismatch: {got} != {SDN_MD5}")
    return path


class _HttpFile(io.RawIOBase):
    def __init__(self, url):
        r = urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=60)
        self.url, self.size, self.pos = r.url, int(r.headers["Content-Length"]), 0

    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else (self.pos + off if whence == 1 else self.size + off)
        return self.pos

    def readinto(self, b):
        n = min(len(b), self.size - self.pos)
        if n <= 0:
            return 0
        for attempt in range(5):
            try:
                req = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{self.pos + n - 1}"})
                d = urllib.request.urlopen(req, timeout=120).read()
                break
            except Exception:
                if attempt == 4:
                    raise
        b[:len(d)] = d
        self.pos += len(d)
        return len(d)


def open_remote_sdn_zip() -> zipfile.ZipFile:
    """Read single clips from the Zenodo zip by HTTP range requests (for small eval runs)."""
    return zipfile.ZipFile(io.BufferedReader(_HttpFile(SDN_URL), buffer_size=1 << 20))


def read_split(tsv_path: str):
    lines = [l for l in open(tsv_path, encoding="utf-8") if not l.startswith("#")]
    return list(csv.DictReader(lines, delimiter="\t"))


def load_sdn(tsv_path: str, zip_source, splits=("train", "dev", "test"), limit=None):
    """zip_source: a path, or an already-open zipfile.ZipFile (e.g. over HTTP for tests)."""
    z = zip_source if isinstance(zip_source, zipfile.ZipFile) else zipfile.ZipFile(zip_source)
    rows = read_split(tsv_path)
    out = {s: [] for s in splits}
    for r in rows:
        if r["split"] not in out or (limit and len(out[r["split"]]) >= limit):
            continue
        text = z.read(SDN_FILES + r["id"] + ".txt").decode("utf-8-sig").strip()
        wav, sr = sf.read(io.BytesIO(z.read(SDN_FILES + r["id"] + ".wav")), dtype="float32")
        out[r["split"]].append(dict(id=r["id"], text=text, audio=_to16k(wav, sr),
                                    speaker=r["speaker"], gender=r["gender"], program=r["program"]))
    for s in splits:
        print(f"[data] SDN {s}: {len(out[s])} utts, {sum(len(x['audio']) for x in out[s]) / SR / 3600:.3f} h", flush=True)
    return out


def load_ookeval(cache_dir: str, limit=None):
    import pyarrow.parquet as pq
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(OOOK_REPO, OOOK_FILE, repo_type="dataset", revision=OOOK_REV, cache_dir=cache_dir)
    t = pq.read_table(p, columns=["id", "title", "text", "audio"]).to_pylist()
    out = []
    for r in t[: limit or None]:
        a = r["audio"]
        wav, sr = sf.read(io.BytesIO(a["bytes"]), dtype="float32")
        out.append(dict(id=f"{r['id']}{r['title']}", text=(r["text"] or "").strip(), audio=_to16k(wav, sr)))
    print(f"[data] OOOK-Eval-Final: {len(out)} utts, {sum(len(x['audio']) for x in out) / SR / 3600:.3f} h", flush=True)
    return out


In [ ]:
%%writefile /kaggle/working/src/asr.py
"""Whisper loading + batched greedy transcription (shared by zero-shot and fine-tuning)."""
from __future__ import annotations

import json
import os
import random
import time

import numpy as np
import torch
from transformers import GenerationConfig, WhisperForConditionalGeneration, WhisperProcessor

GEN_MAX_NEW_TOKENS = 225  # same cap as Mansour's runs (generation_max_length=225)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def device_dtype():
    if torch.cuda.is_available():
        return "cuda", torch.float16
    return "cpu", torch.float32


def _local_if_bin_only(model_id: str) -> str:
    """Repos with only pytorch_model.bin make transformers start a background
    'auto_conversion' thread that contacts the Hub's safetensors-convert service (it can
    open a PR on someone else's repo, and it delays exit by minutes). Loading the same
    files from a local snapshot avoids both. Repos with safetensors load directly."""
    if os.path.isdir(model_id):
        return model_id
    from huggingface_hub import list_repo_files, snapshot_download
    files = list_repo_files(model_id)
    if any(f.endswith(".safetensors") for f in files):
        return model_id
    return snapshot_download(model_id, allow_patterns=["*.json", "pytorch_model*.bin", "*.txt"])


def load(model_id: str, gen_config_from: str | None = None, dtype=None, device=None):
    device, auto_dtype = device_dtype() if device is None else (device, dtype)
    dtype = dtype or auto_dtype
    proc = WhisperProcessor.from_pretrained(gen_config_from or model_id)
    model = WhisperForConditionalGeneration.from_pretrained(_local_if_bin_only(model_id), dtype=dtype).to(device).eval()
    if gen_config_from:  # old 2022 checkpoints lack lang_to_id in generation_config
        model.generation_config = GenerationConfig.from_pretrained(gen_config_from)
    return model, proc


@torch.no_grad()
def transcribe(model, proc, samples, batch_size=16, language="ar", num_beams=1, log_every=20):
    """Greedy (num_beams=1) decoding, language forced to Arabic unless language=None.
    Clips are sorted by length for batching; output order matches `samples`."""
    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype
    order = sorted(range(len(samples)), key=lambda i: len(samples[i]["audio"]))
    hyps = [None] * len(samples)
    t0 = time.time()
    for b, start in enumerate(range(0, len(order), batch_size)):
        idx = order[start:start + batch_size]
        feats = proc.feature_extractor([samples[i]["audio"] for i in idx], sampling_rate=16000,
                                       return_tensors="pt").input_features.to(device, dtype)
        kw = dict(task="transcribe", max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=num_beams)
        if language:
            kw["language"] = language
        out = model.generate(feats, **kw)
        for i, text in zip(idx, proc.batch_decode(out, skip_special_tokens=True)):
            hyps[i] = text.strip()
        if log_every and b % log_every == 0:
            print(f"    batch {b + 1}/{(len(order) + batch_size - 1) // batch_size}  {time.time() - t0:.0f}s", flush=True)
    return hyps


def save_hyps(path, samples, hyps):
    with open(path, "w", encoding="utf-8") as f:
        for s, h in zip(samples, hyps):
            f.write(json.dumps(dict(id=s["id"], ref=s["text"], hyp=h), ensure_ascii=False) + "\n")


def env_info():
    import platform
    import transformers
    info = dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
                cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    try:
        import peft
        info["peft"] = peft.__version__
    except ImportError:
        pass
    try:
        import jiwer
        info["jiwer"] = getattr(jiwer, "__version__", "?")
    except ImportError:
        pass
    return info


def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p


In [ ]:
%%writefile /kaggle/working/src/run_zeroshot.py
"""Zero-shot evaluation of pretrained Whisper checkpoints.

    python experiments/run_zeroshot.py --config experiments/configs/zeroshot.json [--limit 4] [--models openai/whisper-tiny]

Writes <out_dir>/zeroshot_metrics.json and one hyps JSONL per (model, test set).
Test sets: SDN speaker-disjoint `test` split, and Mansour's OOOK-Eval-Final.
Dev is not scored here (zero-shot has nothing to select).
"""
from __future__ import annotations

import argparse
import gc
import json
import os
import time

import torch

import asr
import data
from sudanese_asr_eval import metrics


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--limit", type=int, default=None, help="tiny subset for smoke tests")
    ap.add_argument("--models", nargs="*", help="override model list (ids)")
    ap.add_argument("--zip", default=None, help="local SDN zip path, or 'remote' (else download)")
    args = ap.parse_args()
    cfg = json.load(open(args.config, encoding="utf-8"))
    out_dir = asr.ensure_dir(cfg["out_dir"])
    asr.set_seed(cfg["seed"])
    if args.zip == "remote":  # HTTP range reads of only the needed clips (no 1.45 GB download)
        zip_path = data.open_remote_sdn_zip()
    else:
        zip_path = args.zip or data.ensure_sdn_zip(cfg["cache_dir"])
    sets = {"sdn_test": data.load_sdn(cfg["split_tsv"], zip_path, splits=("test",), limit=args.limit)["test"],
            "ookeval": data.load_ookeval(cfg["cache_dir"], limit=args.limit)}
    runs = cfg["models"] if not args.models else [dict(id=m) for m in args.models]
    path = os.path.join(out_dir, "zeroshot_metrics.json")
    results = json.load(open(path)) if os.path.exists(path) else {"env": asr.env_info(), "config": cfg, "runs": {}}
    for run in runs:
        name = run.get("name", run["id"])
        if name in results["runs"]:
            print(f"[skip] {name} already done")
            continue
        print(f"[model] {name}", flush=True)
        model, proc = asr.load(run["id"], run.get("gen_config_from"))
        res = {"model": run["id"], "language": run.get("language", "ar")}
        for set_name in run.get("sets", list(sets)):
            samples = sets[set_name]
            t0 = time.time()
            hyps = asr.transcribe(model, proc, samples, batch_size=run.get("batch_size", cfg["batch_size"]),
                                  language=run.get("language", "ar"))
            asr.save_hyps(os.path.join(out_dir, f"hyps__{name.replace('/', '__')}__{set_name}.jsonl"), samples, hyps)
            m = metrics.summarize([s["text"] for s in samples], hyps, B=cfg["bootstrap"], seed=cfg["seed"])
            m["seconds"] = round(time.time() - t0, 1)
            res[set_name] = m
            print(f"   {set_name}: WER raw {m['wer_raw']} | basic {m['wer_basic']} | norm {m['wer_norm']} "
                  f"{m['wer_norm_ci95']}  CER norm {m['cer_norm']}  ({m['seconds']}s)", flush=True)
        results["runs"][name] = res
        json.dump(results, open(path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        del model
        gc.collect()
        torch.cuda.empty_cache()
    print("[done]", path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/run_finetune.py
"""Fine-tune Whisper on the SDN speaker-disjoint TRAIN split.

    python experiments/run_finetune.py --config experiments/configs/ft_whisper_small.json [--limit 8 --max_steps 4]

Model selection uses the DEV split only (normalised WER, greedy decoding, every
`eval_every` steps; early stop after `patience` evals without improvement).
The TEST split and OOOK-Eval are decoded exactly ONCE, with the selected checkpoint,
after training ends. This fixes the audit finding where `load_best_model_at_end`
picked the checkpoint on test data.

mode = "full": all weights trained (fp32 master weights, fp16 autocast + GradScaler).
mode = "lora": PEFT LoRA adapters on attention + MLP projections of encoder and decoder;
               base frozen; gradient checkpointing to fit a 16 GB T4/P100.
"""
from __future__ import annotations

import argparse
import json
import math
import os
import time

import numpy as np
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor, get_linear_schedule_with_warmup

import asr
import data
from sudanese_asr_eval import metrics


def make_batches(n, bs, rng):
    idx = rng.permutation(n)
    return [idx[i:i + bs] for i in range(0, n - bs + 1, bs)]


def collate(proc, samples, device):
    feats = proc.feature_extractor([s["audio"] for s in samples], sampling_rate=16000,
                                   return_tensors="pt").input_features
    lab = [proc.tokenizer(s["text"]).input_ids for s in samples]
    start = proc.tokenizer.convert_tokens_to_ids("<|startoftranscript|>")
    lab = [(l[1:] if l and l[0] == start else l)[:448] for l in lab]  # model prepends decoder_start itself
    L = max(len(l) for l in lab)
    labels = torch.full((len(lab), L), -100, dtype=torch.long)
    for i, l in enumerate(lab):
        labels[i, :len(l)] = torch.tensor(l)
    return feats.to(device), labels.to(device)


def build_model(cfg, device):
    proc = WhisperProcessor.from_pretrained(cfg["base_model"], language="arabic", task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(cfg["base_model"], dtype=torch.float32)
    model.config.use_cache = False
    model.generation_config.forced_decoder_ids = None
    model.config.forced_decoder_ids = None
    if cfg.get("spec_augment"):
        model.config.apply_spec_augment = True
        model.config.mask_time_prob = cfg.get("mask_time_prob", 0.05)
    if cfg["mode"] == "lora":
        from peft import LoraConfig, get_peft_model
        if cfg.get("gradient_checkpointing", True):
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            model.enable_input_require_grads()
        lc = LoraConfig(r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"], lora_dropout=cfg["lora_dropout"],
                        target_modules=cfg["lora_targets"], bias="none")
        model = get_peft_model(model, lc)
        model.print_trainable_parameters()
    elif cfg.get("gradient_checkpointing"):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    return model.to(device), proc


@torch.no_grad()
def decode(model, proc, samples, bs, use_amp):
    model.eval()
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    base.config.use_cache = True
    with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
        hyps = asr.transcribe(model, proc, samples, batch_size=bs, log_every=0)
    base.config.use_cache = False
    model.train()
    return hyps


def trainable_state(model):
    return {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()
            if (not hasattr(model, "peft_config")) or "lora_" in k}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--max_steps", type=int, default=None)
    ap.add_argument("--eval_every", type=int, default=None)
    ap.add_argument("--zip", default=None)
    args = ap.parse_args()
    cfg = json.load(open(args.config, encoding="utf-8"))
    if args.max_steps:
        cfg["max_steps"] = args.max_steps
    if args.eval_every:
        cfg["eval_every"] = args.eval_every
    out_dir = asr.ensure_dir(cfg["out_dir"])
    asr.set_seed(cfg["seed"])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = device == "cuda" and cfg.get("fp16", True)

    zip_path = args.zip or data.ensure_sdn_zip(cfg["cache_dir"])
    sdn = data.load_sdn(cfg["split_tsv"], zip_path, limit=args.limit)
    train, dev = sdn["train"], sdn["dev"]
    model, proc = build_model(cfg, device)
    model.train()

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg.get("weight_decay", 0.0))
    sched = get_linear_schedule_with_warmup(opt, cfg["warmup_steps"], cfg["max_steps"])
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    rng = np.random.default_rng(cfg["seed"])
    bs, accum = cfg["batch_size"], cfg.get("grad_accum", 1)

    log = open(os.path.join(out_dir, "train_log.jsonl"), "a", encoding="utf-8")
    best = dict(step=0, dev_wer_norm=math.inf)
    best_state, bad_evals, step, t0 = None, 0, 0, time.time()
    batches, run_loss = [], 0.0
    while step < cfg["max_steps"]:
        for _ in range(accum):
            if not batches:
                batches = make_batches(len(train), bs, rng)
            feats, labels = collate(proc, [train[i] for i in batches.pop()], device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                loss = model(input_features=feats, labels=labels).loss / accum
            scaler.scale(loss).backward()
            run_loss += loss.item()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, cfg.get("max_grad_norm", 1.0))
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        sched.step()
        step += 1
        if step % cfg.get("log_every", 25) == 0:
            rec = dict(step=step, loss=round(run_loss / cfg.get("log_every", 25), 4), lr=sched.get_last_lr()[0],
                       elapsed=round(time.time() - t0))
            print(rec, flush=True)
            log.write(json.dumps(rec) + "\n"); log.flush()
            run_loss = 0.0
        if step % cfg["eval_every"] == 0 or step == cfg["max_steps"]:
            hyps = decode(model, proc, dev, cfg["eval_batch_size"], use_amp)
            m = metrics.summarize([s["text"] for s in dev], hyps, B=200, seed=cfg["seed"])
            rec = dict(step=step, dev_wer_norm=m["wer_norm"], dev_cer_norm=m["cer_norm"], dev_wer_raw=m["wer_raw"],
                       elapsed=round(time.time() - t0))
            print("[dev]", rec, flush=True)
            log.write(json.dumps(rec) + "\n"); log.flush()
            if m["wer_norm"] < best["dev_wer_norm"]:
                best = dict(step=step, dev_wer_norm=m["wer_norm"], dev=m)
                best_state = trainable_state(model)
                bad_evals = 0
            else:
                bad_evals += 1
                if bad_evals >= cfg.get("patience", 4):
                    print(f"[early stop] no dev improvement for {bad_evals} evals", flush=True)
                    break

    print(f"[select] best dev step {best['step']} (dev WER norm {best['dev_wer_norm']})", flush=True)
    model.load_state_dict(best_state, strict=False)
    if cfg.get("save_model", True):
        save_dir = os.path.join(out_dir, "best_model")
        model.save_pretrained(save_dir)
        proc.save_pretrained(save_dir)

    # ---- the ONLY time test data is decoded ----
    del sdn
    results = dict(env=asr.env_info(), config=cfg, selected_step=best["step"], dev=best.get("dev"),
                   train_utts=len(train), dev_utts=len(dev), train_seconds=round(time.time() - t0))
    test_sets = {"sdn_test": data.load_sdn(cfg["split_tsv"], zip_path, splits=("test",), limit=args.limit)["test"],
                 "ookeval": data.load_ookeval(cfg["cache_dir"], limit=args.limit)}
    for name, samples in test_sets.items():
        hyps = decode(model, proc, samples, cfg["eval_batch_size"], use_amp)
        asr.save_hyps(os.path.join(out_dir, f"hyps__{cfg['name']}__{name}.jsonl"), samples, hyps)
        results[name] = metrics.summarize([s["text"] for s in samples], hyps, B=cfg["bootstrap"], seed=cfg["seed"])
        r = results[name]
        print(f"[test] {name}: WER raw {r['wer_raw']} | basic {r['wer_basic']} | norm {r['wer_norm']} {r['wer_norm_ci95']}"
              f" CER norm {r['cer_norm']}", flush=True)
    json.dump(results, open(os.path.join(out_dir, f"{cfg['name']}_metrics.json"), "w", encoding="utf-8"),
              ensure_ascii=False, indent=1)
    print("[done]")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tests/test_textnorm.py
import unittest

from sudanese_asr_eval import full, get_view, raw, whisper_basic
from sudanese_asr_eval.textnorm import ablation_views

CASES = {
    "فِي يوم الأربعاء، أول ديسمبر!": "في يوم الاربعاء اول ديسمبر",
    "ألف و تسعمية تمنية و أربعين": "الف وتسعميه تمنيه واربعين",
    "مدرسة ٣ و ٤": "مدرسه 3 و4",
    "مســـاء الخير؟": "مساء الخير",
    "إلى مسؤول": "الي مسوول",
    "OK يا Ahmed.": "ok يا ahmed",
    "و": "و",
    "": "",
}


class TestFull(unittest.TestCase):
    def test_cases(self):
        for src, want in CASES.items():
            self.assertEqual(full(src), want, src)

    def test_idempotent(self):
        for src in CASES:
            self.assertEqual(full(full(src)), full(src))

    def test_none_safe(self):
        self.assertEqual(full(None), "")
        self.assertEqual(raw(None), "")
        self.assertEqual(whisper_basic(None), "")

    def test_ablation_last_step_is_full(self):
        steps = ablation_views()
        self.assertEqual([label for label, _ in steps], ["raw", "+p/d", "+alef", "+ya", "+ta", "+wa"])
        for src in CASES:
            self.assertEqual(steps[-1][1](src), full(src))


class TestRawBasic(unittest.TestCase):
    def test_raw_only_whitespace(self):
        self.assertEqual(raw("  أهلاً   وسهلاً،\tيا  زول "), "أهلاً وسهلاً، يا زول")

    def test_basic_keeps_letter_variants(self):
        # basic removes marks and punctuation but does NOT unify alef or ta marbuta
        self.assertEqual(whisper_basic("أهلاً، مدرسة!"), "أهلا مدرسة")

    def test_views(self):
        self.assertIs(get_view("norm"), full)
        self.assertIs(get_view("full"), full)
        self.assertIs(get_view("basic"), whisper_basic)
        with self.assertRaises(KeyError):
            get_view("nope")

    def test_basic_matches_transformers(self):
        try:
            from transformers.models.whisper.english_normalizer import BasicTextNormalizer
        except Exception:
            self.skipTest("transformers not installed")
        ref = BasicTextNormalizer()
        for src in list(CASES) + ["فِي (مرحبا) [x] يوم، «الخير»"]:
            self.assertEqual(whisper_basic(src).split(), ref(src).split(), src)


if __name__ == "__main__":
    unittest.main()


In [ ]:
%%writefile /kaggle/working/splits/sdn_split_v1.tsv
# source: Zenodo 10.5281/zenodo.6869079 v1.1 (CC BY 4.0); metadata rows 3549
# dropped: {'unknown_speaker': 182, 'longer_than_30s': 1}
# accepted seed: 354; kept 3366 utts, 3.875 h
# train: 2695 utts, 3.013 h (77.7 %), speakers ['1', '2', '3', '4', '5', '9', '11', '12', '13', '15', '19', '31', '32', '35']
# dev: 302 utts, 0.394 h (10.2 %), speakers ['7', '8', '14', '16', '18', '21', '22', '24', '25', '27', '28', '29', '30', '33', '34']
# test: 369 utts, 0.469 h (12.1 %), speakers ['6', '10', '17', '20', '26']
id	split	speaker	gender	program	dur
hm_01_0001	train	1	Male	hm	2.483
hm_01_0002	train	1	Male	hm	2.483
hm_01_0003	train	2	Male	hm	1.882
hm_01_0004	train	3	Male	hm	0.968
hm_01_0005	train	1	Male	hm	3.449
hm_01_0006	train	3	Male	hm	2.169
hm_01_0007	train	1	Male	hm	19.488
hm_01_0008	train	2	Male	hm	6.166
hm_01_0009	train	3	Male	hm	1.464
hm_01_0010	train	1	Male	hm	1.098
hm_01_0011	train	2	Male	hm	3.554
hm_01_0012	train	1	Male	hm	5.252
hm_01_0013	train	3	Male	hm	23.067
hm_01_0014	train	1	Male	hm	3.345
hm_01_0015	train	2	Male	hm	3.449
hm_01_0016	train	3	Male	hm	7.159
hm_01_0017	train	1	Male	hm	1.464
hm_01_0018	train	3	Male	hm	2.43
hm_01_0019	train	2	Male	hm	4.651
hm_01_0020	train	1	Male	hm	3.867
hm_01_0021	train	2	Male	hm	2.091
hm_01_0022	train	1	Male	hm	3.475
hm_01_0023	train	3	Male	hm	2.117
hm_01_0024	train	1	Male	hm	16.197
hm_01_0025	train	3	Male	hm	2.77
hm_01_0026	train	2	Male	hm	2.195
hm_01_0027	train	1	Male	hm	1.595
hm_01_0028	train	3	Male	hm	4.494
hm_01_0029	train	1	Male	hm	1.542
hm_01_0030	train	3	Male	hm	6.244
hm_02_0001	train	4	Female	hm	2.822
hm_02_0003	train	4	Female	hm	2.039
hm_02_0004	train	4	Female	hm	6.27
hm_02_0005	train	2	Male	hm	1.699
hm_02_0006	train	3	Male	hm	3.71
hm_02_0007	train	2	Male	hm	3.188
hm_02_0009	train	2	Male	hm	2.065
hm_02_0010	train	3	Male	hm	1.699
hm_02_0011	train	2	Male	hm	3.998
hm_02_0012	train	3	Male	hm	3.475
hm_02_0013	train	2	Male	hm	1.542
hm_02_0014	train	3	Male	hm	4.964
hm_02_0015	train	2	Male	hm	5.33
hm_02_0016	train	3	Male	hm	3.005
hm_02_0017	train	2	Male	hm	1.542
hm_02_0019	train	2	Male	hm	3.998
hm_02_0020	train	3	Male	hm	4.886
hm_02_0021	train	3	Male	hm	2.77
hm_02_0022	train	3	Male	hm	3.214
hm_02_0023	train	2	Male	hm	2.378
hm_02_0024	train	3	Male	hm	3.606
hm_02_0025	train	2	Male	hm	2.274
hm_02_0026	train	3	Male	hm	2.248
hm_02_0027	train	2	Male	hm	2.012
hm_02_0028	train	3	Male	hm	2.43
hm_02_0029	train	2	Male	hm	3.815
hm_02_0030	train	3	Male	hm	1.856
hm_02_0031	train	2	Male	hm	1.934
hm_02_0032	train	2	Male	hm	2.692
hm_02_0033	train	5	Male	hm	1.177
hm_02_0034	train	3	Male	hm	2.117
hm_02_0035	train	5	Male	hm	3.893
hm_02_0036	train	2	Male	hm	4.155
hm_02_0037	train	5	Female	hm	3.136
hm_02_0038	train	2	Male	hm	2.718
hm_02_0039	train	5	Female	hm	1.647
hm_02_0040	train	3	Male	hm	8.935
hm_02_0041	train	5	Female	hm	3.345
hm_02_0042	train	3	Male	hm	5.043
hm_02_0043	train	5	Female	hm	4.938
hm_02_0044	train	2	Male	hm	5.199
hm_02_0045	train	3	Male	hm	3.449
hm_02_0046	train	5	Female	hm	5.356
hm_02_0047	train	3	Male	hm	2.796
hm_02_0048	train	5	Female	hm	0.759
hm_02_0049	train	3	Male	hm	1.595
hm_02_0050	train	3	Male	hm	3.005
hm_02_0051	train	2	Male	hm	5.957
hm_02_0053	train	5	Female	hm	4.311
hm_02_0054	train	3	Male	hm	5.983
hm_02_0055	train	2	Male	hm	1.96
hm_02_0056	train	3	Male	hm	1.177
hm_02_0057	train	2	Male	hm	3.345
hm_02_0058	train	3	Male	hm	8.073
hm_02_0059	train	3	Male	hm	2.143
hm_02_0060	train	2	Male	hm	1.49
hm_02_0061	train	3	Male	hm	3.71
hm_02_0062	train	2	Male	hm	0.994
hm_02_0063	train	3	Male	hm	5.199
hm_02_0064	train	2	Male	hm	4.024
hm_02_0065	train	3	Male	hm	1.412
hm_02_0066	train	3	Male	hm	6.061
hm_02_0067	train	2	Male	hm	5.879
hm_02_0068	train	3	Male	hm	4.442
hm_02_0069	train	2	Male	hm	7.838
hm_02_0070	train	2	Male	hm	6.192
hm_02_0071	train	3	Male	hm	2.143
hm_02_0072	test	6	Female	hm	2.117
hm_02_0073	train	3	Male	hm	3.475
hm_02_0074	test	6	Female	hm	1.072
hm_02_0075	train	2	Male	hm	5.931
hm_02_0076	test	6	Female	hm	8.412
hm_02_0077	train	3	Male	hm	4.181
hm_02_0078	test	6	Female	hm	6.349
hm_02_0079	train	2	Male	hm	5.435
hm_02_0080	test	6	Female	hm	4.599
hm_02_0081	train	1	Male	hm	4.886
hm_02_0082	train	3	Male	hm	1.203
hm_02_0083	train	1	Male	hm	3.24
hm_02_0084	train	2	Male	hm	1.804
hm_02_0085	train	3	Male	hm	2.352
hm_02_0086	train	1	Male	hm	0.968
hm_02_0087	train	3	Male	hm	1.777
hm_02_0088	train	1	Male	hm	2.979
hm_02_0089	train	2	Male	hm	5.408
hm_02_0090	train	1	Male	hm	5.408
hm_02_0091	train	3	Male	hm	5.017
hm_02_0092	train	1	Male	hm	4.964
hm_02_0093	train	4	Female	hm	4.677
hm_03_0001	train	3	Male	hm	4.86
hm_03_0002	train	3	Male	hm	9.301
hm_03_0003	train	2	Male	hm	3.162
hm_03_0004	train	3	Male	hm	5.226
hm_03_0005	train	2	Male	hm	1.02
hm_03_0006	train	3	Male	hm	1.542
hm_03_0007	train	2	Male	hm	1.124
hm_03_0008	train	3	Male	hm	1.516
hm_03_0009	train	5	Female	hm	0.915
hm_03_0010	train	3	Male	hm	2.666
hm_03_0011	train	2	Male	hm	9.353
hm_03_0012	train	3	Male	hm	8.543
hm_03_0013	train	5	Female	hm	3.214
hm_03_0014	train	1	Male	hm	1.307
hm_03_0015	train	5	Female	hm	5.983
hm_03_0016	train	3	Male	hm	4.546
hm_03_0017	train	2	Male	hm	12.383
hm_03_0018	train	1	Male	hm	3.606
hm_03_0020	train	1	Male	hm	1.177
hm_03_0022	train	1	Male	hm	1.072
hm_03_0024	train	2	Male	hm	2.274
hm_03_0025	train	3	Male	hm	3.266
hm_03_0026	train	2	Male	hm	4.024
hm_03_0027	train	3	Male	hm	4.181
hm_03_0028	train	2	Male	hm	7.054
hm_03_0029	train	5	Female	hm	2.77
hm_03_0030	train	2	Male	hm	1.934
hm_03_0031	train	1	Male	hm	6.14
hm_03_0032	train	3	Male	hm	5.487
hm_03_0033	train	5	Female	hm	3.763
hm_03_0035	train	5	Female	hm	3.554
hm_03_0036	train	3	Male	hm	1.516
hm_03_0037	train	5	Female	hm	1.333
hm_03_0038	train	1	Male	hm	6.532
hm_03_0039	train	5	Female	hm	1.49
hm_04_0001	train	1	Male	hm	1.96
hm_04_0002	train	3	Male	hm	2.378
hm_04_0003	train	2	Male	hm	2.901
hm_04_0004	train	1	Male	hm	3.084
hm_04_0005	train	3	Male	hm	2.143
hm_04_0006	train	1	Male	hm	9.457
hm_04_0007	train	3	Male	hm	2.352
hm_04_0008	train	1	Male	hm	5.748
hm_04_0009	train	2	Male	hm	6.192
hm_04_0010	train	5	Female	hm	6.845
hm_04_0011	train	3	Male	hm	3.24
hm_04_0012	train	1	Male	hm	13.402
hm_04_0013	train	5	Female	hm	3.946
hm_04_0014	train	1	Male	hm	2.901
hm_04_0015	train	5	Female	hm	4.259
hm_04_0016	train	1	Male	hm	9.196
hm_04_0017	train	3	Male	hm	2.195
hm_04_0018	train	2	Male	hm	7.42
hm_04_0019	train	1	Male	hm	1.751
hm_04_0020	train	3	Male	hm	3.345
hm_04_0021	train	1	Male	hm	7.106
hm_04_0022	train	2	Male	hm	1.83
hm_04_0023	train	3	Male	hm	2.796
hm_04_0024	train	1	Male	hm	29.885
hm_04_0026	train	1	Male	hm	12.2
hm_04_0028	train	1	Male	hm	9.823
hm_04_0029	train	1	Male	hm	20.664
hm_04_0030	train	2	Male	hm	3.554
hm_04_0031	train	3	Male	hm	2.613
hm_04_0032	train	1	Male	hm	13.846
hm_04_0033	train	2	Male	hm	5.121
hm_04_0034	train	5	Female	hm	6.819
hm_04_0035	dev	7	Male	hm	11.181
hm_04_0037	dev	7	Male	hm	9.849
hm_04_0038	train	3	Male	hm	3.919
hm_04_0039	train	2	Male	hm	4.938
hm_04_0040	train	3	Male	hm	4.781
hm_04_0041	train	3	Male	hm	8.83
hm_04_0042	train	2	Male	hm	5.67
hm_04_0043	train	3	Male	hm	1.229
hm_04_0044	train	2	Male	hm	10.45
hm_04_0045	train	3	Male	hm	2.718
hm_04_0046	train	2	Male	hm	7.603
hm_04_0047	train	2	Male	hm	2.535
hm_04_0048	train	3	Male	hm	4.364
hm_04_0049	train	2	Male	hm	0.576
hm_04_0050	train	3	Male	hm	2.43
hm_04_0051	train	2	Male	hm	12.252
hm_04_0052	train	3	Male	hm	3.71
hm_04_0053	train	2	Male	hm	4.233
hm_04_0054	train	3	Male	hm	4.834
hm_04_0055	train	5	Female	hm	6.218
hm_04_0056	train	5	Female	hm	5.879
hm_04_0057	train	2	Male	hm	0.602
hm_04_0058	train	5	Female	hm	3.005
hm_04_0059	train	3	Male	hm	1.568
hm_04_0060	train	5	Female	hm	3.11
hm_04_0061	train	3	Male	hm	4.233
hm_04_0062	train	5	Female	hm	13.376
hm_04_0063	train	3	Male	hm	9.248
hm_04_0064	train	2	Male	hm	1.725
hm_04_0065	train	3	Male	hm	8.439
hm_04_0066	train	3	Male	hm	5.408
hm_04_0067	train	2	Male	hm	5.983
hm_04_0068	train	3	Male	hm	5.852
hm_04_0069	train	2	Male	hm	0.706
hm_04_0070	train	3	Male	hm	7.289
hm_04_0071	train	2	Male	hm	0.524
hm_04_0072	train	3	Male	hm	13.82
hm_04_0073	train	2	Male	hm	2.796
hm_04_0074	train	5	Female	hm	5.591
hm_04_0075	dev	7	Male	hm	6.453
hm_04_0076	train	4	Female	hm	12.357
hm_04_0077	train	4	Female	hm	10.293
hm_05_0001	train	2	Male	hm	2.901
hm_05_0002	train	3	Male	hm	2.587
hm_05_0003	train	2	Male	hm	1.568
hm_05_0004	train	3	Male	hm	7.315
hm_05_0006	train	5	Female	hm	5.905
hm_05_0008	train	3	Male	hm	6.662
hm_05_0009	train	2	Male	hm	2.639
hm_05_0010	train	2	Male	hm	7.394
hm_05_0011	train	3	Male	hm	3.867
hm_05_0012	train	2	Male	hm	5.565
hm_05_0013	train	3	Male	hm	5.278
hm_05_0014	train	2	Male	hm	2.378
hm_05_0015	train	3	Male	hm	2.848
hm_05_0016	train	2	Male	hm	3.371
hm_05_0018	train	3	Male	hm	4.651
hm_05_0019	train	3	Male	hm	2.509
hm_05_0020	train	2	Male	hm	2.169
hm_05_0021	train	3	Male	hm	2.509
hm_05_0022	train	2	Male	hm	4.259
hm_05_0023	train	2	Male	hm	3.057
hm_05_0024	train	3	Male	hm	4.442
hm_05_0025	train	2	Male	hm	2.065
hm_05_0026	train	3	Male	hm	4.599
hm_05_0027	train	2	Male	hm	4.703
hm_05_0028	train	3	Male	hm	6.166
hm_05_0029	train	2	Male	hm	1.15
hm_05_0030	train	3	Male	hm	0.785
hm_05_0031	train	2	Male	hm	2.143
hm_05_0032	train	3	Male	hm	4.781
hm_05_0033	train	2	Male	hm	4.285
hm_05_0034	train	3	Male	hm	3.632
hm_05_0035	train	2	Male	hm	2.613
hm_05_0036	train	5	Female	hm	3.371
hm_05_0037	train	3	Male	hm	1.725
hm_05_0038	train	5	Female	hm	0.68
hm_05_0039	train	3	Male	hm	1.83
hm_05_0040	train	5	Female	hm	1.804
hm_05_0041	train	3	Male	hm	2.274
hm_05_0042	train	5	Female	hm	2.979
hm_05_0043	train	5	Female	hm	0.889
hm_05_0044	train	2	Male	hm	3.972
hm_05_0045	train	5	Female	hm	1.124
hm_05_0046	train	2	Male	hm	4.808
hm_05_0047	train	5	Female	hm	2.457
hm_05_0048	train	3	Male	hm	1.464
hm_05_0049	train	2	Male	hm	4.311
hm_05_0050	train	5	Female	hm	1.255
hm_05_0051	train	5	Female	hm	5.382
hm_05_0052	train	3	Male	hm	1.83
hm_05_0053	train	5	Female	hm	1.804
hm_05_0055	train	2	Male	hm	12.592
hm_05_0056	train	3	Male	hm	10.659
hm_05_0057	train	2	Male	hm	8.674
hm_05_0058	train	3	Male	hm	11.26
hm_05_0059	train	2	Male	hm	6.114
hm_05_0060	train	3	Male	hm	2.639
hm_05_0061	train	2	Male	hm	4.39
hm_05_0062	train	1	Male	hm	0.759
hm_05_0063	train	2	Male	hm	2.195
hm_05_0064	train	1	Male	hm	1.83
hm_05_0065	train	3	Male	hm	3.423
hm_05_0066	train	1	Male	hm	1.882
hm_05_0067	train	3	Male	hm	2.221
hm_05_0068	train	1	Male	hm	1.359
hm_05_0069	train	2	Male	hm	2.404
hm_05_0070	train	1	Male	hm	1.438
hm_05_0071	train	3	Male	hm	5.905
hm_05_0072	train	1	Male	hm	2.613
hm_05_0073	train	3	Male	hm	2.326
hm_05_0074	train	1	Male	hm	1.098
hm_05_0075	train	3	Male	hm	4.233
hm_05_0076	train	2	Male	hm	3.737
hm_05_0077	train	1	Male	hm	4.572
hm_05_0078	train	3	Male	hm	3.475
hm_06_0001	train	3	Male	hm	2.822
hm_06_0002	train	2	Male	hm	2.953
hm_06_0003	train	3	Male	hm	2.3
hm_06_0004	train	2	Male	hm	3.71
hm_06_0005	train	2	Male	hm	2.169
hm_06_0006	train	3	Male	hm	8.648
hm_06_0007	train	2	Male	hm	1.882
hm_06_0008	train	3	Male	hm	1.595
hm_06_0009	train	2	Male	hm	1.412
hm_06_0010	train	3	Male	hm	2.169
hm_06_0011	train	2	Male	hm	8.177
hm_06_0012	train	3	Male	hm	6.636
hm_06_0013	train	2	Male	hm	5.461
hm_06_0014	train	3	Male	hm	8.987
hm_06_0015	train	2	Male	hm	2.639
hm_06_0016	train	3	Male	hm	1.49
hm_06_0017	train	2	Male	hm	13.898
hm_06_0018	train	3	Male	hm	3.214
hm_06_0019	train	2	Male	hm	1.673
hm_06_0020	train	3	Male	hm	2.901
hm_06_0021	train	2	Male	hm	2.352
hm_06_0022	train	3	Male	hm	10.032
hm_06_0023	train	2	Male	hm	3.501
hm_06_0024	train	3	Male	hm	9.875
hm_06_0025	train	2	Male	hm	1.359
hm_06_0026	train	5	Female	hm	1.098
hm_06_0027	train	2	Male	hm	1.83
hm_06_0028	train	5	Female	hm	1.124
hm_06_0029	train	3	Male	hm	1.229
hm_06_0030	train	5	Female	hm	1.96
hm_06_0031	train	2	Male	hm	4.834
hm_06_0032	train	5	Female	hm	1.568
hm_06_0033	train	3	Male	hm	1.464
hm_06_0034	train	5	Male	hm	2.822
hm_06_0035	train	2	Male	hm	2.169
hm_06_0036	dev	8	Male	hm	3.972
hm_06_0037	train	2	Male	hm	5.226
hm_06_0038	dev	8	Male	hm	2.117
hm_06_0039	train	3	Male	hm	3.789
hm_06_0040	dev	8	Male	hm	2.065
hm_06_0041	train	3	Male	hm	2.535
hm_06_0042	dev	8	Male	hm	13.01
hm_06_0043	train	3	Male	hm	1.908
hm_06_0044	train	2	Male	hm	2.927
hm_06_0045	dev	8	Male	hm	6.688
hm_06_0046	train	3	Male	hm	1.124
hm_06_0047	train	5	Male	hm	5.539
hm_06_0048	train	3	Male	hm	1.412
hm_06_0049	train	5	Female	hm	4.285
hm_06_0050	train	3	Male	hm	6.297
hm_06_0051	train	3	Male	hm	4.39
hm_06_0052	train	5	Female	hm	1.359
hm_06_0053	dev	8	Male	hm	17.372
hm_06_0054	train	3	Male	hm	3.919
hm_06_0055	train	2	Male	hm	2.43
hm_06_0056	test	6	Female	hm	1.02
hm_06_0057	train	3	Male	hm	2.221
hm_06_0058	train	2	Male	hm	1.307
hm_06_0059	test	6	Female	hm	2.535
hm_06_0060	train	3	Male	hm	4.285
hm_06_0061	train	2	Male	hm	2.979
hm_06_0062	test	6	Female	hm	1.281
hm_06_0063	train	3	Male	hm	2.535
hm_06_0064	test	6	Female	hm	2.666
hm_06_0065	train	3	Male	hm	1.229
hm_06_0066	test	6	Female	hm	1.908
hm_06_0067	train	3	Male	hm	5.722
hm_06_0068	test	6	Female	hm	2.483
hm_06_0069	train	3	Male	hm	1.124
hm_06_0070	test	6	Female	hm	1.359
hm_06_0071	train	3	Male	hm	5.931
hm_06_0072	test	6	Female	hm	3.136
hm_06_0073	train	3	Male	hm	2.613
hm_06_0074	test	6	Female	hm	3.423
hm_06_0075	train	3	Male	hm	2.221
hm_06_0076	test	6	Female	hm	5.539
hm_06_0077	train	3	Male	hm	3.475
hm_06_0078	test	6	Female	hm	3.737
hm_06_0079	train	3	Male	hm	2.561
hm_06_0080	test	6	Female	hm	1.621
hm_06_0081	train	3	Male	hm	1.725
hm_06_0082	test	6	Female	hm	3.136
hm_06_0083	train	3	Male	hm	1.359
hm_06_0084	train	3	Male	hm	3.867
hm_06_0085	train	3	Male	hm	4.886
hm_06_0086	train	3	Male	hm	5.539
hm_06_0087	train	3	Male	hm	5.983
hm_06_0088	train	2	Male	hm	2.979
hm_06_0089	train	3	Male	hm	5.121
hm_06_0090	test	6	Female	hm	2.822
hm_06_0091	train	3	Male	hm	3.214
hm_06_0092	test	6	Female	hm	3.919
hm_06_0093	train	5	Female	hm	3.371
hm_06_0094	test	6	Female	hm	12.906
hm_06_0095	train	3	Male	hm	2.483
hm_06_0097	train	9	Female	hm	6.95
hm_07_0001	train	1	Male	hm	4.912
hm_07_0002	test	6	Female	hm	2.326
hm_07_0003	train	1	Male	hm	4.729
hm_07_0004	test	6	Female	hm	4.337
hm_07_0005	train	1	Male	hm	7.707
hm_07_0006	test	6	Female	hm	2.796
hm_07_0007	train	1	Male	hm	7.211
hm_07_0008	test	6	Female	hm	1.177
hm_07_0009	train	1	Male	hm	3.867
hm_07_0010	test	6	Female	hm	1.751
hm_07_0011	train	1	Male	hm	2.796
hm_07_0012	train	2	Male	hm	6.27
hm_07_0013	train	3	Male	hm	8.857
hm_07_0014	train	2	Male	hm	14.708
hm_07_0015	train	3	Male	hm	3.005
hm_07_0016	train	2	Male	hm	4.808
hm_07_0017	train	3	Male	hm	6.845
hm_07_0018	train	2	Male	hm	3.292
hm_07_0019	train	3	Male	hm	3.763
hm_07_0020	train	2	Male	hm	2.039
hm_07_0021	train	3	Male	hm	0.889
hm_07_0022	train	2	Male	hm	3.867
hm_07_0023	train	3	Male	hm	4.729
hm_07_0024	train	2	Male	hm	3.292
hm_07_0025	train	3	Male	hm	4.233
hm_07_0026	train	2	Male	hm	2.718
hm_07_0027	train	3	Male	hm	2.901
hm_07_0028	train	2	Male	hm	4.442
hm_07_0029	train	3	Male	hm	2.039
hm_07_0030	train	2	Male	hm	5.487
hm_07_0031	train	3	Male	hm	5.591
hm_07_0032	train	2	Male	hm	2.117
hm_07_0033	test	6	Female	hm	3.475
hm_07_0034	train	1	Male	hm	4.442
hm_07_0035	test	6	Female	hm	1.647
hm_07_0036	train	1	Male	hm	4.337
hm_07_0037	test	6	Female	hm	3.658
hm_07_0038	train	3	Male	hm	4.808
hm_07_0039	train	1	Male	hm	8.674
hm_07_0040	train	2	Male	hm	3.763
hm_07_0041	train	1	Male	hm	4.912
hm_07_0042	train	3	Male	hm	1.647
hm_07_0043	train	1	Male	hm	6.061
hm_07_0044	train	3	Male	hm	3.188
hm_07_0045	train	2	Male	hm	5.774
hm_07_0046	train	3	Male	hm	1.542
hm_07_0047	train	1	Male	hm	3.371
hm_07_0048	train	3	Male	hm	2.117
hm_07_0049	train	2	Male	hm	7.707
hm_07_0050	train	3	Male	hm	3.658
hm_07_0051	train	1	Male	hm	16.171
hm_07_0052	train	2	Male	hm	3.084
hm_07_0053	train	3	Male	hm	5.983
hm_07_0054	test	6	Female	hm	7.812
hm_07_0055	train	1	Male	hm	2.718
hm_07_0056	test	6	Female	hm	2.613
hm_07_0057	train	3	Male	hm	3.005
hm_07_0058	train	2	Male	hm	2.718
hm_07_0059	train	1	Male	hm	5.017
hm_07_0060	test	6	Female	hm	3.188
hm_07_0061	train	1	Male	hm	1.542
hm_07_0062	test	6	Female	hm	1.699
hm_07_0063	train	3	Male	hm	5.983
hm_07_0064	train	2	Male	hm	4.337
hm_07_0065	train	3	Male	hm	7.185
hm_07_0066	train	2	Male	hm	4.442
hm_07_0067	test	6	Female	hm	1.02
hm_07_0069	train	2	Male	hm	3.371
hm_07_0070	test	10	Female	hm	2.039
hm_07_0072	test	10	Female	hm	2.848
hm_07_0074	test	10	Female	hm	2.039
hm_07_0075	train	3	Male	hm	4.912
hm_07_0076	test	10	Female	hm	1.072
hm_07_0077	train	2	Male	hm	3.867
hm_07_0078	train	3	Male	hm	2.117
hm_07_0079	test	10	Female	hm	1.02
hm_07_0080	train	2	Male	hm	5.774
hm_07_0081	train	3	Male	hm	5.356
hm_07_0082	test	10	Female	hm	2.744
hm_07_0083	train	2	Male	hm	5.147
hm_07_0084	test	10	Female	hm	3.188
hm_07_0085	train	3	Male	hm	3.946
hm_07_0086	test	10	Female	hm	3.188
hm_07_0087	train	2	Male	hm	3.423
hm_07_0088	test	10	Female	hm	2.901
hm_07_0089	train	3	Male	hm	4.39
hm_08_0001	train	2	Male	hm	1.725
hm_08_0002	train	5	Female	hm	5.043
hm_08_0003	train	2	Male	hm	5.513
hm_08_0004	train	3	Male	hm	3.684
hm_08_0005	train	5	Female	hm	5.487
hm_08_0006	train	2	Male	hm	4.572
hm_08_0007	train	5	Female	hm	6.035
hm_08_0008	train	2	Male	hm	8.308
hm_08_0009	train	5	Female	hm	5.852
hm_08_0010	train	3	Male	hm	1.777
hm_08_0011	test	6	Female	hm	1.908
hm_08_0012	train	5	Female	hm	1.542
hm_08_0013	test	6	Female	hm	0.915
hm_08_0014	train	5	Female	hm	1.438
hm_08_0015	test	6	Female	hm	3.292
hm_08_0016	train	5	Female	hm	2.901
hm_08_0017	test	6	Female	hm	10.032
hm_08_0018	train	5	Female	hm	5.539
hm_08_0019	test	6	Female	hm	2.639
hm_08_0020	train	5	Female	hm	9.039
hm_08_0021	test	6	Female	hm	9.692
hm_08_0022	train	5	Female	hm	2.352
hm_08_0023	train	1	Male	hm	1.255
hm_08_0024	train	3	Male	hm	4.599
hm_08_0025	train	1	Male	hm	0.968
hm_08_0026	train	3	Male	hm	3.449
hm_08_0027	train	1	Male	hm	5.435
hm_08_0028	train	3	Male	hm	5.852
hm_08_0029	train	1	Male	hm	0.994
hm_08_0030	train	3	Male	hm	1.673
hm_08_0031	train	1	Male	hm	0.785
hm_08_0032	train	3	Male	hm	5.67
hm_08_0033	train	1	Male	hm	7.89
hm_08_0034	train	3	Male	hm	1.15
hm_08_0035	train	1	Male	hm	7.942
hm_08_0036	train	3	Male	hm	4.442
hm_08_0037	train	1	Male	hm	8.23
hm_08_0038	train	3	Male	hm	3.58
hm_08_0039	train	2	Male	hm	3.658
hm_08_0040	train	5	Female	hm	1.568
hm_08_0041	train	2	Male	hm	5.069
hm_08_0042	train	5	Female	hm	1.307
hm_08_0043	train	2	Male	hm	5.957
hm_08_0044	train	5	Female	hm	2.065
hm_08_0045	train	2	Male	hm	3.632
hm_08_0046	train	5	Female	hm	1.438
hm_08_0047	train	2	Male	hm	2.639
hm_08_0048	train	5	Female	hm	1.307
hm_08_0049	train	3	Male	hm	2.378
hm_08_0050	train	5	Female	hm	2.143
hm_08_0051	train	3	Male	hm	2.274
hm_08_0052	train	2	Male	hm	3.162
hm_08_0053	train	5	Female	hm	2.143
hm_08_0054	train	3	Male	hm	1.386
hm_08_0055	train	2	Male	hm	4.442
hm_08_0056	train	5	Female	hm	2.117
hm_08_0057	train	3	Male	hm	2.613
hm_08_0058	train	2	Male	hm	2.822
hm_08_0059	train	1	Male	hm	1.49
hm_08_0060	train	5	Female	hm	4.155
hm_08_0061	train	3	Male	hm	3.658
hm_08_0062	train	2	Male	hm	1.804
hm_08_0063	train	1	Male	hm	0.759
hm_08_0064	train	2	Male	hm	3.449
hm_08_0065	train	1	Male	hm	1.777
hm_08_0066	train	5	Female	hm	2.3
hm_08_0067	train	1	Male	hm	3.684
hm_08_0068	train	3	Male	hm	5.644
hm_09_0001	train	5	Female	hm	2.744
hm_09_0003	train	5	Female	hm	6.166
hm_09_0004	train	3	Male	hm	1.15
hm_09_0005	train	2	Male	hm	2.143
hm_09_0006	train	3	Male	hm	3.737
hm_09_0007	train	2	Male	hm	3.24
hm_09_0008	train	3	Male	hm	2.352
hm_09_0009	train	2	Male	hm	5.879
hm_09_0010	train	3	Male	hm	0.968
hm_09_0011	train	5	Female	hm	1.464
hm_09_0012	train	2	Male	hm	2.77
hm_09_0013	train	5	Female	hm	2.248
hm_09_0014	train	3	Male	hm	1.516
hm_09_0015	train	5	Female	hm	3.841
hm_09_0016	train	2	Male	hm	2.535
hm_09_0017	train	5	Female	hm	0.654
hm_09_0018	train	3	Male	hm	6.14
hm_09_0019	train	5	Female	hm	1.098
hm_09_0020	train	2	Male	hm	0.994
hm_09_0021	train	5	Female	hm	2.248
hm_09_0022	train	3	Male	hm	2.169
hm_09_0023	train	1	Male	hm	1.438
hm_09_0024	train	2	Male	hm	1.908
hm_09_0025	train	3	Male	hm	3.005
hm_09_0026	train	1	Male	hm	1.568
hm_09_0027	train	2	Male	hm	3.319
hm_09_0028	train	3	Male	hm	1.49
hm_09_0029	train	1	Male	hm	3.554
hm_09_0030	train	2	Male	hm	1.229
hm_09_0031	train	3	Male	hm	1.438
hm_09_0032	train	1	Male	hm	6.819
hm_09_0033	train	3	Male	hm	2.143
hm_09_0034	train	1	Male	hm	3.005
hm_09_0035	train	3	Male	hm	1.83
hm_09_0036	train	1	Male	hm	0.837
hm_09_0037	train	2	Male	hm	0.524
hm_09_0038	train	1	Male	hm	2.535
hm_09_0039	train	2	Male	hm	0.628
hm_09_0040	train	1	Male	hm	2.248
hm_09_0041	train	2	Male	hm	1.777
hm_09_0042	train	3	Male	hm	3.737
hm_09_0043	train	1	Male	hm	3.554
hm_09_0044	train	1	Male	hm	13.193
hm_09_0045	train	2	Male	hm	3.292
hm_09_0046	train	1	Male	hm	2.091
hm_09_0047	train	3	Male	hm	1.621
hm_09_0048	train	1	Male	hm	0.524
hm_09_0049	train	3	Male	hm	1.046
hm_09_0050	train	1	Male	hm	0.524
hm_09_0051	train	3	Male	hm	2.692
hm_09_0052	train	1	Male	hm	2.091
hm_09_0054	train	1	Male	hm	2.509
hm_09_0056	train	1	Male	hm	0.968
hm_09_0058	train	1	Male	hm	1.934
hm_09_0059	train	3	Male	hm	4.546
hm_09_0060	train	2	Male	hm	2.927
hm_09_0061	train	1	Male	hm	8.543
hm_09_0062	train	5	Female	hm	2.274
hm_09_0063	train	3	Male	hm	6.793
hm_09_0064	train	2	Male	hm	3.057
hm_09_0065	train	3	Male	hm	2.039
hm_09_0066	train	3	Male	hm	2.43
hm_09_0067	train	1	Male	hm	2.169
hm_09_0068	train	2	Male	hm	3.319
hm_09_0069	train	1	Male	hm	1.568
hm_09_0070	train	3	Male	hm	5.957
hm_09_0071	train	2	Male	hm	3.11
hm_09_0072	train	1	Male	hm	4.86
hm_09_0073	train	2	Male	hm	4.024
hm_09_0074	train	2	Male	hm	3.345
hm_09_0075	train	3	Male	hm	2.979
hm_09_0076	train	2	Male	hm	2.509
hm_09_0077	train	5	Female	hm	1.177
hm_09_0078	train	2	Male	hm	2.718
hm_09_0079	train	5	Female	hm	4.155
hm_09_0080	train	3	Male	hm	1.595
hm_09_0081	train	5	Female	hm	8.491
hm_09_0082	train	3	Male	hm	1.229
hm_09_0083	train	2	Male	hm	0.602
hm_09_0084	train	3	Male	hm	2.483
hm_09_0085	train	3	Male	hm	6.009
hm_09_0086	train	2	Male	hm	2.143
hm_09_0087	train	3	Male	hm	1.542
hm_09_0088	train	5	Female	hm	4.52
hm_09_0089	train	3	Male	hm	2.143
hm_09_0090	train	5	Female	hm	1.412
hm_09_0093	train	3	Male	hm	0.863
hm_09_0094	train	2	Male	hm	1.359
hm_09_0095	train	5	Female	hm	1.438
hm_09_0096	train	1	Male	hm	0.915
hm_09_0097	train	5	Female	hm	2.091
hm_09_0099	train	5	Female	hm	5.983
hm_10_0001	train	2	Male	hm	1.02
hm_10_0002	train	3	Male	hm	1.412
hm_10_0003	train	2	Male	hm	2.457
hm_10_0004	train	3	Male	hm	1.804
hm_10_0005	train	5	Female	hm	2.457
hm_10_0006	train	2	Male	hm	3.005
hm_10_0007	train	5	Female	hm	0.968
hm_10_0008	train	2	Male	hm	5.408
hm_10_0009	train	3	Male	hm	5.435
hm_10_0010	train	5	Female	hm	3.606
hm_10_0011	train	2	Male	hm	2.77
hm_10_0012	train	5	Female	hm	3.763
hm_10_0013	train	2	Male	hm	1.699
hm_10_0014	train	3	Male	hm	5.487
hm_10_0015	train	2	Male	hm	5.304
hm_10_0016	train	3	Male	hm	1.307
hm_10_0017	train	5	Female	hm	1.856
hm_10_0018	train	2	Male	hm	3.24
hm_10_0019	train	3	Male	hm	4.808
hm_10_0020	train	2	Male	hm	1.673
hm_10_0021	train	3	Male	hm	2.848
hm_10_0022	train	2	Male	hm	2.561
hm_10_0023	train	3	Male	hm	2.065
hm_10_0024	train	2	Male	hm	2.352
hm_10_0025	train	2	Male	hm	1.882
hm_10_0026	train	5	Female	hm	2.274
hm_10_0027	train	2	Male	hm	1.281
hm_10_0028	train	5	Female	hm	3.057
hm_10_0029	train	2	Male	hm	2.43
hm_10_0030	train	5	Female	hm	1.934
hm_10_0031	train	3	Male	hm	2.221
hm_10_0032	train	2	Male	hm	8.23
hm_10_0033	train	3	Male	hm	2.848
hm_10_0034	train	1	Male	hm	1.124
hm_10_0036	train	5	Female	hm	1.777
hm_10_0037	train	1	Male	hm	3.084
hm_10_0038	train	5	Female	hm	1.856
hm_10_0039	train	1	Male	hm	4.912
hm_10_0040	train	2	Male	hm	0.889
hm_10_0041	train	5	Female	hm	2.221
hm_10_0042	train	1	Male	hm	10.11
hm_10_0043	train	3	Male	hm	1.934
hm_10_0044	train	1	Male	hm	1.386
hm_10_0045	train	2	Male	hm	2.901
hm_10_0046	train	5	Female	hm	3.815
hm_10_0047	train	3	Male	hm	2.195
hm_10_0048	train	2	Male	hm	4.364
hm_10_0049	train	5	Female	hm	1.412
hm_10_0050	train	1	Male	hm	1.333
hm_10_0051	train	2	Male	hm	2.117
hm_10_0052	train	3	Male	hm	3.136
hm_10_0053	train	1	Male	hm	3.449
hm_10_0054	train	2	Male	hm	7.211
hm_10_0055	train	1	Male	hm	3.919
hm_10_0056	train	3	Male	hm	4.05
hm_10_0057	train	1	Male	hm	1.751
hm_10_0058	train	2	Male	hm	2.848
hm_10_0059	train	1	Male	hm	2.352
hm_10_0060	train	3	Male	hm	3.475
hm_10_0061	train	1	Male	hm	2.039
hm_10_0062	train	2	Male	hm	3.71
hm_10_0063	train	1	Male	hm	1.333
hm_10_0064	train	3	Male	hm	2.875
hm_10_0065	train	2	Male	hm	2.848
hm_10_0066	train	3	Male	hm	1.359
hm_10_0067	train	3	Male	hm	1.359
hm_10_0068	train	1	Male	hm	0.706
hm_10_0069	train	3	Male	hm	2.483
hm_10_0070	train	1	Male	hm	9.928
hm_10_0071	train	3	Male	hm	5.252
hm_10_0072	train	1	Male	hm	6.819
hm_10_0073	train	3	Male	hm	1.333
hm_10_0074	train	1	Male	hm	8.857
hm_10_0075	train	1	Male	hm	16.119
hm_10_0076	train	1	Male	hm	12.514
hm_10_0077	train	3	Male	hm	1.516
hm_10_0078	train	1	Male	hm	1.281
hm_10_0079	train	2	Male	hm	2.796
hm_10_0080	train	1	Male	hm	15.466
hm_10_0082	test	6	Female	hm	1.02
hm_10_0084	test	6	Female	hm	0.994
hm_10_0086	test	6	Female	hm	1.621
hm_10_0087	train	3	Male	hm	3.345
hm_10_0088	test	6	Female	hm	1.777
hm_10_0089	train	2	Male	hm	5.644
hm_10_0090	test	6	Female	hm	3.684
hm_10_0091	train	3	Male	hm	3.397
hm_10_0092	test	6	Female	hm	1.595
hm_10_0093	train	2	Male	hm	5.774
hm_10_0094	test	6	Female	hm	4.259
hm_10_0096	test	6	Female	hm	0.941
hm_10_0098	train	4	Female	hm	5.199
hm_11_0002	train	4	Female	hm	6.558
hm_11_0004	train	2	Male	hm	3.658
hm_11_0005	train	3	Male	hm	3.998
hm_11_0006	train	2	Male	hm	4.181
hm_11_0007	train	3	Male	hm	4.024
hm_11_0008	train	2	Male	hm	1.986
hm_11_0009	train	3	Male	hm	2.012
hm_11_0010	train	2	Male	hm	1.908
hm_11_0011	train	3	Male	hm	3.71
hm_11_0012	train	2	Male	hm	2.43
hm_11_0013	train	3	Male	hm	2.535
hm_11_0017	train	3	Male	hm	3.789
hm_11_0018	train	2	Male	hm	4.703
hm_11_0019	train	1	Male	hm	6.061
hm_11_0020	train	3	Male	hm	2.457
hm_11_0021	train	2	Male	hm	2.535
hm_11_0022	train	1	Male	hm	0.941
hm_11_0023	train	1	Male	hm	2.509
hm_11_0025	train	1	Male	hm	2.953
hm_11_0026	train	3	Male	hm	3.528
hm_11_0027	train	1	Male	hm	1.96
hm_11_0028	train	2	Male	hm	1.882
hm_11_0029	train	3	Male	hm	2.953
hm_11_0030	train	2	Male	hm	7.786
hm_11_0031	train	3	Male	hm	4.416
hm_11_0032	train	1	Male	hm	1.699
hm_11_0033	train	2	Male	hm	8.177
hm_11_0034	train	1	Male	hm	2.248
hm_11_0035	train	3	Male	hm	1.986
hm_11_0036	train	1	Male	hm	1.725
hm_11_0037	train	2	Male	hm	3.031
hm_11_0038	train	5	Male	hm	2.143
hm_11_0039	train	1	Male	hm	3.162
hm_11_0040	train	5	Female	hm	3.11
hm_11_0041	train	3	Male	hm	2.483
hm_11_0042	train	5	Female	hm	1.804
hm_11_0043	train	1	Male	hm	1.621
hm_11_0044	train	2	Male	hm	4.546
hm_11_0045	train	3	Male	hm	1.542
hm_11_0046	train	5	Female	hm	1.856
hm_11_0047	train	3	Male	hm	7.289
hm_11_0048	train	1	Male	hm	7.498
hm_11_0049	train	2	Male	hm	4.311
hm_11_0050	train	5	Female	hm	4.05
hm_11_0051	train	2	Male	hm	3.292
hm_11_0052	train	1	Male	hm	1.438
hm_11_0053	train	3	Male	hm	3.946
hm_11_0054	train	2	Male	hm	5.696
hm_11_0055	train	3	Male	hm	1.124
hm_11_0056	train	5	Female	hm	1.15
hm_11_0057	train	2	Male	hm	3.554
hm_11_0058	train	1	Male	hm	1.307
hm_11_0059	train	2	Male	hm	3.292
hm_11_0060	train	3	Male	hm	1.281
hm_11_0061	train	2	Male	hm	4.337
hm_11_0062	train	3	Male	hm	2.274
hm_11_0063	train	5	Female	hm	1.856
hm_11_0064	train	3	Male	hm	3.789
hm_11_0065	train	1	Male	hm	2.953
hm_11_0066	train	3	Male	hm	2.012
hm_11_0067	train	1	Male	hm	1.307
hm_11_0068	train	2	Male	hm	1.516
hm_11_0069	train	3	Male	hm	3.188
hm_11_0070	train	2	Male	hm	2.77
hm_11_0071	train	3	Male	hm	2.639
hm_11_0072	train	1	Male	hm	4.181
hm_11_0073	train	3	Male	hm	3.684
hm_11_0074	train	2	Male	hm	5.539
hm_11_0075	train	3	Male	hm	6.375
hm_11_0076	train	1	Male	hm	5.539
hm_11_0077	train	3	Male	hm	6.088
hm_11_0078	test	6	Female	hm	0.941
hm_11_0079	train	3	Male	hm	0.863
hm_11_0080	test	6	Female	hm	1.725
hm_11_0081	train	3	Male	hm	2.039
hm_11_0082	test	6	Female	hm	1.673
hm_11_0083	train	3	Male	hm	3.005
hm_11_0084	test	6	Female	hm	1.15
hm_11_0085	train	3	Male	hm	0.889
hm_11_0086	test	6	Female	hm	11.991
hm_11_0087	test	6	Female	hm	9.118
hm_11_0088	train	1	Male	hm	2.195
hm_11_0089	train	3	Male	hm	6.741
hm_11_0090	train	2	Male	hm	5.199
hm_11_0091	train	1	Male	hm	2.143
hm_11_0092	train	2	Male	hm	5.696
hm_11_0093	train	3	Male	hm	8.125
hm_11_0094	train	2	Male	hm	8.282
hm_11_0095	train	1	Male	hm	5.852
hm_11_0096	train	3	Male	hm	1.098
hm_11_0097	train	3	Male	hm	6.218
hm_11_0098	train	1	Male	hm	4.781
hm_11_0099	train	3	Male	hm	2.065
hm_11_0100	train	1	Male	hm	1.15
hm_11_0101	train	3	Male	hm	4.233
hm_11_0102	train	1	Male	hm	1.359
hm_11_0103	train	3	Male	hm	1.986
hm_11_0104	train	2	Male	hm	1.307
hm_11_0105	train	5	Female	hm	1.438
hm_11_0106	train	1	Male	hm	1.464
hm_11_0107	test	6	Female	hm	1.412
hm_11_0108	train	11	Female	hm	1.124
hm_11_0109	train	12	Male	hm	0.706
hm_11_0110	train	4	Female	hm	6.741
hm_12_0001	train	5	Female	hm	2.639
hm_12_0003	train	5	Female	hm	5.983
hm_12_0004	train	2	Male	hm	2.613
hm_12_0005	train	2	Male	hm	2.221
hm_12_0006	train	2	Male	hm	4.102
hm_12_0007	train	2	Male	hm	2.117
hm_12_0008	train	2	Male	hm	2.248
hm_12_0009	train	2	Male	hm	2.091
hm_12_0010	train	2	Male	hm	2.248
hm_12_0011	train	2	Male	hm	2.195
hm_12_0012	train	2	Male	hm	2.012
hm_12_0013	train	2	Male	hm	2.039
hm_12_0014	train	2	Male	hm	1.777
hm_12_0015	train	2	Male	hm	24.844
hm_12_0016	train	5	Female	hm	0.968
hm_12_0018	train	5	Female	hm	0.889
hm_12_0020	train	5	Female	hm	2.352
hm_12_0022	train	5	Female	hm	4.808
hm_12_0023	train	3	Male	hm	2.979
hm_12_0024	train	2	Male	hm	2.822
hm_12_0025	train	5	Female	hm	2.77
hm_12_0026	train	3	Male	hm	9.588
hm_12_0027	train	2	Male	hm	4.99
hm_12_0028	train	5	Female	hm	0.889
hm_12_0030	train	2	Male	hm	2.248
hm_12_0031	train	5	Female	hm	1.255
hm_12_0033	train	5	Female	hm	4.781
hm_12_0034	train	1	Male	hm	10.581
hm_12_0035	train	3	Male	hm	4.546
hm_12_0036	train	2	Male	hm	5.617
hm_12_0037	train	1	Male	hm	1.49
hm_12_0038	train	2	Male	hm	0.811
hm_12_0039	train	1	Male	hm	4.677
hm_12_0040	train	3	Male	hm	5.565
hm_12_0041	train	5	Female	hm	0.55
hm_12_0043	train	5	Female	hm	2.796
hm_12_0044	train	2	Male	hm	3.005
hm_12_0045	train	5	Female	hm	4.729
hm_12_0046	train	3	Male	hm	2.535
hm_12_0047	train	5	Female	hm	2.744
hm_12_0049	train	5	Female	hm	12.07
hm_12_0050	train	3	Male	hm	2.248
hm_12_0051	train	2	Male	hm	4.808
hm_12_0052	train	3	Male	hm	4.311
hm_12_0053	train	2	Male	hm	3.031
hm_12_0054	train	3	Male	hm	1.307
hm_12_0055	train	2	Male	hm	3.11
hm_12_0056	train	3	Male	hm	3.11
hm_12_0057	train	5	Female	hm	4.808
hm_12_0058	train	3	Male	hm	7.472
hm_12_0059	train	5	Female	hm	3.528
hm_12_0060	train	2	Male	hm	1.777
hm_12_0061	train	1	Male	hm	5.356
hm_12_0062	train	5	Female	hm	3.084
hm_12_0063	train	3	Male	hm	3.031
hm_12_0064	train	1	Male	hm	9.823
hm_12_0065	train	2	Male	hm	1.725
hm_12_0066	train	5	Female	hm	3.371
hm_12_0067	train	3	Male	hm	3.031
hm_12_0068	train	5	Female	hm	1.673
hm_12_0069	train	2	Male	hm	1.386
hm_12_0070	train	3	Male	hm	3.031
hm_12_0071	train	2	Male	hm	4.155
hm_12_0072	train	5	Female	hm	3.031
hm_12_0073	train	3	Male	hm	2.718
hm_12_0074	train	5	Female	hm	0.968
hm_12_0075	train	1	Male	hm	5.356
hm_12_0076	train	5	Female	hm	1.804
hm_12_0077	train	3	Male	hm	3.031
hm_12_0078	train	1	Male	hm	8.412
hm_12_0079	train	5	Female	hm	4.755
hm_12_0080	train	5	Female	hm	5.931
hm_12_0083	train	3	Male	hm	0.863
hm_12_0084	train	2	Male	hm	1.386
hm_12_0085	train	5	Female	hm	1.333
hm_12_0086	train	1	Male	hm	0.941
hm_12_0087	train	5	Female	hm	2.039
hm_12_0089	train	5	Female	hm	4.128
hm_13_0001	train	5	Female	hm	2.639
hm_13_0003	train	5	Female	hm	6.114
hm_13_0004	train	2	Male	hm	7.237
hm_13_0005	train	2	Male	hm	6.009
hm_13_0006	train	2	Male	hm	5.33
hm_13_0007	train	2	Male	hm	6.453
hm_13_0008	train	2	Male	hm	4.546
hm_13_0009	train	2	Male	hm	11.495
hm_13_0010	train	3	Male	hm	3.554
hm_13_0011	train	2	Male	hm	2.457
hm_13_0012	train	3	Male	hm	2.666
hm_13_0013	train	2	Male	hm	1.856
hm_13_0014	train	2	Male	hm	6.741
hm_13_0015	train	3	Male	hm	2.457
hm_13_0016	train	2	Male	hm	5.121
hm_13_0017	train	3	Male	hm	3.345
hm_13_0018	train	2	Male	hm	6.088
hm_13_0019	train	3	Male	hm	3.005
hm_13_0020	train	2	Male	hm	3.632
hm_13_0021	train	3	Male	hm	2.639
hm_13_0022	train	2	Male	hm	6.427
hm_13_0023	train	3	Male	hm	5.696
hm_13_0024	train	2	Male	hm	1.673
hm_13_0025	train	3	Male	hm	3.554
hm_13_0026	train	2	Male	hm	1.908
hm_13_0027	train	3	Male	hm	5.226
hm_13_0028	train	2	Male	hm	1.751
hm_13_0029	train	3	Male	hm	2.352
hm_13_0030	train	2	Male	hm	1.673
hm_13_0031	train	3	Male	hm	5.905
hm_13_0032	train	2	Male	hm	1.934
hm_13_0033	train	3	Male	hm	3.031
hm_13_0034	train	2	Male	hm	2.77
hm_13_0035	train	3	Male	hm	2.195
hm_13_0036	train	2	Male	hm	8.752
hm_13_0037	train	3	Male	hm	2.3
hm_13_0038	train	2	Male	hm	2.3
hm_13_0039	train	3	Male	hm	4.311
hm_13_0040	train	2	Male	hm	1.96
hm_13_0041	train	2	Male	hm	3.266
hm_13_0042	train	3	Male	hm	5.382
hm_13_0043	train	2	Male	hm	1.568
hm_13_0044	train	1	Male	hm	3.501
hm_13_0045	train	2	Male	hm	2.195
hm_13_0046	train	3	Male	hm	5.67
hm_13_0047	train	1	Male	hm	1.725
hm_13_0048	train	1	Male	hm	3.136
hm_13_0049	train	2	Male	hm	3.031
hm_13_0050	train	1	Male	hm	2.483
hm_13_0051	train	2	Male	hm	1.908
hm_13_0052	train	3	Male	hm	2.744
hm_13_0053	train	2	Male	hm	4.572
hm_13_0054	train	3	Male	hm	3.266
hm_13_0055	train	2	Male	hm	1.621
hm_13_0056	train	1	Male	hm	2.77
hm_13_0057	train	5	Female	hm	2.848
hm_13_0058	train	3	Male	hm	2.221
hm_13_0059	train	2	Male	hm	2.822
hm_13_0060	train	5	Female	hm	1.804
hm_13_0061	train	3	Male	hm	3.919
hm_13_0062	train	5	Female	hm	4.05
hm_13_0063	train	3	Male	hm	2.796
hm_13_0064	train	5	Female	hm	8.23
hm_13_0065	train	3	Male	hm	4.181
hm_13_0066	train	2	Male	hm	3.658
hm_13_0067	train	5	Female	hm	0.941
hm_13_0068	train	5	Female	hm	3.789
hm_13_0069	train	2	Male	hm	3.449
hm_13_0070	train	5	Female	hm	5.67
hm_13_0071	train	3	Male	hm	4.05
hm_13_0072	train	2	Male	hm	5.043
hm_13_0073	train	3	Male	hm	3.24
hm_13_0074	train	2	Male	hm	7.132
hm_13_0075	train	3	Male	hm	4.886
hm_13_0076	train	2	Male	hm	3.946
hm_13_0077	train	3	Male	hm	3.136
hm_13_0078	train	2	Male	hm	1.882
hm_13_0079	train	3	Male	hm	2.352
hm_13_0080	train	2	Male	hm	1.203
hm_13_0081	train	3	Male	hm	2.143
hm_13_0083	train	3	Male	hm	1.281
hm_13_0085	train	3	Male	hm	1.856
hm_13_0086	train	3	Male	hm	0.471
hm_13_0087	train	2	Male	hm	4.599
hm_13_0088	train	3	Male	hm	5.226
hm_13_0089	train	1	Male	hm	2.169
hm_13_0090	train	5	Female	hm	4.416
hm_13_0091	train	3	Male	hm	2.979
hm_13_0092	train	5	Female	hm	3.136
hm_13_0093	train	3	Male	hm	2.848
hm_13_0094	train	2	Male	hm	1.255
hm_13_0095	train	1	Male	hm	2.091
hm_13_0096	train	2	Male	hm	1.699
hm_13_0098	train	3	Male	hm	0.941
hm_13_0099	train	2	Male	hm	1.386
hm_13_0100	train	5	Female	hm	1.333
hm_13_0101	train	5	Female	hm	1.986
hm_13_0103	train	5	Female	hm	2.248
hm_14_0001	train	3	Male	hm	2.639
hm_14_0002	train	2	Male	hm	2.352
hm_14_0003	train	5	Female	hm	3.554
hm_14_0004	train	3	Male	hm	4.546
hm_14_0006	train	2	Male	hm	26.306
hm_14_0007	train	3	Male	hm	13.663
hm_14_0008	train	2	Male	hm	5.147
hm_14_0009	train	3	Male	hm	4.494
hm_14_0010	train	2	Male	hm	4.311
hm_14_0011	train	3	Male	hm	4.076
hm_14_0012	train	2	Male	hm	7.132
hm_14_0013	train	3	Male	hm	8.935
hm_14_0014	train	2	Male	hm	3.815
hm_14_0015	train	3	Male	hm	5.931
hm_14_0016	train	2	Male	hm	8.543
hm_14_0017	train	3	Male	hm	9.588
hm_14_0018	train	2	Male	hm	5.774
hm_14_0019	train	3	Male	hm	1.124
hm_14_0020	train	3	Male	hm	5.226
hm_14_0021	train	2	Male	hm	6.427
hm_14_0022	train	2	Male	hm	2.927
hm_14_0023	train	2	Male	hm	5.435
hm_14_0024	train	3	Male	hm	6.14
hm_14_0025	train	2	Male	hm	8.83
hm_14_0026	train	3	Male	hm	15.544
hm_14_0027	train	3	Male	hm	3.528
hm_14_0028	train	2	Male	hm	16.354
hm_14_0029	train	3	Male	hm	8.125
hm_14_0030	train	2	Male	hm	0.863
hm_14_0031	train	3	Male	hm	13.376
hm_14_0032	train	2	Male	hm	1.986
hm_14_0033	train	3	Male	hm	14.264
hm_14_0034	train	2	Male	hm	1.438
hm_14_0035	train	3	Male	hm	6.035
hm_14_0036	train	2	Male	hm	6.088
hm_14_0037	train	3	Male	hm	9.928
hm_14_0038	train	2	Male	hm	0.654
hm_14_0039	train	3	Male	hm	4.729
hm_14_0040	train	2	Male	hm	0.811
hm_14_0041	train	3	Male	hm	2.378
hm_14_0042	train	3	Male	hm	4.155
hm_14_0043	train	2	Male	hm	2.039
hm_14_0044	train	3	Male	hm	2.927
hm_14_0045	train	2	Male	hm	4.311
hm_14_0046	train	3	Male	hm	2.352
hm_14_0047	train	2	Male	hm	0.55
hm_14_0048	train	3	Male	hm	2.404
hm_14_0049	train	2	Male	hm	0.732
hm_14_0050	train	3	Male	hm	3.397
hm_14_0051	train	3	Male	hm	1.542
hm_14_0052	train	3	Male	hm	5.121
hm_14_0053	train	2	Male	hm	2.639
hm_14_0054	train	3	Male	hm	7.002
hm_14_0055	train	2	Male	hm	0.602
hm_14_0056	train	3	Male	hm	5.121
hm_14_0057	train	2	Male	hm	0.471
hm_14_0058	train	3	Male	hm	4.729
hm_14_0059	train	3	Male	hm	4.912
hm_14_0060	train	3	Male	hm	2.248
hm_14_0061	train	2	Male	hm	5.304
hm_14_0062	train	3	Male	hm	2.248
hm_14_0063	train	2	Male	hm	2.953
hm_14_0064	train	3	Male	hm	2.169
hm_14_0065	train	3	Male	hm	6.767
hm_14_0066	train	3	Male	hm	4.886
hm_14_0067	train	2	Male	hm	2.587
hm_14_0068	train	3	Male	hm	3.893
hm_14_0069	train	2	Male	hm	3.501
hm_14_0070	train	3	Male	hm	5.199
hm_14_0071	train	2	Male	hm	1.96
hm_14_0072	train	3	Male	hm	4.207
hm_14_0073	train	3	Male	hm	5.644
hm_14_0074	train	2	Male	hm	2.195
hm_14_0075	train	3	Male	hm	6.297
hm_14_0076	train	2	Male	hm	1.856
hm_14_0077	train	3	Male	hm	5.147
hm_14_0078	train	3	Male	hm	4.442
hm_14_0079	train	3	Male	hm	3.397
hm_14_0080	train	2	Male	hm	3.214
hm_14_0081	train	3	Male	hm	4.494
hm_14_0082	train	3	Male	hm	3.345
hm_14_0083	train	2	Male	hm	3.292
hm_14_0084	train	3	Male	hm	3.554
hm_14_0085	train	3	Male	hm	5.644
hm_14_0086	train	2	Male	hm	10.241
hm_14_0087	train	2	Male	hm	10.241
hm_14_0088	train	2	Male	hm	11.364
hm_14_0089	train	3	Male	hm	6.349
hm_14_0090	train	2	Male	hm	3.084
hm_14_0091	train	3	Male	hm	2.666
hm_14_0092	train	2	Male	hm	3.319
hm_14_0093	train	3	Male	hm	17.399
hm_14_0094	train	3	Male	hm	2.692
hm_14_0095	train	2	Male	hm	5.147
hm_14_0096	train	3	Male	hm	7.446
hm_14_0097	train	2	Male	hm	4.442
hm_14_0098	train	2	Male	hm	3.11
hm_14_0099	train	3	Male	hm	4.99
hm_14_0100	train	3	Male	hm	4.494
hm_14_0101	train	2	Male	hm	4.181
hm_14_0102	train	3	Male	hm	11.26
hm_14_0103	train	2	Male	hm	5.905
hm_14_0104	train	2	Male	hm	4.076
hm_14_0105	train	3	Male	hm	6.166
hm_14_0106	train	3	Male	hm	4.05
hm_14_0107	train	2	Male	hm	8.674
hm_14_0108	train	3	Male	hm	4.703
hm_14_0109	train	3	Male	hm	2.744
hm_14_0110	train	2	Male	hm	3.763
hm_14_0111	train	3	Male	hm	4.729
hm_14_0112	train	3	Male	hm	8.621
hm_14_0113	train	3	Male	hm	2.404
hm_14_0114	train	2	Male	hm	6.114
hm_14_0115	train	3	Male	hm	1.699
hm_14_0116	train	2	Male	hm	2.065
hm_14_0117	train	3	Male	hm	3.397
hm_14_0118	train	2	Male	hm	1.856
hm_14_0119	train	3	Male	hm	2.378
hm_14_0120	train	3	Male	hm	5.539
hm_14_0121	train	3	Male	hm	5.905
hm_14_0122	train	3	Male	hm	4.677
hm_14_0124	train	2	Male	hm	3.084
hm_14_0125	train	3	Male	hm	3.919
hm_14_0126	train	5	Female	hm	1.516
hm_14_0127	train	5	Female	hm	1.647
hm_14_0128	train	3	Male	hm	1.908
hm_14_0129	train	5	Female	hm	8.491
hm_14_0130	train	3	Male	hm	2.953
hm_14_0131	train	5	Female	hm	8.021
hm_14_0132	train	5	Female	hm	4.729
hm_14_0133	train	3	Male	hm	1.673
hm_14_0134	train	5	Female	hm	7.368
hm_14_0135	train	5	Female	hm	2.065
hm_14_0136	train	3	Male	hm	3.214
hm_14_0137	train	3	Male	hm	8.674
hm_14_0138	train	3	Male	hm	3.084
hm_14_0139	train	3	Male	hm	2.875
hm_14_0140	train	3	Male	hm	2.509
hm_14_0141	train	3	Male	hm	4.102
hm_14_0142	train	2	Male	hm	5.226
hm_14_0143	train	3	Male	hm	6.323
hm_14_0144	train	3	Male	hm	3.136
hm_14_0145	train	3	Male	hm	2.953
hm_14_0146	train	3	Male	hm	2.326
hm_14_0147	train	3	Male	hm	2.378
hm_14_0148	train	2	Male	hm	6.27
hm_14_0149	train	3	Male	hm	4.128
hm_14_0150	train	2	Male	hm	2.039
hm_14_0151	train	3	Male	hm	2.195
hm_14_0152	train	3	Male	hm	4.05
hm_14_0153	train	2	Male	hm	2.248
hm_14_0154	train	3	Male	hm	5.696
hm_14_0155	train	2	Male	hm	3.841
hm_14_0156	train	3	Male	hm	4.364
hm_14_0157	train	3	Male	hm	8.595
hm_14_0158	train	2	Male	hm	3.162
hm_14_0159	train	2	Male	hm	2.848
hm_14_0160	train	3	Male	hm	13.141
hm_14_0161	train	2	Male	hm	1.595
hm_14_0162	train	3	Male	hm	6.584
hm_14_0163	train	3	Male	hm	4.886
hm_14_0164	train	3	Male	hm	9.301
hm_14_0165	train	2	Male	hm	3.789
hm_14_0166	train	3	Male	hm	5.304
hm_14_0167	train	5	Female	hm	2.012
hm_14_0168	train	3	Male	hm	1.986
hm_14_0169	train	5	Female	hm	1.751
hm_14_0170	train	3	Male	hm	10.868
hm_14_0171	train	3	Male	hm	8.439
hm_14_0172	train	3	Male	hm	12.435
hm_14_0173	train	2	Male	hm	6.871
hm_14_0174	train	3	Male	hm	4.076
hm_14_0175	train	2	Male	hm	4.964
hm_14_0176	train	3	Male	hm	2.744
hm_14_0179	train	2	Male	hm	2.875
hm_14_0180	train	3	Male	hm	1.046
hm_14_0181	train	2	Male	hm	3.763
hm_14_0182	train	3	Male	hm	4.259
hm_14_0183	train	2	Male	hm	0.837
hm_14_0184	train	3	Male	hm	5.591
hm_14_0185	train	2	Male	hm	0.915
hm_14_0186	train	3	Male	hm	2.509
hm_14_0187	train	2	Male	hm	0.863
hm_14_0188	train	3	Male	hm	5.199
hm_14_0189	train	2	Male	hm	3.397
hm_14_0190	train	3	Male	hm	5.356
hm_14_0191	train	2	Male	hm	1.359
hm_14_0192	train	3	Male	hm	2.248
hm_14_0193	train	5	Female	hm	23.485
hm_14_0194	train	5	Female	hm	4.86
hm_14_0195	train	3	Male	hm	1.882
hm_14_0196	train	5	Female	hm	8.021
hm_14_0197	train	2	Male	hm	3.528
hm_14_0198	train	3	Male	hm	4.207
hm_14_0199	train	2	Male	hm	2.326
hm_14_0200	test	6	Female	hm	17.712
hm_14_0201	train	3	Male	hm	1.83
hm_14_0202	train	2	Male	hm	1.934
hm_14_0203	train	3	Male	hm	2.535
hm_14_0204	train	2	Male	hm	1.098
hm_14_0205	train	3	Male	hm	13.532
hm_14_0206	train	2	Male	hm	1.934
hm_14_0207	train	3	Male	hm	2.509
hm_14_0208	test	6	Female	hm	2.248
hm_14_0209	train	3	Male	hm	15.857
hm_14_0210	train	3	Male	hm	5.539
hm_14_0211	train	2	Male	hm	11.443
hm_14_0212	train	3	Male	hm	1.516
hm_14_0213	train	2	Male	hm	2.012
hm_14_0214	train	2	Male	hm	2.457
hm_14_0215	train	3	Male	hm	8.204
hm_14_0216	train	3	Male	hm	2.039
hm_14_0217	test	6	Female	hm	8.595
hm_14_0218	train	3	Male	hm	3.58
hm_14_0219	train	2	Male	hm	2.666
hm_14_0220	test	6	Female	hm	6.114
hm_14_0221	train	3	Male	hm	4.207
hm_14_0222	train	2	Male	hm	3.554
hm_14_0223	test	6	Female	hm	4.677
hm_14_0224	train	3	Male	hm	2.3
hm_14_0225	test	6	Male	hm	7.211
hm_14_0226	train	3	Female	hm	3.162
hm_14_0227	test	6	Female	hm	7.733
hm_14_0228	train	2	Male	hm	1.595
hm_14_0229	test	6	Female	hm	6.349
hm_14_0230	train	3	Male	hm	2.875
hm_14_0231	test	6	Female	hm	3.423
hm_14_0232	train	3	Male	hm	4.05
hm_14_0233	test	6	Female	hm	3.11
hm_14_0234	test	6	Female	hm	4.781
hm_14_0235	test	6	Female	hm	8.021
hm_14_0236	train	3	Male	hm	2.457
hm_14_0237	train	2	Male	hm	10.137
hm_14_0238	train	3	Male	hm	3.554
hm_14_0239	train	2	Male	hm	5.017
hm_14_0240	train	3	Male	hm	4.207
hm_14_0241	train	2	Male	hm	3.11
hm_14_0242	train	3	Male	hm	8.334
hm_14_0243	train	2	Male	hm	1.986
hm_14_0244	train	3	Male	hm	3.867
hm_14_0245	train	3	Male	hm	8.543
hm_14_0246	train	2	Male	hm	1.725
hm_14_0247	train	3	Male	hm	25.523
hm_15_0001	train	5	Female	hm	2.718
hm_15_0003	train	5	Female	hm	5.983
hm_15_0004	train	2	Male	hm	7.289
hm_15_0005	train	2	Male	hm	7.577
hm_15_0006	train	2	Male	hm	9.745
hm_15_0007	train	2	Male	hm	6.793
hm_15_0008	train	2	Male	hm	6.819
hm_15_0009	train	2	Male	hm	7.759
hm_15_0010	train	2	Male	hm	6.427
hm_15_0011	train	2	Male	hm	7.341
hm_15_0012	train	2	Male	hm	6.088
hm_15_0013	train	2	Male	hm	5.8
hm_15_0014	train	2	Male	hm	1.934
hm_15_0015	train	1	Male	hm	1.255
hm_15_0016	train	1	Male	hm	5.957
hm_15_0017	train	1	Male	hm	7.054
hm_15_0018	train	3	Male	hm	1.699
hm_15_0019	train	2	Male	hm	2.039
hm_15_0020	train	3	Male	hm	4.938
hm_15_0021	train	1	Male	hm	7.707
hm_15_0022	train	5	Female	hm	9.745
hm_15_0023	train	2	Male	hm	4.651
hm_15_0024	train	3	Male	hm	5.696
hm_15_0025	train	2	Male	hm	3.972
hm_15_0026	train	3	Male	hm	2.457
hm_15_0027	train	2	Male	hm	6.244
hm_15_0028	train	3	Male	hm	1.333
hm_15_0029	train	3	Male	hm	3.606
hm_15_0030	train	3	Male	hm	2.483
hm_15_0031	train	2	Male	hm	2.875
hm_15_0032	train	3	Male	hm	3.397
hm_15_0033	train	2	Male	hm	3.972
hm_15_0034	train	5	Female	hm	4.755
hm_15_0036	train	5	Female	hm	1.777
hm_15_0037	train	3	Male	hm	3.397
hm_15_0038	train	3	Male	hm	3.867
hm_15_0039	train	5	Female	hm	3.632
hm_15_0040	train	2	Male	hm	6.819
hm_15_0041	train	5	Female	hm	0.994
hm_15_0042	train	3	Male	hm	2.901
hm_15_0043	train	5	Female	hm	7.446
hm_15_0045	train	5	Female	hm	2.587
hm_15_0046	train	2	Male	hm	4.128
hm_15_0047	train	5	Female	hm	1.307
hm_15_0048	train	2	Male	hm	3.841
hm_15_0049	train	3	Male	hm	5.173
hm_15_0050	train	2	Male	hm	1.751
hm_15_0051	train	3	Male	hm	2.875
hm_15_0052	train	2	Male	hm	5.461
hm_15_0053	train	3	Male	hm	1.307
hm_15_0054	train	2	Male	hm	1.255
hm_15_0055	train	3	Male	hm	1.203
hm_15_0056	train	1	Male	hm	0.68
hm_15_0057	train	3	Male	hm	1.02
hm_15_0058	train	1	Male	hm	1.83
hm_15_0059	train	2	Male	hm	3.58
hm_15_0060	train	1	Male	hm	2.535
hm_15_0061	train	3	Male	hm	2.117
hm_15_0062	train	1	Male	hm	3.214
hm_15_0063	train	3	Male	hm	1.83
hm_15_0064	train	1	Male	hm	3.658
hm_15_0066	train	1	Male	hm	0.941
hm_15_0068	train	1	Male	hm	1.281
hm_15_0070	train	1	Male	hm	1.229
hm_15_0072	train	1	Male	hm	1.098
hm_15_0075	train	1	Male	hm	1.124
hm_15_0076	train	3	Male	hm	6.114
hm_15_0077	train	1	Male	hm	2.587
hm_15_0078	train	2	Male	hm	1.804
hm_15_0079	train	1	Male	hm	0.941
hm_15_0080	train	3	Male	hm	3.031
hm_15_0081	train	1	Male	hm	0.602
hm_15_0083	train	1	Male	hm	2.979
hm_15_0084	train	3	Male	hm	7.159
hm_15_0085	train	3	Male	hm	1.804
hm_15_0086	train	1	Male	hm	1.49
hm_15_0087	train	2	Male	hm	2.666
hm_15_0088	train	3	Male	hm	2.3
hm_15_0089	train	1	Male	hm	1.49
hm_15_0090	train	2	Male	hm	1.934
hm_15_0091	train	1	Male	hm	1.934
hm_15_0093	train	3	Male	hm	0.889
hm_15_0094	train	2	Male	hm	1.229
hm_15_0095	train	5	Female	hm	1.333
hm_15_0096	train	1	Male	hm	0.915
hm_15_0097	train	5	Female	hm	1.986
hm_15_0098	train	5	Female	hm	2.796
hm_15_0099	train	5	Female	hm	6.035
hm_16_0001	train	2	Male	hm	10.006
hm_16_0002	train	3	Male	hm	2.483
hm_16_0003	train	2	Male	hm	6.297
hm_16_0004	train	3	Male	hm	3.658
hm_16_0005	train	2	Male	hm	3.815
hm_16_0006	train	3	Male	hm	2.587
hm_16_0007	train	3	Male	hm	1.072
hm_16_0008	train	5	Female	hm	0.811
hm_16_0009	train	3	Male	hm	1.934
hm_16_0010	train	5	Female	hm	0.68
hm_16_0011	train	3	Male	hm	3.057
hm_16_0012	train	5	Female	hm	2.639
hm_16_0013	train	3	Male	hm	5.017
hm_16_0014	train	5	Female	hm	2.012
hm_16_0015	train	3	Male	hm	5.774
hm_16_0016	train	5	Female	hm	3.084
hm_16_0017	train	3	Male	hm	4.808
hm_16_0018	train	5	Female	hm	1.02
hm_16_0019	train	3	Male	hm	1.699
hm_16_0020	train	5	Female	hm	3.214
hm_16_0021	train	5	Female	hm	7.786
hm_16_0022	train	5	Female	hm	0.732
hm_16_0023	train	3	Male	hm	1.49
hm_16_0024	train	5	Female	hm	1.934
hm_16_0025	train	3	Male	hm	2.169
hm_16_0026	train	5	Female	hm	2.012
hm_16_0027	train	3	Male	hm	1.568
hm_16_0028	train	5	Female	hm	3.58
hm_16_0029	train	3	Male	hm	1.699
hm_16_0030	train	5	Female	hm	2.979
hm_16_0031	train	3	Male	hm	3.449
hm_16_0032	train	5	Female	hm	2.091
hm_16_0033	train	3	Male	hm	2.221
hm_16_0034	train	5	Female	hm	4.285
hm_16_0035	train	3	Male	hm	2.639
hm_16_0036	train	5	Female	hm	3.606
hm_16_0037	train	3	Male	hm	4.938
hm_16_0038	train	5	Female	hm	8.439
hm_16_0039	train	3	Male	hm	3.423
hm_16_0040	train	5	Female	hm	2.43
hm_16_0041	train	3	Male	hm	2.274
hm_16_0042	train	5	Female	hm	2.848
hm_16_0043	train	3	Male	hm	1.882
hm_16_0044	train	5	Female	hm	1.438
hm_16_0045	train	3	Male	hm	4.416
hm_16_0046	train	5	Female	hm	2.77
hm_16_0047	train	3	Male	hm	4.468
hm_16_0048	train	5	Female	hm	1.804
hm_16_0049	train	3	Male	hm	3.423
hm_16_0050	train	5	Female	hm	3.684
hm_16_0051	train	3	Male	hm	2.3
hm_16_0052	train	5	Female	hm	4.468
hm_16_0053	train	3	Male	hm	1.595
hm_16_0054	train	5	Female	hm	5.382
hm_16_0055	train	5	Female	hm	1.177
hm_16_0056	train	3	Male	hm	4.729
hm_16_0057	train	5	Female	hm	7.942
hm_16_0058	train	3	Male	hm	3.763
hm_16_0059	train	2	Male	hm	1.072
hm_16_0060	train	5	Female	hm	1.386
hm_16_0061	train	3	Male	hm	1.934
hm_16_0062	train	5	Female	hm	0.968
hm_16_0063	train	3	Male	hm	3.449
hm_16_0064	train	1	Male	hm	3.71
hm_16_0065	train	3	Male	hm	1.908
hm_16_0066	train	2	Male	hm	2.169
hm_16_0067	train	3	Male	hm	4.834
hm_16_0068	train	1	Male	hm	1.83
hm_16_0069	train	5	Female	hm	1.751
hm_16_0070	train	2	Male	hm	5.278
hm_16_0071	train	5	Female	hm	6.166
hm_16_0072	train	1	Male	hm	4.703
hm_16_0073	train	5	Female	hm	2.822
hm_16_0074	train	3	Male	hm	5.67
hm_16_0075	train	2	Male	hm	4.102
hm_16_0076	train	3	Male	hm	2.692
hm_16_0077	train	5	Female	hm	3.632
hm_16_0078	train	1	Male	hm	2.039
hm_16_0079	train	3	Male	hm	1.804
hm_16_0080	train	5	Female	hm	4.99
hm_16_0081	train	1	Male	hm	1.516
hm_16_0082	train	5	Male	hm	1.386
hm_16_0083	train	2	Male	hm	4.102
hm_16_0084	train	5	Female	hm	3.449
hm_16_0085	train	1	Male	hm	2.509
hm_16_0086	train	3	Male	hm	5.565
hm_16_0087	train	5	Female	hm	0.732
hm_16_0088	train	2	Male	hm	1.359
hm_16_0089	train	5	Female	hm	2.561
hm_16_0090	train	2	Male	hm	0.837
hm_16_0091	train	5	Female	hm	3.449
hm_16_0092	train	1	Male	hm	1.281
hm_16_0093	train	3	Male	hm	0.915
hm_16_0094	train	5	Female	hm	1.464
hm_16_0095	train	2	Male	hm	1.281
hm_16_0096	train	1	Male	hm	1.464
hm_16_0097	test	6	Female	hm	1.464
hm_16_0098	train	11	Female	hm	1.124
hm_16_0099	train	13	Male	hm	0.759
hm_17_0001	train	2	Male	hm	2.248
hm_17_0002	train	3	Male	hm	2.613
hm_17_0003	train	4	Female	hm	2.065
hm_17_0004	train	4	Female	hm	6.244
hm_17_0005	train	2	Male	hm	4.625
hm_17_0006	train	3	Male	hm	1.934
hm_17_0007	train	2	Male	hm	3.24
hm_17_0008	train	3	Male	hm	2.77
hm_17_0009	train	2	Male	hm	1.804
hm_17_0010	train	3	Male	hm	2.352
hm_17_0011	train	2	Male	hm	1.725
hm_17_0012	train	3	Male	hm	2.039
hm_17_0013	train	2	Male	hm	8.595
hm_17_0014	train	3	Male	hm	2.848
hm_17_0015	train	2	Male	hm	1.673
hm_17_0016	train	3	Male	hm	2.561
hm_17_0017	train	2	Male	hm	10.633
hm_17_0018	train	3	Male	hm	2.77
hm_17_0019	train	2	Male	hm	2.378
hm_17_0020	train	3	Male	hm	3.292
hm_17_0021	train	2	Male	hm	5.095
hm_17_0023	train	2	Male	hm	3.501
hm_17_0024	train	3	Male	hm	2.561
hm_17_0025	train	3	Male	hm	2.274
hm_17_0026	train	2	Male	hm	8.517
hm_17_0027	train	3	Male	hm	2.457
hm_17_0028	train	2	Male	hm	3.632
hm_17_0029	train	3	Male	hm	2.248
hm_17_0030	train	2	Male	hm	5.408
hm_17_0031	train	3	Male	hm	6.009
hm_17_0032	train	2	Male	hm	3.136
hm_17_0033	train	3	Male	hm	2.378
hm_17_0034	train	2	Male	hm	2.195
hm_17_0035	train	3	Male	hm	4.625
hm_17_0036	train	2	Male	hm	4.076
hm_17_0037	train	3	Male	hm	8.804
hm_17_0038	train	2	Male	hm	2.796
hm_17_0039	train	3	Male	hm	7.211
hm_17_0040	train	2	Male	hm	4.703
hm_17_0041	train	3	Male	hm	8.177
hm_17_0042	train	3	Male	hm	4.233
hm_17_0043	test	6	Female	hm	1.333
hm_17_0044	train	2	Male	hm	1.255
hm_17_0045	test	6	Female	hm	1.15
hm_17_0046	train	2	Male	hm	2.666
hm_17_0047	train	3	Male	hm	3.136
hm_17_0048	train	2	Male	hm	3.893
hm_17_0049	test	6	Female	hm	2.143
hm_17_0050	train	3	Male	hm	5.147
hm_17_0051	train	2	Male	hm	2.535
hm_17_0052	test	6	Female	hm	2.195
hm_17_0053	train	2	Male	hm	3.188
hm_17_0054	test	6	Female	hm	1.281
hm_17_0055	train	3	Male	hm	3.946
hm_17_0056	train	1	Male	hm	1.595
hm_17_0057	train	3	Male	hm	1.699
hm_17_0058	train	1	Male	hm	5.356
hm_17_0059	train	2	Male	hm	2.639
hm_17_0060	train	1	Male	hm	3.528
hm_17_0061	train	2	Male	hm	3.084
hm_17_0062	train	1	Male	hm	4.259
hm_17_0063	train	3	Male	hm	1.124
hm_17_0064	train	1	Male	hm	4.207
hm_17_0065	train	3	Male	hm	1.568
hm_17_0066	train	2	Male	hm	3.266
hm_17_0067	train	1	Male	hm	3.528
hm_17_0068	train	3	Male	hm	2.613
hm_17_0069	train	1	Male	hm	0.968
hm_17_0070	train	3	Male	hm	10.999
hm_17_0071	train	1	Male	hm	1.621
hm_17_0072	train	2	Male	hm	3.188
hm_17_0073	train	3	Male	hm	2.692
hm_17_0074	train	1	Male	hm	0.524
hm_17_0075	train	2	Male	hm	2.457
hm_17_0076	train	1	Male	hm	2.039
hm_17_0077	train	3	Male	hm	0.863
hm_17_0078	train	1	Male	hm	5.722
hm_17_0079	train	2	Male	hm	1.516
hm_17_0080	train	1	Male	hm	1.229
hm_17_0081	train	2	Male	hm	2.274
hm_17_0082	train	1	Male	hm	4.076
hm_17_0083	train	3	Male	hm	2.117
hm_17_0084	train	1	Male	hm	3.998
hm_17_0085	train	3	Male	hm	1.96
hm_17_0086	train	1	Male	hm	14.238
hm_17_0087	train	2	Male	hm	1.516
hm_17_0088	train	1	Male	hm	8.125
hm_17_0089	train	3	Male	hm	0.863
hm_17_0090	train	1	Male	hm	3.449
hm_17_0091	train	2	Male	hm	3.475
hm_17_0092	train	1	Male	hm	4.468
hm_17_0093	train	3	Male	hm	3.292
hm_17_0094	train	1	Male	hm	1.647
hm_17_0095	train	2	Male	hm	3.084
hm_17_0096	train	1	Male	hm	0.941
hm_17_0098	train	1	Male	hm	9.013
hm_17_0100	train	5	Female	hm	3.136
hm_17_0101	train	3	Male	hm	1.699
hm_17_0102	train	5	Female	hm	0.602
hm_17_0104	train	5	Female	hm	0.628
hm_17_0105	train	5	Female	hm	3.449
hm_17_0107	train	4	Female	hm	1.046
hm_17_0108	train	4	Female	hm	11.312
hm_17_0109	train	4	Female	hm	2.692
hm_17_0110	train	4	Female	hm	1.83
hm_18_0001	train	5	Female	hm	2.927
hm_18_0003	train	5	Female	hm	6.061
hm_18_0004	train	2	Male	hm	7.263
hm_18_0005	train	2	Male	hm	5.461
hm_18_0006	train	2	Male	hm	12.697
hm_18_0007	train	5	Female	hm	4.86
hm_18_0008	train	2	Male	hm	1.882
hm_18_0009	train	5	Female	hm	2.561
hm_18_0010	train	2	Male	hm	1.229
hm_18_0011	train	5	Female	hm	1.542
hm_18_0012	train	2	Male	hm	19.488
hm_18_0013	train	3	Male	hm	3.475
hm_18_0014	train	5	Female	hm	1.621
hm_18_0015	train	3	Male	hm	2.901
hm_18_0016	train	2	Male	hm	2.561
hm_18_0017	train	3	Male	hm	3.972
hm_18_0018	train	5	Female	hm	3.737
hm_18_0019	train	3	Male	hm	2.979
hm_18_0020	train	2	Male	hm	4.285
hm_18_0021	train	3	Male	hm	7.681
hm_18_0022	train	2	Male	hm	13.193
hm_18_0023	train	3	Male	hm	1.777
hm_18_0024	train	5	Female	hm	2.561
hm_18_0025	train	3	Male	hm	6.453
hm_18_0026	train	3	Male	hm	2.483
hm_18_0027	train	5	Female	hm	2.483
hm_18_0028	train	3	Male	hm	7.315
hm_18_0029	train	2	Male	hm	6.715
hm_18_0030	train	3	Male	hm	0.994
hm_18_0031	train	2	Male	hm	7.524
hm_18_0032	train	1	Male	hm	2.692
hm_18_0033	train	3	Male	hm	4.337
hm_18_0034	train	1	Male	hm	4.076
hm_18_0035	train	5	Female	hm	1.49
hm_18_0036	train	1	Male	hm	1.255
hm_18_0037	train	3	Male	hm	2.378
hm_18_0038	train	5	Female	hm	2.43
hm_18_0039	train	3	Male	hm	2.744
hm_18_0040	train	1	Male	hm	0.863
hm_18_0041	train	2	Male	hm	3.162
hm_18_0042	train	3	Male	hm	3.919
hm_18_0043	train	1	Male	hm	8.883
hm_18_0044	train	2	Male	hm	3.684
hm_18_0045	train	3	Male	hm	5.67
hm_18_0046	train	1	Male	hm	6.662
hm_18_0047	train	2	Male	hm	2.117
hm_18_0048	train	1	Male	hm	1.516
hm_18_0049	train	5	Female	hm	2.587
hm_18_0050	train	1	Male	hm	1.882
hm_18_0051	train	5	Female	hm	4.128
hm_18_0052	train	1	Male	hm	2.117
hm_18_0053	train	2	Male	hm	3.815
hm_18_0054	train	1	Male	hm	2.091
hm_18_0055	train	3	Male	hm	2.065
hm_18_0056	train	3	Male	hm	3.423
hm_18_0057	train	5	Female	hm	2.77
hm_18_0058	train	3	Male	hm	1.83
hm_18_0059	train	2	Male	hm	2.796
hm_18_0060	train	5	Female	hm	1.49
hm_18_0061	train	2	Male	hm	2.065
hm_18_0062	train	5	Female	hm	2.352
hm_18_0063	train	2	Male	hm	4.416
hm_18_0064	train	5	Female	hm	3.24
hm_18_0065	train	3	Male	hm	1.908
hm_18_0066	train	2	Male	hm	4.39
hm_18_0067	train	3	Male	hm	1.046
hm_18_0068	train	3	Male	hm	3.501
hm_18_0069	train	2	Male	hm	1.516
hm_18_0070	train	1	Male	hm	1.464
hm_18_0071	train	3	Male	hm	1.804
hm_18_0072	train	2	Male	hm	3.214
hm_18_0073	train	3	Male	hm	1.83
hm_18_0074	train	2	Male	hm	2.77
hm_18_0075	train	1	Male	hm	2.666
hm_18_0076	train	5	Female	hm	5.278
hm_18_0077	train	1	Male	hm	3.214
hm_18_0078	train	2	Male	hm	4.964
hm_18_0079	train	1	Male	hm	2.77
hm_18_0080	train	2	Male	hm	1.595
hm_18_0081	train	1	Male	hm	6.897
hm_18_0082	train	5	Female	hm	3.528
hm_18_0083	train	2	Male	hm	2.43
hm_18_0085	train	3	Male	hm	0.915
hm_18_0086	train	2	Male	hm	1.464
hm_18_0087	train	5	Female	hm	1.359
hm_18_0088	train	1	Male	hm	1.046
hm_18_0089	train	5	Female	hm	1.96
hm_18_0090	train	5	Female	hm	2.848
hm_18_0091	train	5	Female	hm	5.931
hm_19_0001	train	1	Male	hm	2.848
hm_19_0002	train	2	Male	hm	2.3
hm_19_0003	train	3	Male	hm	2.692
hm_19_0004	train	1	Male	hm	2.012
hm_19_0005	train	1	Male	hm	6.297
hm_19_0006	train	5	Female	hm	1.412
hm_19_0008	train	5	Female	hm	1.621
hm_19_0009	train	3	Male	hm	4.52
hm_19_0010	train	5	Female	hm	3.737
hm_19_0011	train	2	Male	hm	3.266
hm_19_0012	train	5	Female	hm	1.49
hm_19_0013	train	3	Male	hm	0.941
hm_19_0014	train	3	Male	hm	2.666
hm_19_0015	train	2	Male	hm	6.453
hm_19_0016	train	3	Male	hm	2.692
hm_19_0017	train	2	Male	hm	13.219
hm_19_0018	train	3	Male	hm	2.169
hm_19_0019	train	2	Male	hm	3.449
hm_19_0020	train	3	Male	hm	0.654
hm_19_0021	train	2	Male	hm	0.471
hm_19_0022	train	3	Male	hm	1.934
hm_19_0023	train	5	Female	hm	2.639
hm_19_0024	train	3	Male	hm	2.822
hm_19_0025	train	1	Male	hm	5.513
hm_19_0026	train	1	Male	hm	1.568
hm_19_0027	train	2	Male	hm	3.449
hm_19_0028	train	5	Female	hm	2.692
hm_19_0029	train	2	Male	hm	4.781
hm_19_0030	train	1	Male	hm	3.11
hm_19_0031	train	3	Male	hm	1.595
hm_19_0032	train	1	Male	hm	3.737
hm_19_0033	train	5	Female	hm	2.012
hm_19_0034	train	1	Male	hm	1.333
hm_19_0035	train	3	Male	hm	1.333
hm_19_0036	train	2	Male	hm	2.561
hm_19_0037	train	1	Male	hm	0.55
hm_19_0038	train	1	Male	hm	1.438
hm_19_0039	train	5	Female	hm	1.96
hm_19_0040	train	2	Male	hm	5.461
hm_19_0041	train	1	Male	hm	0.994
hm_19_0042	train	5	Female	hm	4.076
hm_19_0043	train	1	Male	hm	5.199
hm_19_0044	train	2	Male	hm	2.248
hm_19_0045	train	3	Male	hm	4.155
hm_19_0046	train	2	Male	hm	3.24
hm_19_0047	train	5	Female	hm	3.71
hm_19_0048	test	6	Female	hm	1.83
hm_19_0050	train	1	Male	hm	6.819
hm_19_0051	test	6	Female	hm	4.181
hm_19_0052	train	5	Female	hm	4.05
hm_19_0053	test	6	Female	hm	3.345
hm_19_0054	train	3	Male	hm	1.777
hm_19_0055	train	1	Male	hm	1.412
hm_19_0056	train	5	Female	hm	3.789
hm_19_0057	test	6	Female	hm	1.595
hm_19_0058	train	1	Male	hm	1.83
hm_19_0059	test	6	Female	hm	2.457
hm_19_0060	train	2	Male	hm	3.031
hm_19_0061	train	5	Female	hm	7.237
hm_19_0062	train	1	Male	hm	7.211
hm_19_0063	train	3	Male	hm	3.737
hm_19_0064	train	2	Male	hm	2.483
hm_19_0065	train	5	Female	hm	5.095
hm_19_0066	train	1	Male	hm	2.561
hm_19_0068	train	3	Male	hm	3.528
hm_19_0069	train	2	Male	hm	2.901
hm_19_0070	train	3	Male	hm	6.662
hm_19_0071	train	2	Male	hm	6.662
hm_19_0072	train	3	Male	hm	3.449
hm_19_0073	train	2	Male	hm	6.662
hm_19_0074	train	3	Male	hm	3.005
hm_19_0075	train	2	Male	hm	4.52
hm_19_0076	train	3	Male	hm	7.211
hm_19_0077	train	2	Male	hm	1.751
hm_19_0078	train	3	Male	hm	6.244
hm_19_0079	train	3	Male	hm	3.632
hm_19_0080	train	2	Male	hm	3.266
hm_19_0081	train	3	Male	hm	2.509
hm_19_0082	train	2	Male	hm	2.378
hm_19_0083	train	3	Male	hm	5.722
hm_19_0084	train	2	Male	hm	3.737
hm_19_0085	train	3	Male	hm	0.968
hm_19_0086	train	2	Male	hm	2.091
hm_19_0087	train	3	Male	hm	9.666
hm_19_0088	train	2	Male	hm	2.326
hm_19_0089	train	3	Male	hm	1.177
hm_19_0090	train	2	Male	hm	1.02
hm_19_0091	test	6	Female	hm	3.58
hm_19_0092	train	2	Male	hm	1.464
hm_19_0093	test	6	Female	hm	8.491
hm_19_0094	train	3	Male	hm	4.703
hm_19_0095	test	6	Female	hm	7.185
hm_19_0097	test	6	Female	hm	2.039
hm_19_0099	train	3	Male	hm	1.96
hm_19_0100	train	1	Male	hm	2.274
hm_19_0102	train	4	Female	hm	9.484
hm_20_0001	train	3	Male	hm	2.953
hm_20_0002	test	6	Female	hm	2.77
hm_20_0003	test	6	Female	hm	3.057
hm_20_0004	train	3	Male	hm	4.494
hm_20_0005	test	6	Female	hm	3.319
hm_20_0006	train	3	Male	hm	2.457
hm_20_0007	test	6	Female	hm	1.333
hm_20_0008	train	3	Male	hm	0.576
hm_20_0009	test	6	Female	hm	2.43
hm_20_0010	train	3	Male	hm	2.613
hm_20_0011	test	6	Female	hm	3.397
hm_20_0012	train	3	Male	hm	2.796
hm_20_0013	test	6	Female	hm	1.908
hm_20_0014	train	3	Male	hm	0.759
hm_20_0015	test	6	Female	hm	0.785
hm_20_0016	train	3	Male	hm	0.706
hm_20_0017	test	6	Female	hm	3.815
hm_20_0018	train	3	Male	hm	5.696
hm_20_0019	test	6	Female	hm	3.841
hm_20_0020	train	3	Male	hm	7.577
hm_20_0021	test	6	Female	hm	4.442
hm_20_0022	train	3	Male	hm	0.863
hm_20_0023	test	6	Female	hm	5.435
hm_20_0024	train	3	Male	hm	0.68
hm_20_0025	test	6	Female	hm	4.259
hm_20_0026	train	3	Male	hm	3.606
hm_20_0027	test	6	Female	hm	3.11
hm_20_0028	train	3	Male	hm	2.953
hm_20_0029	test	6	Female	hm	5.226
hm_20_0030	train	3	Male	hm	1.02
hm_20_0031	test	6	Female	hm	3.031
hm_20_0032	train	3	Male	hm	5.983
hm_20_0033	test	6	Female	hm	3.58
hm_20_0034	train	3	Male	hm	3.162
hm_20_0035	test	6	Female	hm	6.845
hm_20_0036	train	3	Male	hm	4.442
hm_20_0037	test	6	Female	hm	5.983
hm_20_0038	train	3	Male	hm	1.49
hm_20_0039	test	6	Female	hm	1.333
hm_20_0040	train	3	Male	hm	0.968
hm_20_0041	test	6	Female	hm	5.226
hm_20_0042	train	2	Male	hm	5.591
hm_20_0043	train	2	Male	hm	1.83
hm_20_0044	test	6	Female	hm	1.986
hm_20_0045	test	6	Female	hm	1.986
hm_20_0046	train	2	Male	hm	5.539
hm_20_0047	test	6	Female	hm	3.292
hm_20_0048	train	2	Male	hm	8.961
hm_20_0049	test	6	Female	hm	3.867
hm_20_0050	train	2	Male	hm	1.203
hm_20_0051	test	6	Female	hm	1.412
hm_20_0052	train	2	Male	hm	4.364
hm_20_0053	test	6	Female	hm	4.259
hm_20_0054	train	2	Male	hm	10.633
hm_20_0055	test	6	Female	hm	5.304
hm_20_0056	train	3	Male	hm	2.43
hm_20_0057	train	2	Male	hm	3.789
hm_20_0058	train	3	Male	hm	2.169
hm_20_0059	train	2	Male	hm	5.826
hm_20_0060	train	2	Male	hm	9.745
hm_20_0061	train	3	Male	hm	3.684
hm_20_0062	train	3	Male	hm	7.237
hm_20_0063	train	2	Male	hm	3.528
hm_20_0064	train	3	Male	hm	2.587
hm_20_0065	train	2	Male	hm	3.632
hm_20_0066	train	3	Male	hm	1.307
hm_20_0067	train	2	Male	hm	5.461
hm_20_0068	train	3	Male	hm	2.091
hm_20_0069	train	2	Male	hm	6.636
hm_20_0070	train	2	Male	hm	3.946
hm_20_0071	train	2	Male	hm	7.42
hm_20_0072	train	2	Male	hm	3.162
hm_20_0073	train	3	Male	hm	3.449
hm_20_0074	train	2	Male	hm	0.994
hm_20_0076	train	2	Male	hm	4.259
hm_20_0077	train	3	Male	hm	1.438
hm_20_0080	test	6	Female	hm	0.759
hm_20_0082	test	6	Female	hm	1.333
hm_20_0084	test	6	Female	hm	1.96
hm_20_0086	test	6	Female	hm	2.666
hm_21_0001	train	5	Female	hm	2.927
hm_21_0003	train	5	Female	hm	6.061
hm_21_0004	train	2	Male	hm	5.461
hm_21_0005	train	5	Female	hm	1.699
hm_21_0006	train	3	Male	hm	1.359
hm_21_0007	train	2	Male	hm	4.468
hm_21_0008	train	3	Male	hm	1.804
hm_21_0009	train	5	Female	hm	2.221
hm_21_0010	train	2	Male	hm	4.076
hm_21_0011	train	3	Male	hm	0.968
hm_21_0012	train	2	Male	hm	3.11
hm_21_0013	train	3	Male	hm	3.266
hm_21_0014	train	2	Male	hm	2.326
hm_21_0015	train	5	Female	hm	5.905
hm_21_0016	train	2	Male	hm	4.834
hm_21_0017	train	3	Male	hm	6.793
hm_21_0018	train	2	Male	hm	1.751
hm_21_0019	train	5	Female	hm	5.461
hm_21_0020	train	3	Male	hm	6.061
hm_21_0021	train	2	Male	hm	4.494
hm_21_0022	train	3	Male	hm	1.02
hm_21_0023	train	2	Male	hm	4.599
hm_21_0024	train	5	Female	hm	1.751
hm_21_0025	train	2	Male	hm	2.901
hm_21_0026	train	2	Male	hm	1.882
hm_21_0027	train	3	Male	hm	2.091
hm_21_0028	train	2	Male	hm	2.692
hm_21_0029	train	3	Male	hm	1.255
hm_21_0030	train	2	Male	hm	4.155
hm_21_0031	train	3	Male	hm	2.483
hm_21_0032	train	5	Female	hm	2.012
hm_21_0033	train	3	Male	hm	5.957
hm_21_0034	train	2	Male	hm	3.266
hm_21_0035	train	5	Female	hm	2.143
hm_21_0036	train	2	Male	hm	5.252
hm_21_0037	train	3	Male	hm	3.345
hm_21_0038	train	3	Male	hm	3.58
hm_21_0039	train	3	Male	hm	3.031
hm_21_0040	train	3	Male	hm	3.371
hm_21_0041	train	2	Male	hm	3.214
hm_21_0042	train	1	Male	hm	3.136
hm_21_0043	train	3	Male	hm	2.352
hm_21_0044	train	3	Male	hm	0.654
hm_21_0045	train	3	Male	hm	4.494
hm_21_0046	train	2	Male	hm	8.778
hm_21_0047	train	3	Male	hm	1.882
hm_21_0048	train	1	Male	hm	2.3
hm_21_0049	train	3	Male	hm	1.96
hm_21_0050	train	1	Male	hm	0.915
hm_21_0051	train	3	Male	hm	8.047
hm_21_0052	train	2	Male	hm	3.946
hm_21_0053	train	3	Male	hm	4.546
hm_21_0054	train	2	Male	hm	2.535
hm_21_0055	train	3	Male	hm	6.453
hm_21_0056	train	1	Male	hm	4.337
hm_21_0057	train	3	Male	hm	3.423
hm_21_0058	train	1	Male	hm	2.352
hm_21_0059	train	3	Male	hm	2.274
hm_21_0060	train	1	Male	hm	4.05
hm_21_0061	train	3	Male	hm	2.692
hm_21_0062	train	2	Male	hm	2.169
hm_21_0063	train	3	Male	hm	6.715
hm_21_0064	train	2	Male	hm	3.763
hm_21_0065	train	3	Male	hm	2.378
hm_21_0066	train	2	Male	hm	2.143
hm_21_0067	train	3	Male	hm	5.095
hm_21_0068	train	2	Male	hm	1.255
hm_21_0069	train	3	Male	hm	4.416
hm_21_0070	train	2	Male	hm	2.404
hm_21_0071	train	3	Male	hm	4.364
hm_21_0072	train	2	Male	hm	3.084
hm_21_0073	train	3	Male	hm	2.43
hm_21_0074	train	2	Male	hm	3.057
hm_21_0075	train	3	Male	hm	3.815
hm_21_0076	train	2	Male	hm	1.333
hm_21_0077	train	3	Male	hm	2.352
hm_21_0078	train	2	Male	hm	3.136
hm_21_0079	train	3	Male	hm	4.52
hm_21_0080	train	2	Male	hm	4.311
hm_21_0081	train	3	Male	hm	2.509
hm_21_0083	train	3	Male	hm	0.889
hm_21_0084	train	2	Male	hm	1.307
hm_21_0085	train	5	Female	hm	1.307
hm_21_0086	train	1	Male	hm	0.968
hm_21_0087	train	5	Female	hm	2.039
hm_21_0088	train	5	Female	hm	2.77
hm_21_0089	train	5	Female	hm	5.957
hm_22_0001	train	3	Male	hm	2.639
hm_22_0002	train	2	Male	hm	2.535
hm_22_0003	train	3	Male	hm	2.352
hm_22_0005	train	5	Female	hm	6.035
hm_22_0006	train	5	Female	hm	1.751
hm_22_0007	train	3	Male	hm	0.732
hm_22_0008	train	5	Female	hm	0.915
hm_22_0009	train	3	Male	hm	2.613
hm_22_0010	train	5	Female	hm	1.438
hm_22_0011	train	2	Male	hm	1.908
hm_22_0012	train	3	Male	hm	0.759
hm_22_0013	train	5	Female	hm	1.177
hm_22_0014	train	3	Male	hm	1.203
hm_22_0015	train	2	Male	hm	2.457
hm_22_0016	train	5	Female	hm	2.692
hm_22_0017	train	3	Male	hm	2.091
hm_22_0018	train	5	Female	hm	1.804
hm_22_0019	train	2	Male	hm	2.875
hm_22_0020	train	5	Female	hm	3.998
hm_22_0021	train	3	Male	hm	1.438
hm_22_0022	train	5	Female	hm	1.856
hm_22_0023	train	2	Male	hm	2.509
hm_22_0024	train	5	Female	hm	3.397
hm_22_0025	train	3	Male	hm	4.259
hm_22_0026	train	5	Female	hm	3.423
hm_22_0027	train	2	Male	hm	1.934
hm_22_0028	train	3	Male	hm	0.915
hm_22_0029	train	3	Male	hm	5.121
hm_22_0030	train	5	Female	hm	4.938
hm_22_0031	train	2	Male	hm	2.848
hm_22_0032	train	5	Female	hm	1.15
hm_22_0033	train	3	Male	hm	1.934
hm_22_0034	train	5	Female	hm	4.416
hm_22_0035	train	2	Male	hm	2.091
hm_22_0036	train	3	Male	hm	1.438
hm_22_0037	train	5	Male	hm	3.946
hm_22_0038	train	3	Male	hm	5.722
hm_22_0039	train	5	Female	hm	7.812
hm_22_0040	train	2	Male	hm	4.546
hm_22_0041	train	5	Female	hm	3.789
hm_22_0042	train	3	Male	hm	2.535
hm_22_0043	train	5	Female	hm	5.696
hm_22_0044	train	3	Male	hm	6.558
hm_22_0045	train	3	Male	hm	2.822
hm_22_0046	train	5	Female	hm	2.979
hm_22_0047	train	3	Male	hm	3.58
hm_22_0048	train	5	Female	hm	3.71
hm_22_0049	train	2	Male	hm	4.05
hm_22_0050	train	5	Female	hm	4.155
hm_22_0051	train	3	Male	hm	1.986
hm_22_0052	train	2	Male	hm	2.012
hm_22_0053	train	3	Male	hm	5.748
hm_22_0054	train	5	Female	hm	3.084
hm_22_0056	train	3	Male	hm	5.983
hm_22_0057	train	2	Male	hm	3.319
hm_22_0058	train	3	Male	hm	10.319
hm_22_0059	train	2	Male	hm	7.498
hm_22_0060	train	3	Male	hm	2.143
hm_22_0061	train	2	Male	hm	0.889
hm_22_0062	train	2	Male	hm	3.345
hm_22_0063	train	3	Male	hm	2.117
hm_22_0064	train	2	Male	hm	3.475
hm_22_0065	train	1	Male	hm	4.703
hm_22_0067	train	1	Male	hm	4.494
hm_22_0069	train	1	Male	hm	3.946
hm_22_0070	train	2	Male	hm	2.039
hm_22_0071	train	1	Male	hm	1.96
hm_22_0072	train	3	Male	hm	2.169
hm_22_0073	train	1	Male	hm	5.226
hm_22_0074	train	3	Male	hm	3.162
hm_22_0075	train	2	Male	hm	2.378
hm_22_0076	train	3	Male	hm	3.998
hm_22_0077	train	1	Male	hm	4.938
hm_22_0078	train	3	Male	hm	1.359
hm_22_0079	train	1	Male	hm	0.785
hm_22_0080	train	3	Male	hm	1.386
hm_22_0081	train	1	Male	hm	3.214
hm_22_0082	train	3	Male	hm	3.084
hm_22_0083	train	1	Male	hm	1.908
hm_22_0084	train	3	Male	hm	4.128
hm_22_0085	train	1	Male	hm	2.404
hm_22_0086	train	5	Female	hm	2.718
hm_22_0087	train	3	Male	hm	0.994
hm_22_0088	train	5	Female	hm	3.632
hm_22_0089	train	3	Male	hm	1.072
hm_22_0090	train	5	Female	hm	3.554
hm_22_0091	train	3	Male	hm	2.953
hm_23_0001	train	2	Male	hm	0.759
hm_23_0002	train	3	Male	hm	2.718
hm_23_0003	train	3	Male	hm	6.088
hm_23_0004	train	2	Male	hm	5.513
hm_23_0005	train	3	Male	hm	6.506
hm_23_0006	train	5	Female	hm	3.24
hm_23_0007	train	3	Male	hm	6.427
hm_23_0008	train	5	Female	hm	1.856
hm_23_0009	train	3	Male	hm	2.875
hm_23_0010	train	2	Male	hm	9.431
hm_23_0011	train	3	Male	hm	4.703
hm_23_0012	train	2	Male	hm	2.457
hm_23_0013	train	3	Male	hm	3.136
hm_23_0014	train	5	Female	hm	3.371
hm_23_0015	train	3	Male	hm	2.979
hm_23_0016	train	5	Female	hm	1.281
hm_23_0017	train	3	Male	hm	2.77
hm_23_0018	train	2	Male	hm	8.021
hm_23_0019	train	3	Male	hm	7.368
hm_23_0020	train	5	Female	hm	4.233
hm_23_0021	train	3	Male	hm	7.472
hm_23_0022	train	2	Male	hm	9.98
hm_23_0023	train	3	Male	hm	11.39
hm_23_0024	train	2	Male	hm	1.934
hm_23_0025	train	5	Female	hm	0.889
hm_23_0026	train	1	Male	hm	1.072
hm_23_0027	train	5	Female	hm	1.124
hm_23_0028	train	3	Male	hm	1.96
hm_23_0029	train	2	Male	hm	2.848
hm_23_0030	train	3	Male	hm	1.804
hm_23_0031	train	5	Female	hm	0.759
hm_23_0032	train	3	Male	hm	3.031
hm_23_0033	train	1	Male	hm	1.595
hm_23_0034	train	2	Male	hm	3.815
hm_23_0035	train	1	Male	hm	1.464
hm_23_0036	train	5	Female	hm	1.098
hm_23_0037	train	1	Male	hm	1.621
hm_23_0038	train	3	Male	hm	2.039
hm_23_0039	train	2	Male	hm	3.319
hm_23_0040	train	2	Male	hm	6.166
hm_23_0041	train	1	Male	hm	0.706
hm_23_0042	train	2	Male	hm	0.915
hm_23_0043	train	1	Male	hm	0.471
hm_23_0044	train	3	Male	hm	3.606
hm_23_0045	train	1	Male	hm	0.863
hm_23_0046	train	3	Male	hm	6.114
hm_23_0047	train	1	Male	hm	3.057
hm_23_0048	train	2	Male	hm	5.173
hm_23_0049	train	1	Male	hm	2.404
hm_23_0050	train	2	Male	hm	1.255
hm_23_0051	train	3	Male	hm	2.43
hm_23_0052	train	1	Male	hm	3.606
hm_23_0053	train	2	Male	hm	0.811
hm_23_0054	train	1	Male	hm	4.233
hm_23_0055	train	3	Male	hm	3.266
hm_23_0056	train	1	Male	hm	2.901
hm_23_0057	train	3	Male	hm	2.927
hm_23_0058	train	3	Male	hm	3.214
hm_23_0059	train	3	Male	hm	3.292
hm_23_0060	train	3	Male	hm	2.692
hm_23_0061	train	3	Male	hm	4.311
hm_23_0063	train	3	Male	hm	0.889
hm_23_0064	train	2	Male	hm	1.359
hm_23_0065	train	5	Female	hm	1.307
hm_23_0066	train	1	Male	hm	1.072
hm_23_0067	train	5	Female	hm	2.039
hm_23_0068	train	5	Female	hm	3.815
hm_24_0001	train	2	Male	hm	2.326
hm_24_0002	train	3	Male	hm	2.77
hm_24_0003	train	4	Female	hm	2.091
hm_24_0004	train	4	Female	hm	6.297
hm_24_0005	train	5	Female	hm	1.307
hm_24_0007	train	5	Female	hm	1.621
hm_24_0009	train	5	Female	hm	1.124
hm_24_0010	train	2	Male	hm	6.871
hm_24_0011	train	5	Female	hm	3.031
hm_24_0012	train	3	Male	hm	2.927
hm_24_0013	train	2	Male	hm	2.169
hm_24_0014	train	5	Female	hm	2.378
hm_24_0015	train	3	Male	hm	4.938
hm_24_0016	train	5	Female	hm	1.49
hm_24_0017	train	5	Female	hm	2.796
hm_24_0018	train	3	Male	hm	2.77
hm_24_0019	train	2	Male	hm	3.188
hm_24_0020	train	5	Female	hm	1.255
hm_24_0021	train	3	Male	hm	3.893
hm_24_0022	train	2	Male	hm	5.121
hm_24_0023	train	5	Female	hm	2.535
hm_24_0024	train	3	Male	hm	1.83
hm_24_0025	train	5	Female	hm	5.252
hm_24_0026	train	2	Male	hm	2.326
hm_24_0027	train	5	Female	hm	4.755
hm_24_0028	train	3	Male	hm	1.934
hm_24_0029	train	2	Male	hm	4.572
hm_24_0030	train	5	Female	hm	2.404
hm_24_0032	train	5	Female	hm	2.822
hm_24_0033	train	2	Male	hm	4.285
hm_24_0034	train	3	Male	hm	1.96
hm_24_0035	train	2	Male	hm	3.501
hm_24_0036	train	3	Male	hm	3.057
hm_24_0037	train	5	Female	hm	8.83
hm_24_0038	train	3	Male	hm	4.52
hm_24_0039	train	1	Male	hm	4.05
hm_24_0040	train	5	Female	hm	0.889
hm_24_0042	train	1	Male	hm	3.789
hm_24_0043	train	3	Male	hm	1.777
hm_24_0044	train	2	Male	hm	3.606
hm_24_0045	train	3	Male	hm	5.539
hm_24_0046	train	3	Male	hm	2.457
hm_24_0047	train	2	Male	hm	2.143
hm_24_0048	train	3	Male	hm	3.972
hm_24_0049	train	1	Male	hm	3.136
hm_24_0050	train	5	Female	hm	4.86
hm_24_0051	train	1	Male	hm	1.464
hm_24_0052	train	5	Female	hm	4.729
hm_24_0053	train	1	Male	hm	6.27
hm_24_0054	train	5	Female	hm	5.774
hm_24_0055	train	3	Male	hm	5.199
hm_24_0056	train	1	Male	hm	4.155
hm_24_0057	train	3	Male	hm	1.725
hm_24_0058	train	1	Male	hm	4.677
hm_24_0059	train	3	Male	hm	0.889
hm_24_0060	train	1	Male	hm	0.837
hm_24_0061	train	2	Male	hm	4.546
hm_24_0062	train	1	Male	hm	7.159
hm_24_0063	train	2	Male	hm	8.36
hm_24_0064	test	6	Female	hm	6.793
hm_24_0065	train	3	Male	hm	1.046
hm_24_0066	train	1	Male	hm	3.789
hm_24_0067	train	2	Male	hm	6.27
hm_24_0068	train	1	Male	hm	1.542
hm_24_0069	train	3	Male	hm	2.091
hm_24_0070	train	1	Male	hm	10.894
hm_24_0071	train	2	Male	hm	3.58
hm_24_0072	train	1	Male	hm	7.786
hm_24_0073	train	3	Male	hm	0.994
hm_24_0074	test	6	Female	hm	5.147
hm_24_0075	train	1	Male	hm	2.43
hm_24_0076	test	6	Female	hm	3.867
hm_24_0077	train	1	Male	hm	5.879
hm_24_0078	test	6	Female	hm	4.808
hm_24_0079	train	1	Male	hm	2.117
hm_24_0080	train	1	Male	hm	6.349
hm_24_0081	train	3	Male	hm	5.879
hm_24_0082	train	1	Male	hm	2.744
hm_24_0083	train	2	Male	hm	5.33
hm_24_0084	train	3	Male	hm	5.696
hm_24_0085	train	1	Male	hm	9.588
hm_24_0086	train	5	Female	hm	0.968
hm_24_0087	train	3	Male	hm	1.96
hm_24_0088	train	2	Male	hm	1.203
hm_24_0089	train	5	Female	hm	1.751
hm_24_0090	train	3	Male	hm	2.248
hm_24_0091	train	5	Male	hm	2.77
hm_24_0092	train	2	Male	hm	2.77
hm_24_0093	train	5	Female	hm	3.057
hm_24_0094	train	3	Male	hm	1.203
hm_24_0095	train	5	Female	hm	4.311
hm_24_0097	train	5	Female	hm	0.811
hm_24_0099	train	4	Female	hm	12.331
hm_24_0100	train	4	Female	hm	2.666
hm_24_0101	train	4	Female	hm	7.681
hm_24_0102	dev	14	Female	hm	7.42
hm_25_0001	dev	14	Female	hm	2.117
hm_25_0002	train	4	Female	hm	2.875
hm_25_0003	train	2	Male	hm	2.3
hm_25_0004	train	3	Male	hm	2.718
hm_25_0005	train	4	Female	hm	2.039
hm_25_0006	train	4	Female	hm	6.427
hm_25_0007	train	3	Male	hm	1.359
hm_25_0008	train	2	Male	hm	1.516
hm_25_0009	train	3	Male	hm	1.333
hm_25_0010	train	2	Male	hm	3.162
hm_25_0011	train	3	Male	hm	2.169
hm_25_0012	train	2	Male	hm	3.998
hm_25_0013	train	3	Male	hm	1.386
hm_25_0014	train	5	Female	hm	0.889
hm_25_0015	train	3	Male	hm	1.621
hm_25_0016	train	5	Female	hm	1.15
hm_25_0017	train	2	Male	hm	3.501
hm_25_0018	train	3	Male	hm	7.759
hm_25_0019	train	2	Male	hm	1.83
hm_25_0020	train	5	Female	hm	2.012
hm_25_0021	train	3	Male	hm	2.953
hm_25_0022	train	2	Male	hm	6.27
hm_25_0023	train	5	Female	hm	1.673
hm_25_0024	train	2	Male	hm	1.412
hm_25_0025	train	5	Female	hm	4.729
hm_25_0026	train	3	Male	hm	2.901
hm_25_0027	train	5	Female	hm	1.49
hm_25_0028	train	2	Male	hm	3.998
hm_25_0029	train	5	Female	hm	1.699
hm_25_0030	train	3	Male	hm	3.031
hm_25_0031	train	5	Female	hm	4.39
hm_25_0032	train	3	Male	hm	1.307
hm_25_0033	train	5	Female	hm	3.371
hm_25_0035	train	5	Female	hm	2.587
hm_25_0036	train	2	Male	hm	1.83
hm_25_0037	train	5	Female	hm	1.386
hm_25_0038	train	3	Male	hm	3.24
hm_25_0039	train	5	Female	hm	4.364
hm_25_0041	train	3	Male	hm	3.58
hm_25_0042	train	2	Male	hm	3.475
hm_25_0043	train	3	Male	hm	1.621
hm_25_0044	test	6	Female	hm	1.046
hm_25_0045	train	3	Male	hm	2.666
hm_25_0046	train	2	Male	hm	1.882
hm_25_0047	test	6	Female	hm	1.882
hm_25_0048	train	3	Male	hm	1.15
hm_25_0049	test	6	Female	hm	7.237
hm_25_0050	train	2	Male	hm	4.808
hm_25_0051	test	6	Female	hm	1.595
hm_25_0052	train	3	Male	hm	3.606
hm_25_0053	test	6	Female	hm	4.599
hm_25_0054	train	2	Male	hm	2.012
hm_25_0055	test	6	Female	hm	3.658
hm_25_0056	train	3	Male	hm	1.673
hm_25_0057	test	6	Female	hm	1.124
hm_25_0058	train	2	Male	hm	3.266
hm_25_0059	test	6	Female	hm	6.114
hm_25_0061	train	3	Male	hm	5.8
hm_25_0062	train	1	Male	hm	2.065
hm_25_0063	train	2	Male	hm	9.248
hm_25_0064	train	1	Male	hm	1.386
hm_25_0065	train	3	Male	hm	4.285
hm_25_0066	train	1	Male	hm	2.848
hm_25_0067	train	3	Male	hm	10.137
hm_25_0068	train	2	Male	hm	2.561
hm_25_0069	train	1	Male	hm	8.621
hm_25_0070	train	3	Male	hm	5.826
hm_25_0071	train	1	Male	hm	3.371
hm_25_0072	train	2	Male	hm	3.998
hm_25_0073	train	1	Male	hm	3.815
hm_25_0074	train	3	Male	hm	2.613
hm_25_0075	train	1	Male	hm	3.867
hm_25_0076	train	2	Male	hm	4.05
hm_25_0077	train	3	Male	hm	11.756
hm_25_0078	train	2	Male	hm	9.196
hm_25_0079	train	3	Male	hm	7.916
hm_25_0080	train	2	Male	hm	0.732
hm_25_0081	train	1	Male	hm	2.378
hm_25_0082	train	2	Male	hm	3.449
hm_25_0083	train	1	Male	hm	1.516
hm_25_0084	train	3	Male	hm	5.252
hm_25_0085	train	1	Male	hm	4.207
hm_25_0086	train	2	Male	hm	2.195
hm_25_0087	train	3	Male	hm	5.539
hm_25_0088	train	2	Male	hm	7.786
hm_25_0089	train	1	Male	hm	1.49
hm_25_0090	train	3	Male	hm	2.3
hm_25_0091	train	2	Male	hm	6.088
hm_25_0092	train	1	Male	hm	1.777
hm_25_0094	train	1	Male	hm	0.811
hm_25_0095	train	2	Male	hm	2.953
hm_25_0096	train	5	Female	hm	0.863
hm_25_0097	train	3	Male	hm	1.751
hm_25_0098	train	5	Male	hm	1.02
hm_25_0099	train	3	Male	hm	2.195
hm_25_0100	train	5	Female	hm	2.822
hm_25_0101	train	2	Male	hm	3.528
hm_25_0102	train	5	Female	hm	0.576
hm_25_0103	train	3	Male	hm	2.535
hm_25_0104	train	5	Female	hm	3.214
hm_25_0105	train	2	Male	hm	4.181
hm_25_0106	train	3	Male	hm	3.397
hm_25_0108	train	4	Female	hm	4.207
hm_26_0001	train	3	Male	hm	3.737
hm_26_0002	test	6	Female	hm	1.255
hm_26_0003	train	3	Male	hm	2.091
hm_26_0004	test	6	Female	hm	1.595
hm_26_0005	train	3	Male	hm	4.886
hm_26_0006	test	6	Female	hm	2.326
hm_26_0007	train	3	Male	hm	6.297
hm_26_0008	test	6	Female	hm	1.281
hm_26_0009	train	3	Male	hm	1.986
hm_26_0010	test	6	Female	hm	3.449
hm_26_0011	train	3	Male	hm	2.195
hm_26_0012	test	6	Female	hm	5.722
hm_26_0013	train	3	Male	hm	3.11
hm_26_0014	test	6	Female	hm	0.576
hm_26_0015	train	3	Male	hm	5.461
hm_26_0016	test	6	Female	hm	0.732
hm_26_0017	train	3	Male	hm	0.55
hm_26_0018	test	6	Female	hm	5.33
hm_26_0019	test	6	Female	hm	3.946
hm_26_0020	train	3	Male	hm	4.938
hm_26_0021	train	3	Male	hm	0.706
hm_26_0022	test	6	Female	hm	6.819
hm_26_0023	test	6	Female	hm	4.546
hm_26_0024	test	6	Female	hm	3.554
hm_26_0025	train	3	Male	hm	2.692
hm_26_0026	test	6	Female	hm	1.438
hm_26_0027	train	3	Male	hm	3.632
hm_26_0028	test	6	Female	hm	6.871
hm_26_0029	train	3	Male	hm	6.819
hm_26_0030	test	6	Female	hm	2.639
hm_26_0031	train	3	Male	hm	1.412
hm_26_0032	test	6	Female	hm	4.938
hm_26_0033	train	3	Male	hm	2.274
hm_26_0034	test	6	Female	hm	4.285
hm_26_0035	train	3	Male	hm	0.732
hm_26_0036	test	6	Female	hm	2.639
hm_26_0037	train	3	Male	hm	4.703
hm_26_0038	test	6	Female	hm	1.072
hm_26_0039	train	3	Male	hm	6.375
hm_26_0040	test	6	Female	hm	3.057
hm_26_0041	train	3	Male	hm	8.204
hm_26_0042	test	6	Female	hm	3.554
hm_26_0043	train	3	Male	hm	5.199
hm_26_0044	test	6	Female	hm	7.498
hm_26_0045	test	6	Female	hm	2.796
hm_26_0046	test	6	Female	hm	6.793
hm_26_0047	train	3	Male	hm	1.934
hm_26_0048	test	6	Female	hm	7.629
hm_26_0049	test	6	Female	hm	10.842
hm_26_0050	test	6	Female	hm	3.057
hm_26_0051	train	3	Male	hm	1.777
hm_26_0052	test	6	Female	hm	5.983
hm_26_0053	train	3	Male	hm	1.934
hm_26_0054	test	6	Female	hm	6.767
hm_26_0055	test	6	Female	hm	2.718
hm_26_0056	train	3	Male	hm	3.946
hm_27_0002	train	5	Female	hm	5.983
hm_27_0003	train	3	Male	hm	6.166
hm_27_0004	train	2	Male	hm	2.509
hm_27_0005	train	3	Male	hm	2.639
hm_27_0006	train	2	Male	hm	3.919
hm_27_0007	train	3	Male	hm	10.528
hm_27_0008	train	2	Male	hm	8.151
hm_27_0009	train	3	Male	hm	4.442
hm_27_0010	train	2	Male	hm	3.946
hm_27_0011	train	3	Male	hm	3.606
hm_27_0012	train	2	Male	hm	2.326
hm_27_0013	train	3	Male	hm	5.644
hm_27_0014	train	3	Male	hm	8.308
hm_27_0015	train	2	Male	hm	10.346
hm_27_0016	train	3	Male	hm	8.517
hm_27_0017	train	2	Male	hm	3.919
hm_27_0018	train	3	Male	hm	5.017
hm_27_0019	train	2	Male	hm	3.11
hm_27_0020	train	3	Male	hm	9.562
hm_27_0021	train	2	Male	hm	5.774
hm_27_0022	train	3	Male	hm	5.226
hm_27_0023	train	2	Male	hm	8.073
hm_27_0024	train	3	Male	hm	6.479
hm_27_0025	train	2	Male	hm	10.842
hm_27_0026	train	1	Male	hm	0.785
hm_27_0027	train	2	Male	hm	2.796
hm_27_0028	train	3	Male	hm	0.994
hm_27_0029	train	2	Male	hm	4.886
hm_27_0030	train	3	Male	hm	3.789
hm_27_0031	train	2	Male	hm	2.927
hm_27_0032	train	1	Male	hm	3.449
hm_27_0033	train	3	Male	hm	3.423
hm_27_0034	train	2	Male	hm	7.159
hm_27_0035	train	3	Male	hm	4.39
hm_27_0036	train	3	Male	hm	9.666
hm_27_0037	train	1	Male	hm	3.606
hm_27_0038	train	2	Male	hm	7.106
hm_27_0039	train	1	Male	hm	2.639
hm_27_0040	train	2	Male	hm	2.796
hm_27_0041	train	3	Male	hm	3.893
hm_27_0042	train	1	Male	hm	1.046
hm_27_0043	train	3	Male	hm	3.031
hm_27_0044	train	2	Male	hm	1.96
hm_27_0045	train	3	Male	hm	0.55
hm_27_0046	train	1	Male	hm	1.046
hm_27_0047	train	3	Male	hm	2.457
hm_27_0048	train	2	Male	hm	2.613
hm_27_0049	train	1	Male	hm	2.117
hm_27_0050	train	3	Male	hm	0.759
hm_27_0051	train	1	Male	hm	1.203
hm_27_0052	train	3	Male	hm	4.233
hm_27_0053	train	2	Male	hm	1.568
hm_27_0054	train	1	Male	hm	3.266
hm_27_0055	train	1	Male	hm	3.763
hm_27_0056	train	2	Male	hm	1.516
hm_27_0057	train	1	Male	hm	4.024
hm_27_0058	train	3	Male	hm	7.341
hm_27_0059	train	5	Female	hm	7.681
hm_27_0060	train	5	Female	hm	5.644
hm_27_0061	train	2	Male	hm	4.677
hm_27_0062	train	2	Male	hm	1.516
hm_27_0063	train	1	Male	hm	5.356
hm_27_0064	train	3	Male	hm	3.345
hm_27_0065	train	2	Male	hm	2.953
hm_27_0066	train	3	Male	hm	3.528
hm_27_0067	train	2	Male	hm	4.808
hm_27_0068	train	3	Male	hm	3.057
hm_27_0069	train	1	Male	hm	6.95
hm_27_0070	train	1	Male	hm	2.692
hm_27_0071	train	5	Female	hm	5.931
hm_27_0072	train	1	Male	hm	6.584
hm_27_0073	train	3	Male	hm	8.256
hm_27_0074	train	1	Male	hm	3.266
hm_27_0075	train	3	Male	hm	3.763
hm_27_0076	train	3	Male	hm	6.636
hm_27_0078	train	3	Male	hm	0.863
hm_27_0079	train	2	Male	hm	1.333
hm_27_0080	train	5	Female	hm	1.281
hm_27_0081	train	5	Female	hm	1.96
hm_27_0082	train	5	Female	hm	5.931
hm_28_0001	train	5	Female	hm	2.587
hm_28_0003	train	5	Female	hm	5.983
hm_28_0004	train	3	Male	hm	7.968
hm_28_0005	train	3	Male	hm	7.838
hm_28_0006	train	3	Male	hm	2.613
hm_28_0007	train	3	Male	hm	4.572
hm_28_0008	train	3	Male	hm	8.621
hm_28_0009	train	3	Male	hm	5.121
hm_28_0010	train	3	Male	hm	6.035
hm_28_0011	train	3	Male	hm	3.319
hm_28_0012	train	3	Male	hm	3.423
hm_28_0013	train	3	Male	hm	7.603
hm_28_0014	train	2	Male	hm	3.319
hm_28_0015	train	3	Male	hm	1.203
hm_28_0016	train	2	Male	hm	4.259
hm_28_0017	train	3	Male	hm	1.438
hm_28_0018	train	2	Male	hm	1.438
hm_28_0019	train	3	Male	hm	3.658
hm_28_0020	train	2	Male	hm	2.012
hm_28_0021	train	3	Male	hm	3.554
hm_28_0022	train	2	Male	hm	3.136
hm_28_0023	train	3	Male	hm	3.292
hm_28_0024	train	2	Male	hm	2.692
hm_28_0025	train	3	Male	hm	1.333
hm_28_0026	train	2	Male	hm	3.188
hm_28_0027	train	3	Male	hm	2.77
hm_28_0028	train	3	Male	hm	2.77
hm_28_0029	train	2	Male	hm	5.852
hm_28_0030	train	3	Male	hm	2.091
hm_28_0031	train	2	Male	hm	3.136
hm_28_0032	train	3	Male	hm	3.005
hm_28_0033	train	2	Male	hm	5.069
hm_28_0034	train	1	Male	hm	3.371
hm_28_0035	train	3	Male	hm	1.359
hm_28_0036	train	5	Female	hm	2.352
hm_28_0037	train	3	Male	hm	5.539
hm_28_0038	train	1	Male	hm	4.703
hm_28_0039	train	1	Male	hm	4.39
hm_28_0040	train	1	Male	hm	3.632
hm_28_0041	train	3	Male	hm	2.248
hm_28_0042	train	1	Male	hm	0.445
hm_28_0043	train	3	Male	hm	0.759
hm_28_0044	train	5	Male	hm	0.889
hm_28_0045	train	3	Male	hm	2.692
hm_28_0046	train	1	Male	hm	1.699
hm_28_0047	train	3	Male	hm	4.625
hm_28_0048	train	5	Male	hm	0.759
hm_28_0049	train	3	Male	hm	1.124
hm_28_0050	train	1	Male	hm	0.941
hm_28_0051	train	3	Male	hm	2.404
hm_28_0052	train	3	Male	hm	2.561
hm_28_0053	train	3	Male	hm	10.163
hm_28_0054	train	5	Female	hm	1.568
hm_28_0055	train	3	Male	hm	0.759
hm_28_0056	train	1	Male	hm	3.606
hm_28_0057	train	3	Male	hm	5.095
hm_28_0058	train	5	Female	hm	1.072
hm_28_0059	train	3	Male	hm	1.699
hm_28_0060	train	5	Male	hm	2.3
hm_28_0061	train	1	Male	hm	2.404
hm_28_0062	train	3	Male	hm	6.871
hm_28_0063	train	1	Male	hm	6.741
hm_28_0064	train	5	Female	hm	6.924
hm_28_0065	train	3	Male	hm	2.3
hm_28_0066	train	1	Male	hm	3.684
hm_28_0067	train	3	Male	hm	3.005
hm_28_0068	train	5	Female	hm	3.554
hm_28_0069	train	3	Male	hm	1.856
hm_28_0070	train	1	Male	hm	4.024
hm_28_0071	train	3	Male	hm	3.71
hm_28_0072	train	5	Female	hm	1.777
hm_28_0073	train	3	Male	hm	4.39
hm_28_0074	train	3	Male	hm	4.625
hm_28_0075	train	1	Male	hm	1.359
hm_28_0076	train	3	Male	hm	2.483
hm_28_0077	train	2	Male	hm	2.169
hm_28_0078	train	3	Male	hm	3.397
hm_28_0079	train	2	Male	hm	2.457
hm_28_0080	train	3	Male	hm	5.617
hm_28_0081	train	2	Male	hm	0.968
hm_28_0082	train	3	Male	hm	2.666
hm_28_0083	train	2	Male	hm	2.065
hm_28_0084	train	3	Male	hm	5.8
hm_28_0085	train	2	Male	hm	3.632
hm_28_0086	train	3	Male	hm	1.856
hm_28_0087	train	2	Male	hm	4.86
hm_28_0088	train	3	Male	hm	8.36
hm_28_0090	train	3	Male	hm	0.837
hm_28_0091	train	2	Male	hm	1.359
hm_28_0092	train	5	Female	hm	1.359
hm_28_0093	train	1	Male	hm	1.02
hm_28_0094	train	5	Female	hm	1.934
hm_28_0095	train	5	Female	hm	2.77
hm_28_0096	train	5	Female	hm	6.009
hm_29_0001	train	5	Female	hm	2.613
hm_29_0003	train	5	Female	hm	6.009
hm_29_0004	train	5	Female	hm	1.699
hm_29_0006	train	5	Female	hm	3.136
hm_29_0007	train	3	Male	hm	3.423
hm_29_0008	train	2	Male	hm	1.333
hm_29_0009	train	3	Male	hm	3.606
hm_29_0010	train	5	Female	hm	2.221
hm_29_0012	train	5	Female	hm	2.169
hm_29_0013	train	3	Male	hm	1.255
hm_29_0014	train	2	Male	hm	6.506
hm_29_0015	train	3	Male	hm	2.509
hm_29_0016	train	5	Female	hm	1.725
hm_29_0017	train	3	Male	hm	4.834
hm_29_0018	train	2	Male	hm	6.453
hm_29_0019	train	5	Female	hm	2.221
hm_29_0020	train	3	Male	hm	4.233
hm_29_0021	train	5	Female	hm	5.931
hm_29_0022	train	5	Female	hm	4.52
hm_29_0023	train	3	Male	hm	1.255
hm_29_0024	train	5	Female	hm	3.449
hm_29_0025	train	2	Male	hm	1.542
hm_29_0026	train	3	Male	hm	0.915
hm_29_0027	train	5	Female	hm	2.169
hm_29_0028	train	2	Male	hm	1.856
hm_29_0029	train	3	Male	hm	1.177
hm_29_0030	train	5	Female	hm	1.49
hm_29_0031	train	3	Male	hm	1.281
hm_29_0032	train	5	Female	hm	2.587
hm_29_0033	train	3	Male	hm	0.706
hm_29_0034	train	5	Female	hm	2.587
hm_29_0035	train	3	Male	hm	7.211
hm_29_0036	train	5	Female	hm	1.49
hm_29_0037	train	2	Male	hm	2.012
hm_29_0038	train	3	Male	hm	2.692
hm_29_0039	train	5	Female	hm	2.901
hm_29_0040	train	2	Male	hm	3.606
hm_29_0041	train	5	Female	hm	2.692
hm_29_0042	train	3	Male	hm	4.05
hm_29_0043	train	2	Male	hm	1.281
hm_29_0044	train	5	Female	hm	1.804
hm_29_0045	train	3	Male	hm	8.021
hm_29_0046	train	5	Female	hm	4.337
hm_29_0047	train	3	Male	hm	2.901
hm_29_0048	train	2	Male	hm	3.005
hm_29_0049	train	5	Female	hm	2.535
hm_29_0050	train	3	Male	hm	1.072
hm_29_0051	train	5	Female	hm	3.841
hm_29_0052	train	2	Male	hm	3.475
hm_29_0053	train	5	Female	hm	2.457
hm_29_0054	train	3	Male	hm	3.449
hm_29_0055	train	5	Female	hm	2.091
hm_29_0056	train	2	Male	hm	1.49
hm_29_0057	train	3	Male	hm	1.02
hm_29_0058	train	5	Female	hm	3.423
hm_29_0059	train	2	Male	hm	7.759
hm_29_0060	train	5	Female	hm	3.057
hm_29_0061	train	3	Male	hm	1.333
hm_29_0062	train	5	Female	hm	5.121
hm_29_0063	train	2	Male	hm	3.528
hm_29_0064	train	3	Male	hm	2.039
hm_29_0065	train	1	Male	hm	3.005
hm_29_0066	train	2	Male	hm	1.516
hm_29_0067	train	1	Male	hm	1.856
hm_29_0068	train	3	Male	hm	1.072
hm_29_0069	train	1	Male	hm	2.901
hm_29_0070	train	2	Male	hm	2.065
hm_29_0071	train	1	Male	hm	7.185
hm_29_0072	train	1	Male	hm	2.117
hm_29_0073	train	3	Male	hm	1.072
hm_29_0074	train	1	Male	hm	0.837
hm_29_0075	train	2	Male	hm	2.221
hm_29_0076	train	1	Male	hm	7.289
hm_29_0077	train	3	Male	hm	2.979
hm_29_0078	train	1	Male	hm	3.972
hm_29_0079	train	2	Male	hm	2.274
hm_29_0080	train	1	Male	hm	3.71
hm_29_0081	train	3	Male	hm	4.442
hm_29_0082	train	5	Female	hm	1.307
hm_29_0083	train	3	Male	hm	1.516
hm_29_0084	train	2	Male	hm	0.915
hm_29_0085	train	5	Female	hm	4.259
hm_29_0086	train	2	Male	hm	1.098
hm_29_0087	train	3	Male	hm	1.281
hm_29_0088	train	1	Male	hm	0.994
hm_29_0089	train	3	Male	hm	2.326
hm_29_0090	train	1	Male	hm	2.352
hm_29_0091	train	3	Male	hm	0.68
hm_29_0092	train	3	Male	hm	1.15
hm_29_0093	train	2	Male	hm	2.979
hm_29_0094	train	3	Male	hm	0.628
hm_29_0095	train	1	Male	hm	5.017
hm_29_0098	train	3	Male	hm	0.837
hm_29_0099	train	2	Male	hm	1.412
hm_29_0100	train	5	Male	hm	1.386
hm_29_0101	train	1	Male	hm	0.941
hm_29_0102	train	5	Female	hm	1.934
hm_29_0103	train	5	Female	hm	2.77
hm_29_0104	train	5	Female	hm	6.009
hm_30_0001	train	3	Male	hm	9.64
hm_30_0002	train	3	Male	hm	10.79
hm_30_0003	train	3	Male	hm	5.147
hm_30_0004	train	3	Male	hm	11.26
hm_30_0005	train	3	Male	hm	2.77
hm_30_0006	train	3	Male	hm	4.233
hm_30_0007	train	3	Male	hm	5.017
hm_30_0008	train	3	Male	hm	4.128
hm_30_0009	train	3	Male	hm	5.043
hm_30_0010	train	3	Male	hm	4.102
hm_30_0011	train	2	Male	hm	2.378
hm_30_0012	train	2	Male	hm	6.192
hm_30_0013	train	3	Male	hm	4.964
hm_30_0014	train	2	Male	hm	6.035
hm_30_0015	train	3	Male	hm	2.274
hm_30_0016	train	3	Male	hm	9.066
hm_30_0017	train	2	Male	hm	5.826
hm_30_0018	train	3	Male	hm	8.987
hm_30_0019	train	2	Male	hm	1.882
hm_30_0020	train	3	Male	hm	8.099
hm_30_0021	train	3	Male	hm	3.893
hm_30_0022	train	2	Male	hm	3.162
hm_30_0023	train	3	Male	hm	8.491
hm_30_0024	train	2	Male	hm	1.725
hm_30_0025	train	3	Male	hm	8.177
hm_30_0026	train	2	Male	hm	3.919
hm_30_0027	train	3	Male	hm	3.319
hm_30_0028	train	3	Male	hm	2.3
hm_30_0029	train	2	Male	hm	5.8
hm_30_0030	train	2	Male	hm	1.934
hm_30_0031	train	3	Male	hm	1.568
hm_30_0032	train	2	Male	hm	3.658
hm_30_0033	train	3	Male	hm	0.602
hm_30_0034	train	2	Male	hm	6.819
hm_30_0035	train	3	Male	hm	5.644
hm_30_0036	train	3	Male	hm	1.751
hm_30_0037	train	2	Male	hm	4.599
hm_30_0038	train	3	Male	hm	2.587
hm_30_0039	train	2	Male	hm	5.852
hm_30_0040	train	3	Male	hm	2.404
hm_30_0041	train	3	Male	hm	4.781
hm_30_0042	train	3	Male	hm	3.998
hm_30_0043	train	3	Male	hm	3.893
hm_30_0044	train	2	Male	hm	3.136
hm_30_0045	train	3	Male	hm	4.233
hm_30_0046	train	3	Male	hm	1.386
hm_30_0047	train	3	Male	hm	5.304
hm_30_0048	train	3	Male	hm	4.259
hm_30_0049	train	3	Male	hm	3.658
hm_30_0050	train	2	Male	hm	3.684
hm_30_0051	train	3	Male	hm	2.378
hm_30_0052	train	3	Male	hm	4.599
hm_30_0053	train	3	Male	hm	3.658
hm_30_0054	train	3	Male	hm	3.815
hm_30_0055	train	3	Male	hm	2.796
hm_30_0056	train	3	Male	hm	3.162
hm_30_0057	train	3	Male	hm	3.188
hm_30_0058	train	3	Male	hm	4.076
hm_30_0059	train	3	Male	hm	1.595
hm_30_0060	train	2	Male	hm	5.696
hm_30_0061	train	3	Male	hm	1.124
hm_30_0062	train	3	Male	hm	7.08
hm_30_0063	train	2	Male	hm	3.815
hm_30_0064	train	2	Male	hm	4.468
hm_30_0065	train	3	Male	hm	3.005
hm_30_0066	train	3	Male	hm	4.834
hm_30_0067	train	2	Male	hm	2.953
hm_30_0068	train	2	Male	hm	5.33
hm_30_0069	train	3	Male	hm	1.725
hm_30_0070	train	3	Male	hm	4.364
hm_30_0071	train	3	Male	hm	5.774
hm_30_0072	train	2	Male	hm	4.024
hm_30_0073	train	3	Male	hm	1.464
hm_30_0074	train	3	Male	hm	2.901
hm_30_0075	train	2	Male	hm	0.628
hm_30_0076	train	3	Male	hm	3.423
hm_30_0077	train	2	Male	hm	1.412
hm_30_0078	train	3	Male	hm	5.539
hm_30_0079	train	3	Male	hm	1.046
hm_30_0080	train	2	Male	hm	0.419
hm_30_0081	train	3	Male	hm	1.177
hm_30_0082	train	2	Male	hm	0.393
hm_30_0084	train	3	Male	hm	9.092
hm_30_0085	train	2	Male	hm	0.654
hm_30_0086	train	3	Male	hm	3.71
hm_30_0087	train	3	Male	hm	3.005
hm_30_0088	train	3	Male	hm	3.946
hm_30_0089	train	3	Male	hm	2.143
hm_30_0090	train	2	Male	hm	4.677
hm_30_0091	train	3	Male	hm	1.307
hm_30_0092	train	3	Male	hm	5.435
wb_01_0001	train	15	Male	wb	3.501
wb_01_0002	train	15	Male	wb	2.535
wb_01_0003	train	15	Male	wb	6.636
wb_01_0004	dev	16	Male	wb	0.941
wb_01_0005	test	17	Male	wb	0.602
wb_01_0006	dev	16	Male	wb	0.759
wb_01_0007	test	17	Male	wb	0.654
wb_01_0008	dev	16	Male	wb	2.404
wb_01_0009	test	17	Male	wb	0.785
wb_01_0010	dev	16	Male	wb	5.487
wb_01_0011	dev	18	Male	wb	1.882
wb_01_0012	dev	18	Male	wb	1.96
wb_01_0013	dev	16	Male	wb	5.931
wb_01_0014	dev	18	Male	wb	1.516
wb_01_0015	train	19	Male	wb	7.707
wb_01_0016	test	20	Male	wb	7.864
wb_01_0017	dev	21	Male	wb	2.012
wb_01_0018	train	19	Male	wb	1.098
wb_01_0019	dev	22	Male	wb	7.315
wb_01_0020	dev	24	Male	wb	8.151
wb_01_0021	dev	25	Male	wb	5.147
wb_01_0022	test	26	Male	wb	7.916
wb_01_0023	test	26	Male	wb	6.27
wb_01_0024	test	26	Male	wb	4.128
wb_01_0026	test	26	Male	wb	6.688
wb_01_0027	dev	22	Male	wb	4.755
wb_01_0028	dev	28	Male	wb	0.706
wb_01_0029	dev	18	Male	wb	2.692
wb_01_0030	test	26	Male	wb	0.941
wb_01_0031	dev	18	Male	wb	0.863
wb_01_0032	test	26	Male	wb	0.759
wb_01_0033	dev	18	Male	wb	9.118
wb_01_0034	train	19	Male	wb	8.648
wb_01_0036	train	19	Male	wb	3.528
wb_01_0037	train	19	Male	wb	3.841
wb_01_0039	train	19	Male	wb	3.136
wb_01_0041	train	19	Male	wb	9.771
wb_01_0042	dev	18	Male	wb	2.796
wb_01_0043	train	19	Male	wb	9.066
wb_01_0044	train	19	Male	wb	5.043
wb_01_0045	dev	22	Male	wb	7.838
wb_01_0046	dev	16	Male	wb	9.353
wb_01_0047	dev	18	Male	wb	3.919
wb_01_0048	test	17	Male	wb	2.065
wb_01_0049	dev	18	Male	wb	9.666
wb_01_0051	dev	18	Male	wb	1.229
wb_01_0052	test	26	Male	wb	1.908
wb_01_0053	dev	18	Male	wb	5.121
wb_01_0054	train	19	Male	wb	2.378
wb_01_0055	train	19	Male	wb	10.137
wb_01_0056	test	26	Male	wb	9.797
wb_01_0057	test	26	Male	wb	1.255
wb_01_0058	dev	24	Male	wb	3.475
wb_01_0059	dev	24	Male	wb	2.692
wb_01_0060	test	26	Male	wb	9.431
wb_01_0061	dev	18	Male	wb	2.613
wb_01_0062	test	26	Male	wb	2.039
wb_01_0063	train	19	Male	wb	1.046
wb_01_0064	test	26	Male	wb	3.632
wb_01_0065	dev	18	Male	wb	0.576
wb_01_0066	test	17	Male	wb	4.625
wb_01_0068	dev	22	Male	wb	7.498
wb_01_0069	test	17	Male	wb	1.699
wb_01_0070	test	17	Male	wb	3.292
wb_01_0071	dev	18	Male	wb	7.341
wb_01_0072	train	19	Male	wb	10.006
wb_01_0073	train	19	Male	wb	7.185
wb_01_0074	dev	22	Male	wb	6.14
wb_01_0075	dev	16	Male	wb	2.378
wb_01_0076	train	19	Male	wb	2.796
wb_01_0078	dev	18	Male	wb	6.166
wb_01_0079	test	26	Male	wb	5.199
wb_01_0080	dev	18	Male	wb	7.002
wb_01_0081	test	20	Male	wb	1.986
wb_01_0082	dev	18	Male	wb	10.79
wb_01_0083	test	26	Male	wb	0.602
wb_01_0084	dev	18	Male	wb	2.274
wb_01_0085	dev	16	Male	wb	1.777
wb_01_0086	dev	18	Male	wb	1.15
wb_01_0087	dev	16	Male	wb	0.68
wb_01_0088	dev	16	Male	wb	3.606
wb_01_0089	dev	16	Male	wb	1.438
wb_01_0090	dev	16	Male	wb	6.479
wb_01_0091	test	20	Male	wb	5.147
wb_01_0092	test	26	Male	wb	8.021
wb_01_0093	train	19	Male	wb	11.835
wb_01_0094	test	26	Male	wb	4.076
wb_01_0095	test	17	Male	wb	11.521
wb_01_0096	dev	18	Male	wb	7.916
wb_01_0097	train	19	Male	wb	11.861
wb_01_0098	dev	24	Male	wb	6.035
wb_01_0099	test	20	Male	wb	1.882
wb_01_0100	test	17	Male	wb	8.334
wb_01_0101	dev	27	Male	wb	2.378
wb_01_0102	dev	18	Male	wb	17.738
wb_01_0103	test	20	Male	wb	1.412
wb_01_0104	dev	16	Male	wb	1.804
wb_01_0105	train	19	Male	wb	17.712
wb_01_0106	dev	27	Male	wb	1.621
wb_01_0107	test	17	Male	wb	10.868
wb_01_0108	test	20	Male	wb	3.841
wb_01_0109	dev	18	Male	wb	9.849
wb_01_0110	test	26	Male	wb	1.333
wb_01_0111	train	19	Male	wb	8.935
wb_01_0112	dev	22	Male	wb	1.856
wb_01_0113	train	19	Male	wb	7.916
wb_01_0114	test	17	Male	wb	1.072
wb_01_0115	test	17	Male	wb	5.8
wb_01_0116	train	19	Male	wb	21.709
wb_01_0117	test	20	Male	wb	2.43
wb_01_0118	train	19	Male	wb	13.611
wb_01_0119	dev	16	Male	wb	0.941
wb_01_0120	train	19	Male	wb	18.026
wb_01_0121	dev	16	Male	wb	3.449
wb_01_0122	train	19	Male	wb	2.535
wb_01_0123	dev	27	Male	wb	5.487
wb_01_0124	dev	18	Male	wb	1.046
wb_01_0125	dev	27	Male	wb	1.124
wb_01_0126	dev	18	Male	wb	6.349
wb_01_0127	test	20	Male	wb	6.297
wb_01_0128	train	19	Male	wb	25.052
wb_01_0129	test	20	Male	wb	2.666
wb_01_0130	test	26	Male	wb	6.845
wb_01_0131	test	17	Male	wb	4.808
wb_01_0132	train	19	Male	wb	26.855
wb_01_0133	test	17	Male	wb	9.457
wb_01_0134	dev	27	Male	wb	3.084
wb_01_0135	dev	18	Male	wb	4.311
wb_01_0136	train	19	Male	wb	29.493
wb_01_0137	dev	27	Male	wb	2.927
wb_01_0138	train	19	Male	wb	5.461
wb_01_0139	dev	24	Male	wb	5.826
wb_01_0140	test	17	Male	wb	3.266
wb_01_0141	train	19	Male	wb	3.345
wb_01_0142	test	17	Male	wb	3.919
wb_01_0143	train	19	Male	wb	19.097
wb_01_0144	dev	27	Male	wb	5.356
wb_01_0145	dev	18	Male	wb	2.065
wb_01_0147	test	20	Male	wb	1.751
wb_01_0148	test	17	Male	wb	13.402
wb_01_0149	train	19	Male	wb	6.976
wb_01_0150	test	17	Male	wb	6.845
wb_01_0151	train	19	Male	wb	5.199
wb_01_0152	test	17	Male	wb	4.964
wb_01_0153	train	19	Male	wb	5.539
wb_01_0154	test	17	Male	wb	5.199
wb_01_0155	train	19	Male	wb	5.931
wb_01_0156	test	26	Male	wb	6.688
wb_01_0157	dev	27	Male	wb	3.606
wb_01_0158	test	26	Male	wb	0.889
wb_01_0159	test	17	Male	wb	10.398
wb_01_0160	test	20	Male	wb	1.751
wb_01_0161	test	20	Male	wb	7.55
wb_01_0162	test	26	Male	wb	4.39
wb_01_0163	train	19	Male	wb	3.084
wb_01_0164	test	26	Male	wb	3.345
wb_01_0165	dev	22	Male	wb	5.748
wb_01_0167	train	19	Male	wb	11.364
wb_01_0168	test	17	Male	wb	5.539
wb_01_0169	dev	24	Male	wb	11.077
wb_01_0170	test	17	Male	wb	6.244
wb_01_0171	dev	22	Male	wb	7.968
wb_01_0172	train	19	Male	wb	15.413
wb_01_0173	test	17	Male	wb	10.528
wb_01_0174	train	19	Male	wb	17.216
wb_01_0176	train	19	Male	wb	19.802
wb_01_0177	test	17	Male	wb	0.654
wb_01_0178	test	17	Male	wb	14.368
wb_01_0179	train	19	Male	wb	10.398
wb_01_0180	test	20	Male	wb	2.352
wb_01_0181	train	19	Male	wb	5.382
wb_01_0183	dev	29	Female	wb	4.964
wb_01_0184	dev	22	Male	wb	5.591
wb_01_0185	dev	16	Male	wb	11.991
wb_01_0186	test	26	Male	wb	6.218
wb_01_0187	test	20	Male	wb	2.274
wb_01_0188	dev	30	Male	wb	1.359
wb_01_0189	dev	30	Male	wb	1.203
wb_01_0190	dev	29	Female	wb	0.68
wb_01_0191	dev	30	Male	wb	0.576
wb_01_0192	dev	29	Female	wb	1.751
wb_01_0193	dev	30	Male	wb	1.046
wb_01_0194	dev	24	Male	wb	2.117
wb_01_0195	dev	30	Male	wb	0.889
wb_01_0196	dev	24	Male	wb	3.919
wb_01_0197	dev	30	Male	wb	0.994
wb_01_0198	test	26	Male	wb	3.528
wb_01_0199	train	19	Male	wb	11.364
wb_01_0200	train	19	Male	wb	9.431
wb_01_0201	dev	29	Female	wb	4.285
wb_01_0202	dev	16	Male	wb	0.889
wb_01_0203	dev	30	Male	wb	2.274
wb_01_0204	test	17	Male	wb	5.121
wb_01_0205	test	17	Male	wb	9.666
wb_01_0206	test	26	Male	wb	2.143
wb_01_0207	train	19	Male	wb	18.104
wb_01_0209	test	17	Male	wb	4.546
wb_01_0210	test	17	Male	wb	7.89
wb_01_0211	test	26	Male	wb	1.046
wb_01_0212	train	19	Male	wb	3.423
wb_01_0213	dev	30	Male	wb	2.822
wb_01_0214	test	17	Male	wb	10.79
wb_01_0215	dev	16	Male	wb	1.203
wb_01_0216	train	19	Male	wb	11.678
wb_01_0217	dev	29	Female	wb	1.804
wb_01_0218	dev	30	Male	wb	1.96
wb_01_0219	test	26	Male	wb	8.909
wb_01_0220	dev	27	Male	wb	0.968
wb_01_0221	dev	16	Male	wb	6.218
wb_01_0222	dev	27	Male	wb	3.919
wb_01_0223	dev	24	Male	wb	3.684
wb_01_0224	train	19	Male	wb	5.617
wb_01_0225	dev	24	Male	wb	3.528
wb_01_0226	test	17	Male	wb	1.359
wb_01_0227	dev	24	Male	wb	7.707
wb_01_0228	test	26	Male	wb	1.595
wb_01_0229	dev	27	Male	wb	7.132
wb_01_0230	test	17	Male	wb	10.163
wb_01_0231	dev	29	Female	wb	2.3
wb_01_0232	dev	29	Female	wb	2.979
wb_01_0233	dev	30	Male	wb	2.613
wb_01_0234	dev	18	Male	wb	0.915
wb_01_0235	dev	16	Male	wb	4.625
wb_01_0236	train	19	Male	wb	2.666
wb_01_0237	train	19	Male	wb	16.876
wb_01_0238	test	26	Male	wb	2.039
wb_01_0239	dev	27	Male	wb	1.281
wb_01_0240	train	19	Male	wb	8.857
wb_01_0241	dev	27	Male	wb	0.889
wb_01_0242	train	19	Male	wb	6.061
wb_01_0243	test	26	Male	wb	4.99
wb_01_0244	dev	27	Male	wb	1.882
wb_01_0245	dev	24	Male	wb	4.05
wb_01_0247	train	15	Male	wb	6.506
wb_02_0001	train	15	Male	wb	3.528
wb_02_0002	train	15	Male	wb	2.613
wb_02_0003	train	15	Male	wb	6.479
wb_02_0004	dev	16	Male	wb	4.599
wb_02_0005	dev	18	Male	wb	3.057
wb_02_0006	dev	16	Male	wb	1.542
wb_02_0007	dev	18	Male	wb	4.259
wb_02_0008	train	19	Male	wb	7.733
wb_02_0009	dev	16	Male	wb	1.072
wb_02_0010	train	19	Male	wb	4.233
wb_02_0011	dev	18	Male	wb	1.751
wb_02_0012	dev	27	Male	wb	4.677
wb_02_0013	train	19	Male	wb	0.759
wb_02_0014	dev	27	Male	wb	0.837
wb_02_0015	train	19	Male	wb	6.453
wb_02_0016	dev	27	Male	wb	0.889
wb_02_0017	train	19	Male	wb	1.307
wb_02_0018	test	20	Male	wb	5.565
wb_02_0019	train	19	Male	wb	7.185
wb_02_0020	dev	29	Female	wb	3.057
wb_02_0021	train	19	Male	wb	6.767
wb_02_0022	dev	29	Female	wb	2.378
wb_02_0023	test	26	Male	wb	1.621
wb_02_0024	dev	27	Male	wb	0.524
wb_02_0025	test	26	Male	wb	1.124
wb_02_0026	test	17	Male	wb	6.166
wb_02_0027	dev	16	Male	wb	0.968
wb_02_0028	dev	18	Male	wb	11.495
wb_02_0030	test	20	Male	wb	12.07
wb_02_0031	dev	24	Male	wb	7.106
wb_02_0032	train	19	Male	wb	8.073
wb_02_0033	train	19	Male	wb	7.786
wb_02_0035	train	19	Male	wb	7.655
wb_02_0036	test	17	Male	wb	12.644
wb_02_0037	dev	16	Male	wb	1.621
wb_02_0038	dev	18	Male	wb	4.311
wb_02_0039	dev	27	Male	wb	4.05
wb_02_0040	dev	18	Male	wb	3.319
wb_02_0041	dev	18	Male	wb	5.539
wb_02_0042	test	20	Male	wb	3.554
wb_02_0043	train	19	Male	wb	2.3
wb_02_0044	train	19	Male	wb	4.677
wb_02_0046	train	19	Male	wb	21.709
wb_02_0047	dev	18	Male	wb	15.23
wb_02_0048	dev	29	Female	wb	2.457
wb_02_0049	dev	18	Male	wb	3.371
wb_02_0050	dev	18	Male	wb	13.271
wb_02_0051	test	26	Male	wb	7.759
wb_02_0052	train	19	Male	wb	6.401
wb_02_0054	dev	29	Female	wb	3.031
wb_02_0055	dev	18	Male	wb	2.404
wb_02_0056	test	17	Male	wb	5.879
wb_02_0057	test	20	Male	wb	2.457
wb_02_0058	train	19	Male	wb	6.532
wb_02_0059	dev	29	Female	wb	1.621
wb_02_0060	train	19	Male	wb	8.752
wb_02_0061	train	19	Male	wb	3.136
wb_02_0062	train	19	Male	wb	5.513
wb_02_0063	dev	18	Male	wb	7.89
wb_02_0064	train	19	Male	wb	1.934
wb_02_0065	test	17	Male	wb	10.999
wb_02_0066	train	19	Male	wb	2.639
wb_02_0067	test	20	Male	wb	4.625
wb_02_0068	test	17	Male	wb	2.039
wb_02_0069	train	19	Male	wb	5.043
wb_02_0071	dev	24	Male	wb	9.745
wb_02_0072	dev	18	Male	wb	5.147
wb_02_0073	train	19	Male	wb	1.647
wb_02_0074	test	17	Male	wb	3.084
wb_02_0075	train	19	Male	wb	4.886
wb_02_0076	dev	22	Male	wb	5.382
wb_02_0077	test	17	Male	wb	2.326
wb_02_0079	test	17	Male	wb	8.935
wb_02_0080	train	19	Male	wb	1.96
wb_02_0083	dev	18	Male	wb	1.804
wb_02_0084	dev	18	Male	wb	4.285
wb_02_0085	train	19	Male	wb	1.673
wb_02_0086	train	19	Male	wb	8.883
wb_02_0087	train	19	Male	wb	6.14
wb_02_0088	train	19	Male	wb	5.382
wb_02_0089	train	19	Male	wb	3.554
wb_02_0090	train	19	Male	wb	8.491
wb_02_0091	dev	27	Male	wb	0.602
wb_02_0092	dev	22	Male	wb	1.856
wb_02_0093	dev	29	Female	wb	1.804
wb_02_0094	test	17	Male	wb	5.879
wb_02_0095	dev	29	Female	wb	1.516
wb_02_0096	dev	16	Male	wb	0.445
wb_02_0097	dev	29	Female	wb	2.274
wb_02_0098	test	17	Male	wb	5.644
wb_02_0099	train	19	Male	wb	1.464
wb_02_0100	train	19	Male	wb	4.337
wb_02_0101	dev	22	Male	wb	13.637
wb_02_0102	test	26	Male	wb	6.479
wb_02_0103	test	20	Male	wb	1.177
wb_02_0104	train	19	Male	wb	6.166
wb_02_0106	train	19	Male	wb	1.464
wb_02_0107	test	20	Male	wb	0.497
wb_02_0108	train	19	Male	wb	13.376
wb_02_0109	test	17	Male	wb	7.341
wb_02_0110	test	17	Male	wb	7.237
wb_02_0111	test	17	Male	wb	3.789
wb_02_0112	train	19	Male	wb	9.17
wb_02_0113	train	19	Male	wb	5.617
wb_02_0114	test	17	Male	wb	12.2
wb_02_0115	train	19	Male	wb	5.67
wb_02_0116	test	26	Male	wb	0.785
wb_02_0117	train	19	Male	wb	7.838
wb_02_0118	train	19	Male	wb	10.189
wb_02_0120	test	17	Male	wb	6.323
wb_02_0121	dev	16	Male	wb	9.928
wb_02_0122	test	17	Male	wb	12.383
wb_02_0123	test	20	Male	wb	6.166
wb_02_0124	train	19	Male	wb	10.79
wb_02_0125	dev	22	Male	wb	1.255
wb_02_0126	train	19	Male	wb	19.201
wb_02_0128	test	17	Male	wb	7.185
wb_02_0129	dev	24	Male	wb	10.11
wb_02_0130	test	17	Male	wb	13.271
wb_02_0131	test	17	Male	wb	11.103
wb_02_0132	train	19	Male	wb	7.707
wb_02_0133	test	17	Male	wb	6.375
wb_02_0134	train	19	Male	wb	6.819
wb_02_0135	train	19	Male	wb	6.244
wb_02_0136	train	19	Male	wb	6.715
wb_02_0137	dev	22	Male	wb	0.602
wb_02_0138	test	17	Male	wb	10.816
wb_02_0139	train	19	Male	wb	7.838
wb_02_0140	train	19	Male	wb	9.248
wb_02_0141	dev	22	Male	wb	4.468
wb_02_0143	dev	16	Male	wb	10.685
wb_02_0144	dev	22	Male	wb	1.02
wb_02_0145	dev	22	Male	wb	1.255
wb_02_0146	dev	16	Male	wb	2.274
wb_02_0147	test	20	Male	wb	5.382
wb_02_0148	train	19	Male	wb	15.387
wb_02_0149	dev	16	Male	wb	2.039
wb_02_0150	train	19	Male	wb	0.445
wb_02_0151	dev	16	Male	wb	4.024
wb_02_0152	dev	16	Male	wb	7.916
wb_02_0153	train	19	Male	wb	17.947
wb_02_0154	dev	16	Male	wb	1.255
wb_02_0155	test	17	Male	wb	7.942
wb_02_0156	dev	22	Male	wb	4.964
wb_02_0157	test	17	Male	wb	7.786
wb_02_0158	train	19	Male	wb	12.827
wb_02_0160	dev	22	Male	wb	4.886
wb_02_0161	train	19	Male	wb	2.039
wb_02_0162	dev	22	Male	wb	5.226
wb_02_0163	test	26	Male	wb	7.603
wb_02_0164	test	26	Male	wb	1.281
wb_02_0165	test	20	Male	wb	1.856
wb_02_0166	test	20	Male	wb	1.438
wb_02_0167	test	17	Male	wb	9.275
wb_02_0168	dev	22	Male	wb	0.941
wb_02_0169	test	17	Male	wb	9.431
wb_02_0171	train	19	Male	wb	8.308
wb_02_0172	dev	22	Male	wb	3.136
wb_02_0173	train	19	Male	wb	13.376
wb_02_0174	test	20	Male	wb	0.941
wb_02_0175	train	19	Male	wb	7.341
wb_02_0176	test	20	Male	wb	4.625
wb_02_0177	test	17	Male	wb	13.141
wb_02_0179	test	17	Male	wb	16.092
wb_02_0180	train	19	Male	wb	14.499
wb_02_0181	dev	22	Male	wb	1.359
wb_02_0182	test	17	Male	wb	10.424
wb_02_0183	train	19	Male	wb	13.506
wb_02_0184	train	19	Male	wb	12.279
wb_02_0185	test	20	Male	wb	2.692
wb_02_0186	dev	24	Male	wb	10.11
wb_02_0187	train	19	Male	wb	16.902
wb_02_0188	test	17	Male	wb	14.76
wb_02_0189	train	19	Male	wb	4.207
wb_02_0190	dev	22	Male	wb	3.449
wb_02_0191	train	19	Male	wb	8.204
wb_02_0192	dev	22	Male	wb	4.651
wb_02_0193	test	26	Male	wb	14.421
wb_02_0194	dev	16	Male	wb	4.207
wb_02_0195	train	19	Male	wb	1.725
wb_02_0196	dev	16	Male	wb	1.02
wb_02_0197	train	19	Male	wb	4.729
wb_02_0198	test	20	Male	wb	10.346
wb_02_0200	test	17	Male	wb	13.402
wb_02_0201	test	17	Male	wb	2.326
wb_02_0202	dev	16	Male	wb	3.292
wb_02_0204	train	19	Male	wb	4.755
wb_02_0205	dev	22	Male	wb	3.632
wb_02_0206	test	17	Male	wb	0.994
wb_02_0207	dev	22	Male	wb	10.685
wb_02_0209	test	17	Male	wb	10.502
wb_02_0210	train	19	Male	wb	8.465
wb_02_0211	train	19	Male	wb	10.79
wb_02_0212	train	31	Male	wb	1.934
wb_02_0214	train	31	Male	wb	3.737
wb_02_0216	train	31	Male	wb	2.953
wb_02_0217	test	17	Male	wb	7.968
wb_02_0218	train	19	Male	wb	7.498
wb_02_0219	dev	22	Male	wb	0.837
wb_02_0220	test	17	Male	wb	4.442
wb_02_0221	train	19	Male	wb	7.838
wb_02_0222	test	17	Male	wb	8.256
wb_02_0223	train	31	Male	wb	5.173
wb_02_0224	test	20	Male	wb	6.009
wb_02_0225	train	31	Male	wb	4.572
wb_02_0227	test	17	Male	wb	6.819
wb_02_0228	train	31	Male	wb	1.02
wb_02_0229	dev	24	Male	wb	1.96
wb_02_0230	dev	24	Male	wb	4.808
wb_02_0231	dev	22	Male	wb	1.856
wb_02_0232	train	19	Male	wb	14.891
wb_02_0233	train	19	Male	wb	8.883
wb_02_0235	train	19	Male	wb	7.577
wb_02_0236	test	17	Male	wb	10.555
wb_02_0237	train	19	Male	wb	3.867
wb_02_0238	test	17	Male	wb	3.815
wb_02_0239	train	19	Male	wb	4.572
wb_02_0240	test	17	Male	wb	4.625
wb_02_0241	test	17	Male	wb	10.11
wb_02_0242	train	19	Male	wb	19.384
wb_02_0243	dev	16	Male	wb	7.054
wb_02_0244	train	19	Male	wb	1.934
wb_02_0245	dev	22	Male	wb	0.55
wb_02_0246	train	19	Male	wb	8.308
wb_02_0247	train	31	Male	wb	2.77
wb_02_0248	train	19	Male	wb	3.789
wb_02_0249	train	19	Male	wb	7.707
wb_02_0250	test	20	Male	wb	4.311
wb_02_0251	test	17	Male	wb	6.323
wb_02_0252	dev	22	Male	wb	3.789
wb_02_0253	test	17	Male	wb	7.916
wb_02_0254	train	19	Male	wb	13.48
wb_02_0255	train	19	Male	wb	9.98
wb_02_0257	train	15	Male	wb	6.636
wb_03_0001	train	15	Male	wb	2.875
wb_03_0002	train	15	Male	wb	2.43
wb_03_0003	train	15	Male	wb	3.423
wb_03_0004	dev	16	Male	wb	2.561
wb_03_0005	test	20	Male	wb	3.11
wb_03_0006	dev	22	Male	wb	8.021
wb_03_0007	test	26	Male	wb	3.763
wb_03_0011	test	26	Male	wb	1.124
wb_03_0012	dev	18	Male	wb	1.046
wb_03_0013	test	26	Male	wb	2.979
wb_03_0014	dev	24	Male	wb	2.561
wb_03_0017	test	26	Male	wb	1.751
wb_03_0018	test	26	Male	wb	3.972
wb_03_0019	test	26	Male	wb	1.856
wb_03_0020	test	26	Male	wb	3.763
wb_03_0021	dev	24	Male	wb	4.52
wb_03_0023	dev	33	Male	wb	6.035
wb_03_0024	dev	33	Male	wb	4.781
wb_03_0025	dev	22	Male	wb	11.547
wb_03_0026	dev	33	Male	wb	2.012
wb_03_0027	dev	18	Male	wb	6.166
wb_03_0028	dev	18	Male	wb	0.889
wb_03_0029	dev	18	Male	wb	2.692
wb_03_0030	test	20	Male	wb	6.636
wb_03_0031	train	19	Male	wb	8.125
wb_03_0032	test	26	Male	wb	0.941
wb_03_0033	train	19	Male	wb	10.868
wb_03_0034	dev	33	Male	wb	10.555
wb_03_0035	dev	18	Male	wb	11.364
wb_03_0036	test	26	Male	wb	4.86
wb_03_0037	dev	16	Male	wb	1.621
wb_03_0038	dev	22	Male	wb	1.542
wb_03_0039	test	26	Male	wb	3.475
wb_03_0040	dev	18	Male	wb	2.875
wb_03_0041	train	19	Male	wb	2.091
wb_03_0042	test	26	Male	wb	5.226
wb_03_0043	dev	16	Male	wb	7.263
wb_03_0044	dev	16	Male	wb	4.207
wb_03_0045	test	26	Male	wb	0.497
wb_03_0046	dev	18	Male	wb	11.129
wb_03_0048	dev	18	Male	wb	1.02
wb_03_0049	train	19	Male	wb	14.159
wb_03_0050	dev	18	Male	wb	11.234
wb_03_0051	dev	16	Male	wb	1.046
wb_03_0052	train	35	Male	wb	10.293
wb_03_0053	dev	34	Male	wb	9.379
wb_03_0054	dev	24	Male	wb	8.177
wb_03_0055	test	20	Male	wb	7.289
wb_03_0056	dev	18	Male	wb	5.173
wb_03_0058	dev	18	Male	wb	4.834
wb_03_0060	dev	18	Male	wb	0.889
wb_03_0061	train	19	Male	wb	2.639
wb_03_0062	train	19	Male	wb	1.177
wb_03_0063	train	19	Male	wb	5.278
wb_03_0064	dev	18	Male	wb	12.932
wb_03_0065	dev	16	Male	wb	0.889
wb_03_0066	train	19	Male	wb	12.801
wb_03_0067	dev	18	Male	wb	4.128
wb_03_0068	dev	18	Male	wb	1.359
wb_03_0069	train	19	Male	wb	3.031
wb_03_0070	dev	18	Male	wb	1.177
wb_03_0071	dev	16	Male	wb	1.725
wb_03_0072	test	26	Male	wb	1.386
wb_03_0073	train	19	Male	wb	19.07
wb_03_0074	dev	22	Male	wb	5.722
wb_03_0075	dev	34	Male	wb	6.506
wb_03_0076	dev	22	Male	wb	1.124
wb_03_0077	dev	34	Male	wb	8.674
wb_03_0079	test	26	Male	wb	6.636
wb_03_0080	test	26	Male	wb	1.02
wb_03_0081	train	19	Male	wb	9.066
wb_03_0082	train	19	Male	wb	1.386
wb_03_0083	dev	18	Male	wb	0.994
wb_03_0084	train	19	Male	wb	3.475
wb_03_0085	train	19	Male	wb	1.255
wb_03_0086	dev	18	Male	wb	4.494
wb_03_0087	dev	18	Male	wb	7.211
wb_03_0088	dev	18	Male	wb	7.237
wb_03_0089	dev	24	Male	wb	1.307
wb_03_0090	train	19	Male	wb	4.364
wb_03_0091	test	20	Male	wb	0.889
wb_03_0092	train	19	Male	wb	11.103
wb_03_0093	test	20	Male	wb	3.188
wb_03_0094	test	26	Male	wb	0.889
wb_03_0095	test	20	Male	wb	0.497
wb_03_0096	test	26	Male	wb	1.751
wb_03_0097	test	20	Male	wb	0.732
wb_03_0098	test	26	Male	wb	9.353
wb_03_0099	dev	18	Male	wb	3.946
wb_03_0100	train	19	Male	wb	4.755
wb_03_0101	dev	18	Male	wb	0.497
wb_03_0102	train	19	Male	wb	11.234
wb_03_0103	dev	24	Male	wb	5.382
wb_03_0104	dev	34	Male	wb	16.563
wb_03_0105	dev	24	Male	wb	0.576
wb_03_0106	dev	34	Male	wb	3.292
wb_03_0107	dev	24	Male	wb	0.811
wb_03_0108	dev	18	Male	wb	11.234
wb_03_0109	dev	16	Male	wb	3.606
wb_03_0110	dev	34	Male	wb	0.994
wb_03_0111	dev	18	Male	wb	1.124
wb_03_0112	train	19	Male	wb	9.144
wb_03_0113	test	26	Male	wb	2.927
wb_03_0114	train	19	Male	wb	1.986
wb_03_0115	train	19	Male	wb	10.79
wb_03_0116	train	19	Male	wb	8.256
wb_03_0117	dev	18	Male	wb	6.401
wb_03_0118	dev	18	Male	wb	9.928
wb_03_0119	dev	34	Male	wb	14.577
wb_03_0120	dev	16	Male	wb	3.998
wb_03_0121	dev	18	Male	wb	4.86
wb_03_0122	dev	16	Male	wb	0.732
wb_03_0123	dev	18	Male	wb	2.77
wb_03_0124	test	26	Male	wb	2.796
wb_03_0125	train	19	Male	wb	11.573
wb_03_0126	train	19	Male	wb	8.543
wb_03_0127	train	19	Male	wb	6.897
wb_03_0128	dev	34	Male	wb	15.1
wb_03_0129	dev	33	Male	wb	2.535
wb_03_0130	dev	18	Male	wb	4.546
wb_03_0132	dev	18	Male	wb	0.393
wb_03_0133	dev	18	Male	wb	6.349
wb_03_0134	test	26	Male	wb	1.046
wb_03_0135	dev	18	Male	wb	1.098
wb_03_0136	train	19	Male	wb	5.539
wb_03_0137	train	19	Male	wb	5.043
wb_03_0138	dev	24	Male	wb	0.889
wb_03_0139	train	19	Male	wb	9.066
wb_03_0141	dev	33	Male	wb	7.733
wb_03_0142	dev	18	Male	wb	2.117
wb_03_0143	dev	34	Male	wb	1.098
wb_03_0144	dev	18	Male	wb	0.759
wb_03_0145	dev	18	Male	wb	0.706
wb_03_0146	dev	34	Male	wb	8.804
wb_03_0147	dev	18	Male	wb	1.177
wb_03_0148	dev	24	Male	wb	1.908
wb_03_0149	dev	18	Male	wb	2.43
wb_03_0150	dev	24	Male	wb	1.15
wb_03_0151	dev	18	Male	wb	13.715
wb_03_0153	dev	24	Male	wb	4.755
wb_03_0154	train	19	Male	wb	4.755
wb_03_0155	test	20	Male	wb	3.606
wb_03_0156	train	19	Male	wb	14.447
wb_03_0157	dev	18	Male	wb	5.774
wb_03_0158	dev	18	Male	wb	11.286
wb_03_0159	test	20	Male	wb	1.124
wb_03_0160	dev	18	Male	wb	5.017
wb_03_0161	test	26	Male	wb	2.457
wb_03_0162	dev	18	Male	wb	9.275
wb_03_0163	dev	24	Male	wb	1.281
wb_03_0164	train	19	Male	wb	21.108
wb_03_0166	test	17	Male	wb	16.38
wb_03_0167	dev	18	Male	wb	1.804
wb_03_0168	dev	34	Male	wb	18.6
wb_03_0169	dev	18	Male	wb	12.958
wb_03_0170	test	20	Male	wb	1.568
wb_03_0171	train	19	Male	wb	18.261
wb_03_0173	train	19	Male	wb	8.151
wb_03_0174	test	20	Male	wb	1.568
wb_03_0175	dev	34	Male	wb	7.524
wb_03_0176	dev	24	Male	wb	2.091
wb_03_0177	train	19	Male	wb	3.084
wb_03_0178	train	19	Male	wb	12.618
wb_03_0179	test	26	Male	wb	1.281
wb_03_0180	test	17	Male	wb	2.848
wb_03_0182	test	17	Male	wb	1.542
wb_03_0183	dev	24	Male	wb	1.359
wb_03_0184	dev	24	Male	wb	2.091
wb_03_0185	dev	24	Male	wb	2.117
wb_03_0186	test	26	Male	wb	4.05
wb_03_0189	dev	34	Male	wb	24.243
wb_03_0190	dev	18	Male	wb	2.457
wb_03_0192	dev	18	Male	wb	9.536
wb_03_0193	test	26	Male	wb	6.088
wb_03_0194	test	26	Male	wb	8.386
wb_03_0195	test	17	Male	wb	4.52
wb_03_0196	test	26	Male	wb	1.177
wb_03_0197	test	17	Male	wb	1.751
wb_03_0198	train	19	Male	wb	3.658
wb_03_0199	train	19	Male	wb	22.597
wb_03_0200	dev	34	Male	wb	2.3
wb_03_0201	test	17	Male	wb	18.417
wb_03_0203	dev	18	Male	wb	6.166
wb_03_0204	dev	18	Male	wb	10.476
wb_03_0206	train	19	Male	wb	11.469
wb_03_0207	test	17	Male	wb	12.592
wb_03_0208	dev	34	Male	wb	13.559
wb_03_0209	dev	18	Male	wb	11.704
wb_03_0211	train	19	Male	wb	10.946
wb_03_0212	test	17	Male	wb	11.051
wb_03_0213	dev	34	Male	wb	11.652
wb_03_0214	test	20	Male	wb	3.815
wb_03_0215	test	20	Male	wb	2.091
wb_03_0216	test	17	Male	wb	1.177
wb_03_0217	dev	18	Male	wb	9.64
wb_03_0218	train	19	Male	wb	10.79
wb_03_0219	train	19	Male	wb	7.42
wb_03_0220	test	17	Male	wb	13.35
wb_03_0221	dev	34	Male	wb	11.756
wb_03_0222	dev	18	Male	wb	5.069
wb_03_0223	dev	18	Male	wb	8.204
wb_03_0225	dev	24	Male	wb	2.796
wb_03_0226	test	26	Male	wb	1.464
wb_03_0228	test	26	Male	wb	1.986
wb_03_0229	test	17	Male	wb	4.024
wb_03_0230	train	35	Male	wb	3.841
wb_03_0231	dev	34	Male	wb	7.786
wb_03_0232	test	26	Male	wb	6.218
wb_03_0233	dev	18	Male	wb	5.382
wb_03_0234	dev	16	Male	wb	8.282
wb_03_0235	test	20	Male	wb	6.061
wb_03_0236	dev	24	Male	wb	4.834
wb_03_0237	dev	16	Male	wb	3.266
wb_03_0238	dev	24	Male	wb	1.751
wb_03_0239	test	26	Male	wb	9.588
wb_03_0240	test	20	Male	wb	2.274
wb_03_0241	dev	27	Male	wb	1.359
wb_03_0242	dev	34	Male	wb	5.748
wb_03_0243	train	32	Male	wb	0.811
wb_03_0244	train	32	Male	wb	2.666
wb_03_0245	test	26	Male	wb	2.352
wb_03_0246	train	32	Male	wb	12.331


In [ ]:
%%writefile /kaggle/working/configs/zeroshot.json
{
  "name": "zeroshot",
  "seed": 42,
  "bootstrap": 1000,
  "batch_size": 16,
  "split_tsv": "/kaggle/working/splits/sdn_split_v1.tsv",
  "cache_dir": "/tmp/cache",
  "out_dir": "/kaggle/working/outputs",
  "models": [
    {"id": "openai/whisper-small"},
    {"id": "openai/whisper-medium"},
    {"id": "openai/whisper-large-v2", "batch_size": 8},
    {"id": "openai/whisper-large-v2", "name": "openai/whisper-large-v2@auto-lang", "language": null, "sets": ["ookeval"], "batch_size": 8},
    {"id": "openai/whisper-large-v3", "batch_size": 8},
    {"id": "openai/whisper-large-v3-turbo"},
    {"id": "AymanMansour/Whisper-Sudanese-Dialect-small", "name": "AymanMansour/Whisper-Sudanese-Dialect-small (2022, external ref)", "gen_config_from": "openai/whisper-small", "sets": ["ookeval"]}
  ]
}


In [ ]:
!cd /kaggle/working/tests && PYTHONPATH=/kaggle/working/src python -m unittest -v test_textnorm && cd /kaggle/working/src && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u run_zeroshot.py --config ../configs/zeroshot.json 2>&1 | tee /kaggle/working/outputs/log_sudanese-asr-zeroshot.txt

In [ ]:
!cp /kaggle/working/splits/sdn_split_v1.tsv /kaggle/working/configs/*.json /kaggle/working/outputs/ && pip freeze > /kaggle/working/outputs/pip_freeze.txt && ls -la /kaggle/working/outputs